# **CS781 Assignment 2** : **Neural Sequence Models[1100 marks]**


#**Enter your details below:**

Full Name: YourName

Roll No: 007

Email: youremail@iitk.ac.in



#Table of Content

#Motivation & Structure- Neural Sequence Models
In Assignment-1, we explored language modeling at the token and local-context level—covering whitespace tokenization, corpus statistics, n-gram models, smoothing, Word2Vec, NNLM, and BPE sub-tokenization. While these methods helped us understand word representations and local context, they possess fundamental limitations: n-gram models rely on a fixed, limited history and suffer from data sparsity, whereas traditional NNLMs operate on fixed-size context windows and fail to capture long-range dependencies.

Assignment-2 transitions from local language modeling to explicit sequence modeling, sequence-to-sequence learning, and attention mechanisms. Rather than studying architectures in isolation, A-2 is structured as a guided evolutionary chain where every new design choice directly addresses a specific limitation observed in the previous model.

Key Progression & Architectural Evolution
A-1: Corpus Tokens/Subtokens Local Context → N-grams → Word Embeddings → NNLM

A-2: Sequential Context → RNN/LSTM/GRU (Single, Stacked, Bidirectional) → Seq2Seq Translation → Attention Mechanisms → Native PyTorch Optimizations



#**Multi-Label Emotion Classification on GoEmotion dataset**



## **Q1:Implementation of RNN, LSTM and GRU from scratch for Emotion Prediction Task[Total = 230 Marks]**


### **Step 1: Initial Setup**

1. Install the `datasets` library made by huggingface to download and interact with the data.
2. Import the `load_dataset` method from `datasets`
3. Copy and paste the huggingface token you generated above to start using the assingment data
4. The repository id of the data stored in https://huggingface.co/datasets/Exploration-Lab/CS781/tree/main/Assignment2

The notebook contains a placeholder (`SEED = "Fill In Seed Value"`). You are free to use your own chosen standard fixed integer seed (such as `42` or `123`) in the notebook configuration.

Autograder Requirement: The automated evaluation scripts and hidden test suites check your saved state dictionaries (`.pt` files) and exported model definitions rather than enforcing a specific hardcoded seed value.
Best Practice: Choose a fixed integer (e.g., `42`), keep it consistent across all cells, and document it in your notebook report as required by the reproducibility guidelines.



Significance of a Seed Value
A random seed initializes the pseudo-random number generator used across libraries like Python's `random`, NumPy, and PyTorch. Setting a seed ensures deterministic behavior, meaning that random operations—such as weight initialization, data shuffling during batch creation, dropout masks, and stochastic sampling—will follow the exact same sequence every time you run the code. This is fundamental for reproducibility, allowing you and the TAs to replicate training runs, reproduce exact metric scores, and debug effectively.

Why It Was Not Hard-Coded
Leaving the seed unassigned (`SEED = "Fill In Seed Value"`) and letting you define own serves a couple of purposes:

1. Promotes Independence and Experimentation: It encourages students to understand reproducibility mechanics firsthand rather than treating configuration fields as a black box.
2. Prevents Trivial Overlaps:In an individual assignment setting where you will train models on shared infrastructure, allowing custom seeds ensures that random variations remain distinct across individual submissions, verifying that each student ran their own independent pipeline.

In [ ]:

# --- Hugging Face authentication ---
#Initialize REPO_ID : Exploration Lab dataset folder
REPO_ID = "Exploration-Lab/CS781"  # Do not Change this repo id
# TODO: Compute or initialize hf_token
hf_token = "Put Your Hugging face Token Here"

# Install and import dependencies

# TODO: Run the required notebook setup command

import os, random, time, math, re, json, pickle as _pkl
from getpass import getpass
from datasets import load_dataset
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import emoji
import sacrebleu
import seaborn as sns
import random
import torch.nn.functional as F




# --- Reproducibility ---

SEED = "Fill In Seed Value" # Don't Change
random.seed(SEED);
np.random.seed(SEED);
torch.manual_seed(SEED);
torch.cuda.manual_seed_all(SEED)



# --- Setting up the device (for putting the tensors on GPU) ---

def get_device():
    """
      Return the computing device used in the current session.
    """
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = get_device()
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))



"""Checkpointing utilities save model states to CKPT_DIR/<tag>.pt after each epoch, providing several key benefits:
Resilience: Crashed or disconnected training runs automatically resume from the last completed epoch rather than restarting from scratch.
Efficiency: Re-running a training cell that has already finished execution skips the retraining process entirely.
Inspectability: Trained models can be reloaded later for qualitative analysis without requiring a full retraining pass."""


CKPT_DIR = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def save_checkpoint(tag: str, model: nn.Module, optimizer=None, epoch: int = None, history: dict = None):
    state = {"model": model.state_dict(), "epoch": epoch, "history": history}
    if optimizer is not None:
        state["optimizer"] = optimizer.state_dict()
    torch.save(state, os.path.join(CKPT_DIR, f"{tag}.pt"))

def load_checkpoint(tag: str, model: nn.Module, optimizer=None, map_location=None):
    """Loads weights (+ optimizer/history/epoch if present) into `model` in place.
    Returns (epoch, history), or (None, None) if no checkpoint exists for `tag`."""
    path = os.path.join(CKPT_DIR, f"{tag}.pt")
    if not os.path.exists(path):
        return None, None
    state = torch.load(path, map_location=map_location or DEVICE)
    model.load_state_dict(state["model"])
    if optimizer is not None and "optimizer" in state:
        optimizer.load_state_dict(state["optimizer"])
    return state.get("epoch"), state.get("history")

def count_parameters(model: nn.Module) -> int:
    """Total trainable scalar parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

###**Step 2: Understanding the Dataset [Marks : 15]**

**What is GoEmotions?**

Developed by researchers at Google Research, GoEmotions is a dataset of more than 58,000 curated Reddit comments labelled across **28 emotion categories** (27 emotions plus a neutral class). Unlike ordinary single-label sentiment classification, GoEmotions is **multi-label**: one comment can express several emotions at once. For example, *"I am so shocked and thrilled by this unexpected news!"* can correctly receive both **surprise** and **joy**.

The official dataset and reference implementation are available at [Google Research — GoEmotions](https://github.com/google-research/google-research/tree/master/goemotions).



**Data Preparation for Emotion Classification** : In this module, we will load, clean, tokenize, and format the **GoEmotions** dataset to prepare it for neural networks (RNN/LSTM/GRU) training.



**Loading the Dataset**

We will be using Hugging Face’s `datasets.load_dataset` to stream three parquet files directly into a structured `DatasetDict`.

In [ ]:
#Do not Change content of this cell
from datasets import load_dataset

def load_goemotions(repo_id: str, token: str):
    """Load train, validation, and test splits from the course HF repository."""
    return load_dataset(
        repo_id,
        data_files={
            "train": "Assignment2/goemotions_1.parquet",
            "validation": "Assignment2/goemotions_2.parquet",
            "test": "Assignment2/goemotions_3_1.parquet",
        },
        token=token,
    )

raw = load_goemotions(REPO_ID, hf_token)

**Inspection, Schema Alignment & Multi-Label Preparation**

**Why do we need this?**

1. **Schema Check:** Hugging Face datasets can come in different shapes. Sometimes multi-label targets are saved as a single list of positive indices (`[1, 5]`), and other times as individual binary columns (`admiration=0, amusement=1, ...`). We standardize both formats into a uniform list of positive label indices.
2. **Filtering Emojis:** GoEmotions originates from Reddit comments, which often contain emojis. Filtering them out via `remove_emojis` keeps our vocabulary clean and prevents rare symbols from clogging our dictionary.
3. **Multi-Hot Targets:** Unlike single-label restriction, we retain **every example** and all **28 emotion classes**. Each comment's target is converted into a length-28 binary vector (`multi_hot`) with `1` for each present emotion and `0` otherwise, allowing the model to predict multiple emotions simultaneously.

**Example**

* **Raw Comment:** `"I love this so much! ❤️ rare moment of peace."`
* **After Emoji Removal:** `"I love this so much!  rare moment of peace."`
* **Multi-Hot Target Vector:** A 28-dimensional vector where indices corresponding to *admiration* and *joy* are set to `1`, and all other emotion indices are set to `0`.

In [ ]:
#Observe the raw train data before preprocessing:
print(raw["train"])

In [ ]:
# --- GoEmotions Multi-Label Preprocessing ---

TEXT_COL, LABEL_COL = "text", "labels"

_GOEMOTIONS_28 = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring", "confusion",
    "curiosity", "desire", "disappointment", "disapproval", "disgust", "embarrassment",
    "excitement", "fear", "gratitude", "grief", "joy", "love", "nervousness", "optimism",
    "pride", "realization", "relief", "remorse", "sadness", "surprise", "neutral",
]


# TODO 1: Check if LABEL_COL is present in dataset schema; if not, extract emotion columns and map index lists
if __________________________ not in raw["train"].column_names:
    emotion_cols = __________________________

    def extract_indices(row):
        return {
            LABEL_COL: __________________________
        }

    for split in raw:
        raw[split] = __________________________
else:
    emotion_cols = _GOEMOTIONS_28


# TODO 2: Define emoji-removal function using the emoji library
def remove_emojis(text: str) -> str:
    return __________________________


# TODO 3: Define preprocessing function to clean text and create multi-hot target vectors
def preprocess_fn(example: dict) -> dict:
    positive_labels = __________________________
    return {
        TEXT_COL: __________________________0,
        "multi_hot": __________________________
    }


# TODO 4: Apply preprocessing across all dataset splits
filtered = {
    split: __________________________
    for split in raw
}


# TODO 5: Shuffle dataset splits using SEED and use the full dataset
train_ds = __________________________
val_ds = __________________________
test_ds = __________________________

# Verify that the loaded splits are separate and correct

print(f"Multi-label samples: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")

### **Task 3: Tokenization & Vocabulary Building[Marks :15]**

Tokenization splits text into smaller, discrete pieces called tokens, while vocabulary building assigns each token a unique numerical ID so that a neural network can process it.

**What We Are Doing in This Task**

We are implementing a Regex Word-Level Tokenization strategy. This involves scanning the training data, splitting comments into individual words and punctuation marks using regular expressions, and picking the most frequent words to form a vocabulary. We will build a vocabulary mapping top words to unique IDs, keeping slot `0` for `<pad>` (padding short sentences) and slot `1` for `<unk>` (unknown/rare words).

**Comparison of Tokenization Techniques**

**1. Character Tokenization**

* **What it does:** Splits text into individual letters (e.g., `"cat"` becomes `["c", "a", "t"]`).


* **What it is good for:** Keeping the vocabulary size extremely small and eliminating out-of-vocabulary errors entirely.
* **Why not for this task:** It causes text sequences to become exceptionally long. In custom, from-scratch recurrent networks used for emotion prediction, excessively long sequences worsen the vanishing gradient problem and increase memory overhead, making it difficult for the model to capture the overall sentiment.



**2. Subword Tokenization (BPE/WordPiece)**

* **What it does:** Splits words into frequent morphological pieces or subwords (e.g., `"unhappiness"` becomes `["un", "happi", "ness"]`).


* **What it is good for:** Modern large-scale transformer models like BERT and GPT, as it handles rare, misspelled, or out-of-vocabulary words cleanly.


* **Why not for this task:** It introduces complex vocabulary merging algorithms that are unnecessary for standard word-level emotion classification pipelines and add unwanted overhead to the assignment.



**3. Word/Regex Tokenization**

* **What it does:** Splits text strictly on whitespace and punctuation (e.g., `"I'm happy!"` becomes `["i", "'m", "happy", "!"]`).


* **What it is good for:** Keeping sequence lengths short and clean while isolating the model's performance strictly to the neural network design.


* **Why it is chosen for this task:** For the GoEmotions multi-label emotion prediction task, word-level tokenization keeps sequence lengths tightly bounded (e.g., `MAX_LEN = 30`). This ensures that our custom recurrent emotion classifier processes data efficiently, avoids memory bottlenecks, and successfully maps full comment structures to 28 emotion categories.

Before filling out the code below, familiarize yourself with these key terms and variables:

* **`TOKEN_RE`**: A compiled regular expression object used to split raw text into clean word and punctuation tokens.
* **`PAD` (`"<pad>"`)**: A special padding token. Since neural networks require uniform tensor batch sizes, shorter sentences are filled up to `MAX_LEN` using this token (assigned index `0`).
* **`UNK` (`"<unk>"`)**: The unknown token. Any rare word encountered during testing that is not part of our fixed vocabulary is mapped to this token (assigned index `1`).
* **`MAX_VOCAB`**: The upper limit on how many unique words our vocabulary can store (e.g., 12,000) to keep model parameter sizes manageable.
* **`MAX_LEN`**: The fixed sequence length constraint. Sentences longer than this are truncated, and shorter ones are padded.
* **`counter`**: A frequency dictionary (`Counter`) used to track how often each word appears across the training corpus.
* **`itos` ("Index-to-String")**: A Python list where an integer index position maps back to its corresponding word string (e.g., `itos[42] -> 'happy'`).
* **`stoi` ("String-to-Index")**: A Python dictionary mapping word strings to their unique integer IDs (e.g., `stoi['happy'] -> 42`), enabling fast numerical lookups.
* **`VOCAB_SIZE`**: The total length of the `itos` list, which determines the input dimension of embedding layers.

In [ ]:
# Simple regex tokenizer (words + punctuation)
# TODO 1: Define TOKEN_RE
TOKEN_RE = __________________________

def tokenize(text: str) -> list[str]:
    """Splits raw text string into a list of lowercase word/punctuation tokens."""
    # TODO 2: Return the computed result
    return __________________________

# Special tokens for padding uniform lengths and handling unknown words
# TODO 3: Define PAD, UNK
PAD, UNK = __________________________

# Maximum unique vocabulary size
# TODO 4: Define MAX_VOCAB
MAX_VOCAB = __________________________

# Fixed sequence length constraint
# TODO 5: Define MAX_LEN
MAX_LEN = __________________________

# Build Vocabulary on Training Data
# TODO 6: Define counter
counter = __________________________

# TODO 7: Iterate over the required data
for ex in __________________________:
    # TODO 8: Perform the required operation (update counter with tokens)
    __________________________

# itos: Index-to-String list (reserving index 0 for PAD and 1 for UNK)
# TODO 9: Define itos
itos = __________________________

# stoi: String-to-Index dictionary mapping words to integer IDs
# TODO 10: Define stoi
stoi = __________________________

# Total vocabulary size
# TODO 11: Define VOCAB_SIZE
VOCAB_SIZE = __________________________

# TODO 12: Perform the required operation (Print vocabulary size)
__________________________

def encode(text: str) -> list[int]:
    """Tokenizes text, maps words to IDs using stoi, truncates, and pads to MAX_LEN."""
    # TODO 13: Define ids
    ids = __________________________

    # TODO 14: Return the computed result
    return __________________________

### Task 4: PyTorch DataLoaders[15 marks]

**Why do we need this?**

Neural networks train in parallel minibatches on GPUs. We need to convert our lists of string tokens and integer labels into uniform `torch.Tensor` objects, then stream them to the model using PyTorch `DataLoader` objects.

**What are we doing?**

We wrap our preprocessed HF dataset inside a custom `torch.utils.data.Dataset` class and instantiate training, validation, and test dataloaders.

**Understanding PyTorch Datasets and DataLoaders**

Before filling out the code below, familiarize yourself with these core components:

* **`Dataset` (`torch.utils.data.Dataset`)**: An abstract class representing a dataset. You must implement three methods:
  * `__init__`: Initializes and stores your data splits (e.g., source texts and multi-hot target labels).
  * `__len__`: Returns the total number of samples in the dataset.
  * `__getitem__`: Retrieves a single sample at a given index `idx`, converting text tokens into a `torch.LongTensor` and multi-hot labels into a `torch.FloatTensor`.
* **`DataLoader` (`torch.utils.data.DataLoader`)**: A utility that wraps a `Dataset` to provide batching, shuffling, and parallel data loading for training on GPUs.
* **`batch_size`**: The number of samples processed together in a single forward/backward pass (e.g., `64` for training, `128` for validation/testing).
* **Batch Tensors (`sample_x`, `sample_y`)**: The resulting batched tensors extracted from the dataloader to verify dimensions (expected shapes: `[batch_size, MAX_LEN]` for inputs and `[batch_size, 28]` for multi-hot emotion targets).

In [ ]:
class EmotionDataset(Dataset):
    """Returns token IDs and a 28-dimensional floating-point multi-hot target."""

    def __init__(self, hf_ds):
        super().__init__()
        # TODO 1: Define self.texts
        self.texts = __________________________
        # TODO 2: Define self.targets
        self.targets = __________________________

    def __len__(self):
        # TODO 3: Return the computed result (total number of samples)
        return __________________________

    def __getitem__(self, idx):
        # TODO 4: Define x (encode the text at idx into a torch.long tensor)
        x = __________________________
        # TODO 5: Define y (convert the multi-hot target at idx into a torch.float32 tensor)
        y = __________________________
        return (x, y)

# Instantiate DataLoaders for streaming minibatches
# TODO 6: Define train_loader (batch_size=64, shuffle=True)
train_loader = __________________________

# TODO 7: Define val_loader (batch_size=128, shuffle=False)
val_loader = __________________________

# TODO 8: Define test_loader (batch_size=128, shuffle=False)
test_loader = __________________________

# Fetch a sample batch for dimension and shape verification
# TODO 9: Define sample_x, sample_y
sample_x, sample_y = __________________________

# TODO 10: Perform the required operation (Print batch tensor shape)
__________________________

# TODO 11: Perform the required operation (Print multi-hot label shape)
__________________________

### Task 5 : Defining Models : Architecture, Math & Limitations

####**A. [The Vanilla RNN — Architecture, Math & Limitations](https://arxiv.org/abs/1912.05911)**[Marks: 15]


Now that our GoEmotions text data is tokenized and padded into uniform tensors, we need a model capable of processing sequential data.

In this section, we will build a **Vanilla Recurrent Neural Network (RNN)** cell from scratch using PyTorch tensor operations, break down its mathematical foundations step-by-step, and explore why this classic architecture struggles with long-term memory.



1. Why Do We Need a *Recurrent* Model?

**The Problem with Standard Feed-Forward Networks**

A traditional Feed-Forward Neural Network (MLP) assumes that all inputs are independent and fixed in size. However, human language has two distinct properties:

1. **Variable Length:** A Reddit comment can be 3 words long (*"I love this!"*) or 30 words long (*"I was really hoping for a better outcome, but honestly, it isn't that bad."*).
2. **Sequential Order:** Word order fundamentally changes meaning:
* *"Not bad, actually good"* (Positive / Relief)
* *"Not good, actually bad"* (Negative / Disappointment)



If we collapse a sentence into a single static bag-of-words or average vector, we destroy word order.

![image alt](https://media.geeksforgeeks.org/wp-content/uploads/20251215090811480834/neural_network.webp)

(img src : https://www.geeksforgeeks.org/machine-learning/introduction-to-recurrent-neural-network/)

**The Solution: Memory via Recurrence**

In 1990, **Jeffrey Elman** introduced the **Simple Recurrent Network (SRN)**. Instead of taking the entire sentence all at once, an RNN processes text **one token at a time**, maintaining a **hidden state ($h_t$)** that acts as its internal memory.

At step $t$, the network reads token $x_t$ **AND** remembers what it saw previously in $h_{t-1}$ to update its memory to $h_t$.


**Recurrent Neural Network**
![image alt](https://media.geeksforgeeks.org/wp-content/uploads/20251215090149060237/nfa-.webp)
![image alt](https://media.geeksforgeeks.org/wp-content/uploads/20251215090232519391/nfa1.webp)

(img src : https://www.geeksforgeeks.org/machine-learning/introduction-to-recurrent-neural-network/)

**2. Anatomy of the Forward Pass: Connecting Math to the Diagram**

Let's break down what happens inside an RNN cell at a single timestep $t$.


![image alt](https://media.geeksforgeeks.org/wp-content/uploads/20260429164557969550/introduction_to_recurrent_neural_network.webp)

(img src : https://www.geeksforgeeks.org/machine-learning/introduction-to-recurrent-neural-network/)

**Step-by-Step Mathematical Computation**

For a sequence of input vectors $x = (x_1, x_2, \dots, x_T)$:

$$h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$$

Let's dissect each component:

1. **Input Transformation ($W_{ih} x_t + b_{ih}$):**
* $x_t$ is the vector embedding of the current word at timestep $t$.
* $W_{ih}$ (Input-to-Hidden weight matrix) projects the word embedding space into the hidden dimension.
* $b_{ih}$ is the input bias vector.


2. **Memory Propagation ($W_{hh} h_{t-1} + b_{hh}$):**
* $h_{t-1}$ is the hidden state vector from the *previous* word at timestep $t-1$.
* $W_{hh}$ (Hidden-to-Hidden weight matrix) determines how much of the old memory to carry forward.
* $b_{hh}$ is the recurrent bias vector.


3. **Combination & Non-linear Activation ($\tanh$):**
* We add the transformed input and transformed previous memory together.
* We apply the hyperbolic tangent ($\tanh$) activation function to squash all values into the range $[-1, 1]$, preventing hidden values from exploding to infinity as time steps increase.


4. **Initial Hidden State ($h_0$):**
* Before reading the first word ($t=1$), there is no previous state. We initialize $h_0$ as a vector of zeros.


**3. Weight Sharing & Parameter Breakdown**

A key innovation of RNNs is **weight sharing across time**. The exact same weight matrices ($W_{ih}$ and $W_{hh}$) are reused at timestep $1, 2, \dots, T$. This keeps the parameter count constant regardless of sequence length!

| Parameter Tensor | Shape | Function |
| --- | --- | --- |
| `W_ih` (`x2h.weight`) | `(hidden_size, input_size)` | Maps input token vector $x_t$ to hidden dimension |
| `b_ih` (`x2h.bias`) | `(hidden_size,)` | Input projection bias |
| `W_hh` (`h2h.weight`) | `(hidden_size, hidden_size)` | Maps previous hidden state $h_{t-1}$ to new hidden dimension |
| `b_hh` (`h2h.bias`) | `(hidden_size,)` | Recurrent connection bias |


#####**`RNNCell` from Scratch[15 marks]**

 Instead of using PyTorch's black-box `nn.RNN`, we will implement `RNNCell` manually using `nn.Linear` layers.

 **NOTE: DO NOT USE nn.RNN from PyTorch and and implement it from scartch. However, you can use .backward() (automatic differentiation from PyTorch) for backpropogation. If you use nn.RNN, you will be penalized.**

**Understanding the Vanilla RNN Cell (`RNNCell`)**

Before filling out the code below, familiarize yourself with these core components:

* **`RNNCell` (`nn.Module`)**: Implements a single timestep computation of a Vanilla Recurrent Neural Network .
* **`self.x2h` (Input-to-Hidden Projection)**: A linear layer (`nn.Linear(input_size, hidden_size)`) that transforms the current token embedding vector $x_t$ into the hidden dimension ($W_{ih} x_t + b_{ih}$) .
* **`self.h2h` (Hidden-to-Hidden Projection)**: A linear layer (`nn.Linear(hidden_size, hidden_size)`) that transforms the previous hidden state $h_{t-1}$ to carry memory forward ($W_{hh} h_{t-1} + b_{hh}$) .
* **`torch.tanh`**: The non-linear activation function that squashes the combined linear transformations into the range $[-1, 1]$ to prevent values from exploding .
* **Sanity Check Tensors**: Dummy batch tensors used to verify that input shapes (`[4, 128]`), previous state shapes (`[4, 256]`), and output state shapes (`[4, 256]`) match expectations .

In [ ]:

class RNNCell(nn.Module):
    """A single timestep of a Vanilla RNN Cell.

    Formula: h_t = tanh(W_ih @ x_t + b_ih + W_hh @ h_{t-1} + b_hh)
    """

    def __init__(self, input_size: int, hidden_size: int):
        """
        Args:
            input_size (int): Dimension of input token embeddings (e.g., 128).
            hidden_size (int): Dimension of hidden memory state h_t (e.g., 256).
        """
        super().__init__()
        # Input-to-Hidden projection (contains W_ih and b_ih)
        # TODO 1: Define self.x2h
        self.x2h = __________________________

        # Hidden-to-Hidden projection (contains W_hh and b_hh)
        # TODO 2: Define self.h2h
        self.h2h = __________________________

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        """Processes a single timestep t.

        Args:
            x_t (torch.Tensor): Current token embeddings of shape [batch_size, input_size]
            h_prev (torch.Tensor): Previous hidden state of shape [batch_size, hidden_size]

        Returns:
            torch.Tensor: Updated hidden state h_t of shape [batch_size, hidden_size]
        """
        # Sum the linear transformations and squash through tanh
        # TODO 3: Define h_t
        h_t = __________________________
        return h_t


# --- Quick Sanity Check ---
# TODO 4: Define batch_size, input_dim, hidden_dim
batch_size, input_dim, hidden_dim = __________________________

# TODO 5: Define cell
cell = __________________________
# TODO 6: Define dummy_x_t
dummy_x_t = __________________________
# TODO 7: Define dummy_h_0
dummy_h_0 = __________________________

# TODO 8: Define dummy_h_1
dummy_h_1 = __________________________

# TODO 9: Perform the required operation (Print input tensor shape)
__________________________
# TODO 10: Perform the required operation (Print previous hidden state shape)
__________________________
# TODO 11: Perform the required operation (Print output hidden state shape)
__________________________


> ***[!TIP]***
> **Learning & Experimentation Note (Crucial for Viva & Subjective Understanding)**
> For a deeper conceptual understanding of how models evolve from problems to solutions, try this **hands-on experimentation phase** before applying the fixes:
> 1. **Train a Vanilla RNN *without* gradient clipping** alongside basic LSTM and GRU models.
> 2. **Plot your gradient norms or loss curves** over time (e.g., tracking the backward pass gradient magnitudes using a simple `matplotlib` plot or printing epoch-wise gradient norms) so you can actually *see* **vanishing gradients** (norms decaying flat to zero) and **exploding gradients** (wild spikes or sudden `NaN` losses).
> 3. **Why this matters:** Experiencing these failure modes firsthand is the ultimate preparation for your **viva** and subjective questions. It transforms memorization into deep intuition—helping you explain the exact *why* and *how* behind the transition from basic recurrent layers to clipping and gated architectures.
>
>
> Once you have observed these bottlenecks in action, proceed with the code blocks and concepts below to implement the solutions!

##### Why Vanilla RNNs Fail: The Vanishing & Exploding Gradient Problem

While conceptually elegant, vanilla RNNs suffer from a major limitation during training known as **Backpropagation Through Time (BPTT)**.

**Understanding BPTT Mathematically**

When calculating how much the loss at timestep $T$ (e.g., word 30) depends on the hidden state at timestep 1 (word 1), we use the chain rule:

$$\frac{\partial h_T}{\partial h_1} = \prod_{t=2}^{T} \frac{\partial h_t}{\partial h_{t-1}} = \prod_{t=2}^{T} \text{diag}\left(1 - \tanh^2(\cdot)\right) \cdot W_{hh}^T$$


![img alt](https://media.geeksforgeeks.org/wp-content/uploads/20251215090316988911/frame_3252.webp)


(img src : [https://www.geeksforgeeks.org/machine-learning/introduction-to-recurrent-neural-network/](https://www.geeksforgeeks.org/machine-learning/introduction-to-recurrent-neural-network/))

Notice that calculating this gradient requires multiplying by the weight matrix $W_{hh}$ **$T-1$ times in a row**.

1. **Vanishing Gradients ($\Vert{}W_{hh}\Vert{} < 1$ or $\tanh' < 1$):**

* If you multiply numbers smaller than 1 repeatedly ($0.9^{30} \approx 0.04$), the gradient exponentially decays to **zero**.
* **Consequence:** The model "forgets" distant context. Information from the beginning of a sentence cannot influence predictions at the end.

2. **Exploding Gradients ($\Vert{}W_{hh}\Vert{} > 1$):**

* If the weights are larger than 1 ($1.5^{30} \approx 191,751$), the gradient exponentially explodes to **infinity** (resulting in `NaN` losses).
* **Consequence:** Unstable, wildly fluctuating training steps.

**How Do We Fix This?**

Because vanishing and exploding gradients stem from different mathematical root causes, they require distinct solutions:

1. **Fixing Exploding Gradients with Gradient Clipping:**
* When gradients grow uncontrollably large and cause `NaN` losses, we apply **Gradient Clipping** (e.g., `torch.nn.utils.clip_grad_norm_`). This caps the maximum norm of the gradients (such as `max_norm=5.0`), forcefully scaling them down if they exceed the threshold.


2. **Fixing Vanishing Gradients with LSTMs and GRUs:**
* Gradient clipping *cannot* solve vanishing gradients. To prevent early context from decaying to zero over long sequences, modern architectures like **LSTM (Long Short-Term Memory)** and **GRU (Gated Recurrent Unit)** replace purely multiplicative weight chains with **gated additive pathways**, allowing gradients to flow unimpeded.



*(Note: Features like **Bidirectionality** do not fix vanishing or exploding gradients; rather, they are used to grant the network access to future context by processing sequences in both forward and backward directions).*



#####**Handling Gradient Issues in Vanilla RNNs[15 marks]**
[Fixing Exploding Gradients with Gradient Clipping](https://www.geeksforgeeks.org/deep-learning/vanishing-and-exploding-gradients-problems-in-deep-learning/):

When gradients grow uncontrollably large and cause NaN losses during backpropagation through time, we apply Gradient Clipping (using torch.nn.utils.clip_grad_norm_). This caps the maximum norm of the gradients (such as max_norm=5.0), forcefully scaling them down if they exceed the threshold.

[Fixing Vanishing Gradients with LSTMs and GRUs](https://www.geeksforgeeks.org/deep-learning/vanishing-and-exploding-gradients-problems-in-deep-learning/):

Gradient clipping cannot solve vanishing gradients. To prevent early context from decaying to zero over long sequences, modern architectures like LSTM (Long Short-Term Memory) and GRU (Gated Recurrent Unit)—which we will implement in upcoming sections—replace purely multiplicative weight chains with gated additive pathways, allowing gradients to flow unimpeded.

**Before filling out the code below, familiarize yourself with these core concepts:**

* **Backpropagation Through Time (BPTT)**: The process of unrolling a recurrent neural network across multiple timesteps ($T$) to compute gradients with respect to shared weights using standard `loss.backward()`.
* **Exploding Gradients**: When sequence lengths or weight matrices are large, repeated multiplications during BPTT can cause gradient values to blow up exponentially, resulting in `NaN` losses.
* **From-Scratch Gradient Clipping (Do Not use`torch.nn.utils.clip_grad_norm_`)**: Instead of using built-in utilities, you compute the global L2 norm of all parameter gradients manually: $\|\mathbf{g}\|_2 = \sqrt{\sum_{p} \|\mathbf{g}_p\|_2^2}$. If $\|\mathbf{g}\|_2 > \text{max\_norm}$, all gradients are scaled down by $\frac{\text{max\_norm}}{\|\mathbf{g}\|_2}$ to preserve training stability.
* **Optimization Flow**: Standard training loop steps (`zero_grad()`, forward pass, loss calculation, `backward()`, custom gradient clipping, and `optimizer.step()`).

In [ ]:

# --- Training Demonstration with From-Scratch Gradient Clipping for Exploding Gradients ---
batch_size, input_dim, hidden_dim = 4, 128, 256
cell = RNNCell(input_size=input_dim, hidden_size=hidden_dim)

# TODO 1: Define the optimizer using torch.optim.Adam with cell parameters and learning rate 1e-3
optimizer = __________________________

# TODO 2: Define the loss criterion using nn.MSELoss
criterion = __________________________

# Simulate a sequence forward pass over time (BPTT)
seq_len = 5
inputs = [torch.randn(batch_size, input_dim) for _ in range(seq_len)]
target = torch.randn(batch_size, hidden_dim)

optimizer.zero_grad()

# TODO 3: Initialize the initial hidden state h_0 to zeros of shape [batch_size, hidden_dim]
h = __________________________

# TODO 4: Write the sequence loop to unroll the RNN cell over time across inputs
for x_t in inputs:
    h = __________________________

# TODO 5: Compute the loss between the final hidden state h and the target using criterion
loss = __________________________

# TODO 6: Perform the backward pass via BPTT to compute gradients using loss.backward() (Do NOT use built-in clipping utilities here)
__________________________

# Apply From-Scratch Gradient Clipping to handle Exploding Gradients safely (WITHOUT torch.nn.utils.clip_grad_norm_)
max_norm = 5.0

# TODO 7: Compute the global L2 norm of all gradients across cell.parameters() from scratch
total_norm = __________________________

# TODO 8: Calculate the scaling coefficient and scale parameter gradients in-place if total_norm exceeds max_norm
__________________________

# TODO 9: Perform the optimizer step to update weights using optimizer.step()
__________________________

# TODO 10: Print gradient norm before and after clipping details for verification
__________________________

####B.  [Long Short-Term Memory (LSTM) Networks](https://deeplearning.cs.cmu.edu/S23/document/readings/LSTM.pdf) [Marks : 20]

As covered previously, **Vanilla RNNs** struggle with a major issue called **vanishing gradients** . When processing long text sequences, repeating matrix multiplications over and over causes early information to fade away to zero, preventing the network from remembering long-term context .

To fix this, Sepp Hochreiter and Jürgen Schmidhuber introduced **Long Short-Term Memory (LSTM)** networks in 1997 .

---

**1. The Core Idea: The "Information Superhighway"**

To picture how an LSTM works, think of a long road trip :
* **In a Vanilla RNN:** You have to constantly unpack and repack your entire trunk (the hidden state $h_t$) at every single stop (timestep $t$) . By the end of a long trip, your original items are either crushed, lost, or overwritten .
* **In an LSTM:** The network adds a dedicated conveyor belt running right along the top called the **Cell State ($c_t$)** . Important information can travel across this belt for many steps with very little friction or loss .

**Why Does the Cell State Prevent Vanishing Gradients?**
Look at how the cell state is updated :
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

Instead of just multiplying weights together repeatedly, the past memory ($c_{t-1}$) is updated **additively** (using addition, $+$) . During backpropagation, gradients flow smoothly through these addition steps without shrinking exponentially . This unbroken pathway is known as the **Constant Error Carousel** .

---

**2. Demystifying the 3 Gates of an LSTM**

An LSTM uses three core "valves" called **gates** to manage data flow . These gates rely on the **sigmoid activation function ($\sigma$)**, which outputs numbers strictly between $0$ and $1$ :
* **Close to 0:** "Block this information / Close the gate."
* **Close to 1:** "Let this information pass through / Open the gate."

The diagram below illustrates the overall architecture and flow of an LSTM cell

![image alt](https://media.geeksforgeeks.org/wp-content/uploads/20250528172141936296/architecture_of_lstms.webp)

(img src: https://www.geeksforgeeks.org/deep-learning/deep-learning-introduction-to-long-short-term-memory/)

**1. The Forget Gate ($f_t$): What should we drop?**
* **What it does:** Decides how much of the old cell state ($c_{t-1}$) we should throw away .
* **Analogy:** Starting a new chapter in a book and clearing out details from the previous chapter that no longer matter .
* **Formula:** $f_t = \sigma(W_{if} x_t + b_{if} + W_{hf} h_{t-1} + b_{hf})$

**2. The Input Gate ($i_t$) & Candidate Memory ($\tilde{c}_t$): What should we store?**
* **What it does:**
  * $\tilde{c}_t$ (Candidate): Builds a new set of possible memories using a $\tanh$ function (outputting values between $-1$ and $1$) .
  * $i_t$ (Input Gate): Decides which parts of that candidate memory are worth keeping .
* **Analogy:** Scanning a news feed and saving only the major breaking news items to your reading list .
* **Formulas:**
  * $i_t = \sigma(W_{ii} x_t + b_{ii} + W_{hi} h_{t-1} + b_{hi})$
  * $\tilde{c}_t = \tanh(W_{ig} x_t + b_{ig} + W_{hg} h_{t-1} + b_{hg})$

**3. Cell State Update ($c_t$): Updating the conveyor belt**
* **What it does:** Merges the old memory (filtered by the forget gate) with the new memory (filtered by the input gate) .
* **Formula:** $c_t = (f_t \odot c_{t-1}) + (i_t \odot \tilde{c}_t)$

**4. The Output Gate ($o_t$) & Hidden State ($h_t$): What should we output?**
* **What it does:** Takes the updated cell state ($c_t$), applies a $\tanh$ function, and multiplies it by the output gate ($o_t$) to create the final hidden state ($h_t$) .
* **Analogy:** Having a rich personal database in your head ($c_t$), but only speaking out loud the specific answer ($h_t$) needed for the current question .
* **Formulas:**
  * $o_t = \sigma(W_{io} x_t + b_{io} + W_{ho} h_{t-1} + b_{ho})$
  * $h_t = o_t \odot \tanh(c_t)$

---

**3. Parameter Packing for Efficiency**

Computing four separate matrix operations step-by-step for $f_t$, $i_t$, $\tilde{c}_t$, and $o_t$ runs slowly on GPUs . To speed things up, LSTMs perform **one single wide linear projection** of size `4 * hidden_size`, and then split the result into four clean pieces using `.chunk(4, dim=-1)` .

**Parameter Comparison: Vanilla RNN vs. LSTM**

| Architecture | Total Weight Matrices | Parameter Count (per layer) |
| :--- | :--- | :--- |
| **Vanilla RNN** | 2 (`W_ih`, `W_hh`) | $H \times I + H \times H + 2H$  |
| **LSTM** | 8 (4 packed into 2) | $4 \times (H \times I + H \times H + 2H)$  |

*(Here, $I$ = `input_size` and $H$ = `hidden_size`) . Because of this structure, an LSTM carries **4 times** as many parameters as a Vanilla RNN of the same hidden dimension* .

The detailed internal operation of these gates within an LSTM cell can be visualized below:
![img alt](https://media.geeksforgeeks.org/wp-content/uploads/20250404172141987003/gate_of_lstm.webp)


(img src : https://www.geeksforgeeks.org/deep-learning/deep-learning-introduction-to-long-short-term-memory/)

#####**PyTorch Implementation: `LSTMCell` from Scratch**

Before filling out the code below, familiarize yourself with these core components:

* **Donot use nn.lstm**

* **`LSTMCell` (`nn.Module`)**: Implements a single timestep of an LSTM, utilizing an additive cell state conveyor belt ($c_t$) to prevent vanishing gradients.
* **Packed Linear Projections (`self.x2g`, `self.h2g`)**: Instead of computing 4 separate weight matrices for the gates sequentially, PyTorch packs them into single wide linear layers of size `4 * hidden_size` (representing the forget, input, candidate, and output gates: $f, i, \tilde{c}, o$).
* **Chunking (`.chunk(4, dim=-1)`)**: Splits the unified gate pre-activation tensor into four equal parts along the last dimension.
* **Gate Activations**:
  * `torch.sigmoid` is applied to the forget ($f$), input ($i$), and output ($o$) gates to output values between $0$ and $1$.
  * `torch.tanh` is applied to the candidate state ($\tilde{c}$) to bound values between $-1$ and $1$.
* **Cell State & Hidden State Update**: Computes the additive cell state update ($c_t = f_t \odot c_{prev} + i_t \odot \tilde{c}_t$) and the filtered hidden state ($h_t = o_t \odot \tanh(c_t)$).

In [ ]:


class LSTMCell(nn.Module):
    """A single timestep of an LSTM Cell.

    Uses packed linear transformations for efficiency.
    """

    def __init__(self, input_size: int, hidden_size: int):
        """
        Args:
            input_size (int): Dimension of input token embeddings.
            hidden_size (int): Dimension of hidden state h_t and cell state c_t.
        """
        super().__init__()
        # TODO 1: Define self.hidden_size
        self.hidden_size = __________________________

        # Packed Input-to-Gates Linear Layer (computes f, i, g, o at once)
        # TODO 2: Define self.x2g
        self.x2g = __________________________

        # Packed Recurrent-to-Gates Linear Layer (computes f, i, g, o at once)
        # TODO 3: Define self.h2g
        self.h2g = __________________________

    def forward(self, x_t: torch.Tensor, state: tuple) -> tuple[torch.Tensor, torch.Tensor]:
        """Processes a single timestep t.

        Args:
            x_t (torch.Tensor): Tensor of shape [batch_size, input_size]
            state (tuple): Pair of (h_prev, c_prev), each of shape [batch_size, hidden_size]

        Returns:
            tuple: (h_t, c_t), each of shape [batch_size, hidden_size]
        """
        # TODO 4: Define h_prev, c_prev by unpacking the state tuple
        h_prev, c_prev = __________________________

        # 1. Compute pre-activations for all 4 gates in a single matrix operation
        # TODO 5: Define gates
        gates = __________________________

        # 2. Split the packed gates into 4 equal chunks
        # TODO 6: Define f_pre, i_pre, g_pre, o_pre
        f_pre, i_pre, g_pre, o_pre = __________________________

        # 3. Apply appropriate activation functions
        # TODO 7: Define f_t (forget gate)
        f_t = __________________________

        # TODO 8: Define i_t (input gate)
        i_t = __________________________

        # TODO 9: Define g_t (candidate cell state)
        g_t = __________________________

        # TODO 10: Define o_t (output gate)
        o_t = __________________________

        # 4. Perform additive cell state update
        # TODO 11: Define c_t
        c_t = __________________________

        # 5. Compute hidden state
        # TODO 12: Define h_t
        h_t = __________________________

        return (h_t, c_t)


# --- Quick Sanity Check ---
# TODO 13: Define batch_size, input_dim, hidden_dim
batch_size, input_dim, hidden_dim = __________________________

# TODO 14: Define lstm_cell
lstm_cell = __________________________

# TODO 15: Define dummy_x_t
dummy_x_t = __________________________

# TODO 16: Define dummy_h_0
dummy_h_0 = __________________________

# TODO 17: Define dummy_c_0
dummy_c_0 = __________________________

# TODO 18: Define dummy_h_1, dummy_c_1 by passing inputs through lstm_cell
dummy_h_1, dummy_c_1 = __________________________

# TODO 19: Perform the required operation (Print input shape)
__________________________

# TODO 20: Perform the required operation (Print new hidden state shape)
__________________________

# TODO 21: Perform the required operation (Print new cell state shape)
__________________________

##### **Summary: Trade-Offs of the LSTM**

**Advantages:**

* **Solves Vanishing Gradients:** Additive cell state updates allow long-term memory to persist.


* **Selective Retention:** Internal gates learn what to keep, forget, or expose automatically.



**Trade-Offs:**

* **4x Parameters:** Requires significantly more GPU memory and compute overhead than a Vanilla RNN.


* **Slower Training:** Additional gate operations and calculations create heavier sequential bottlenecks.



---

While LSTMs successfully solve the vanishing gradient problem of vanilla RNNs, their heavy parameter footprint and slower training speeds highlight the need for a more streamlined design. In the next section, we will explore the **Gated Recurrent Unit (GRU)**, which simplifies the LSTM architecture while retaining its robust gradient flow.


#### **C. [Gated Recurrent Unit (GRU)[15 marks]](https://d2l.ai/chapter_recurrent-modern/gru.html)**

In the previous section, we saw how **LSTMs** solve the vanishing gradient problem using a dedicated cell state ($c_t$) and three separate gates. While effective, LSTMs come with a heavy performance cost: **four times the parameters** of a Vanilla RNN, which leads to higher GPU memory usage and slower training times.

To streamline this design, **Cho et al. (2014)** introduced the **Gated Recurrent Unit (GRU)**. In this module, we will explore how a GRU achieves gradient flow similar to an LSTM while using **fewer parameters and a simpler architecture**.

---

**1. The Core Idea: Simplifying the LSTM**

The GRU simplifies the recurrent cell through two major changes:

1. **No Separate Cell State:** The GRU drops the separate cell state ($c_t$) entirely. Instead, the hidden state ($h_t$) acts as both the internal memory and the final output.
2. **Merged Gates:** Rather than using three separate gates, the GRU relies on **two gates**:
   * **Update Gate ($z_t$):** Combines the roles of the LSTM's forget and input gates to decide how much old memory to keep versus how much new memory to add.
   * **Reset Gate ($r_t$):** Determines how much of the past memory to ignore when computing the new candidate state.

The overall layout and flow of a GRU cell can be seen below:

![GRU Architecture](https://media.geeksforgeeks.org/wp-content/uploads/20251215093334420369/how_gru_works_.webp)  
*(Source: [GeeksforGeeks - Gated Recurrent Unit Networks](https://www.geeksforgeeks.org/machine-learning/gated-recurrent-unit-networks/))*

---

**2. Dissecting the GRU Equations**

Let's break down how a GRU processes input $x_t$ and the previous hidden state $h_{t-1}$ at timestep $t$:

![GRU Internal Structure and Gates](https://media.geeksforgeeks.org/wp-content/uploads/20260501104325986090/structure-of-GRU.webp)  
*(Source: [GeeksforGeeks - Gated Recurrent Unit Networks](https://www.geeksforgeeks.org/machine-learning/gated-recurrent-unit-networks/))*

**1. The Update Gate ($z_t$): Balancing Old vs. New**
* **Role:** Controls the balance between remembering past context and accepting new input.
* **Formula:** $z_t = \sigma(W_{iz} x_t + b_{iz} + W_{hz} h_{t-1} + b_{hz})$
* **Interpretation:**
  * If $z_t \approx 1$: Keep the old hidden state $h_{t-1}$ (great for long-term memory).
  * If $z_t \approx 0$: Replace it with the new candidate state.

**2. The Reset Gate ($r_t$): Filtering Past Memory**
* **Role:** Decides how much of the previous hidden state to drop when forming new memories.
* **Formula:** $r_t = \sigma(W_{ir} x_t + b_{ir} + W_{hr} h_{t-1} + b_{hr})$
* **Interpretation:**
  * If $r_t \approx 0$: Ignore past history completely and treat the current input like a fresh start (useful at sentence boundaries or topic shifts).

**3. Candidate Hidden State ($\tilde{h}_t$): Preparing New Content**
* **Role:** Creates a candidate update vector by applying the reset gate to filter past context.
* **Formula:** $\tilde{h}_t = \tanh(W_{in} x_t + b_{in} + r_t \odot (W_{hn} h_{t-1} + b_{hn}))$

**4. Final Hidden State Update ($h_t$): Linear Interpolation**
* **Role:** Blends the old state $h_{t-1}$ and the new candidate state $\tilde{h}_t$ using the update gate $z_t$ as a smooth blending factor.
* **Formula:** $h_t = (1 - z_t) \odot \tilde{h}_t + z_t \odot h_{t-1}$

---

**3. Parameter Comparison: RNN vs. LSTM vs. GRU**

For a model with `input_size` $I$ and `hidden_size` $H$:

| Architecture | Number of Gates | Linear Layers per Cell | Relative Parameter Count |
| :--- | :--- | :--- | :--- |
| **Vanilla RNN** | 0 | 2 (`x2h`, `h2h`) | **1.0x** (Baseline) |
| **LSTM** | 3 (+ Candidate) | 8 (Packed into 2 wide layers) | **4.0x** |
| **GRU** | 2 (+ Candidate) | 6 (3 input-side, 3 hidden-side) | **3.0x** (25% fewer than LSTM) |

Because GRUs have **25% fewer parameters** than LSTMs, they compute faster, consume less memory, and are less prone to overfitting on smaller datasets.

#####**PyTorch Implementation: `GRUCell` from Scratch[15 Marks]**

To keep the logic clear, we implement the GRU cell using three distinct pairs of linear layers

Before filling out the code below, familiarize yourself with these core components:
* **Donot use nn.GRU**
* **`GRUCell` (`nn.Module`)**: Implements a single timestep computation of a Gated Recurrent Unit, combining the update and reset gates to manage memory efficiently without a separate cell state.
* **Update Gate Projections (`self.x2z`, `self.h2z`)**: Linear layers that compute the update gate ($z_t$), controlling how much of the old hidden state to retain versus update.
* **Reset Gate Projections (`self.x2r`, `self.h2r`)**: Linear layers that compute the reset gate ($r_t$), deciding how much past memory to forget when forming the new candidate state.
* **Candidate State Projections (`self.x2n`, `self.h2n`)**: Linear layers that compute the candidate hidden state ($n_t$), using the reset gate to selectively mask past context.
* **Linear Interpolation ($h_t$)**: Blends the old hidden state and new candidate state smoothly using $(1 - z_t) \odot n_t + z_t \odot h_{prev}$.

In [ ]:


class GRUCell(nn.Module):
    """A single timestep of a Gated Recurrent Unit (GRU) Cell.

    Formula:
        z_t = sigmoid(W_iz @ x_t + b_iz + W_hz @ h_{t-1} + b_hz)
        r_t = sigmoid(W_ir @ x_t + b_ir + W_hr @ h_{t-1} + b_hr)
        n_t = tanh(W_in @ x_t + b_in + r_t * (W_hn @ h_{t-1} + b_hn))
        h_t = (1 - z_t) * n_t + z_t * h_{t-1}
    """

    def __init__(self, input_size: int, hidden_size: int):
        """
        Args:
            input_size (int): Dimension of input token embeddings.
            hidden_size (int): Dimension of the hidden memory state h_t.
        """
        super().__init__()
        # TODO 1: Define self.hidden_size
        self.hidden_size = __________________________

        # Update Gate (z) Linear Layers
        # TODO 2: Define self.x2z
        self.x2z = __________________________
        # TODO 3: Define self.h2z
        self.h2z = __________________________

        # Reset Gate (r) Linear Layers
        # TODO 4: Define self.x2r
        self.x2r = __________________________
        # TODO 5: Define self.h2r
        self.h2r = __________________________

        # Candidate State (n) Linear Layers
        # TODO 6: Define self.x2n
        self.x2n = __________________________
        # TODO 7: Define self.h2n
        self.h2n = __________________________

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        """Processes a single timestep t.

        Args:
            x_t (torch.Tensor): Current input tensor of shape [batch_size, input_size]
            h_prev (torch.Tensor): Previous hidden state of shape [batch_size, hidden_size]

        Returns:
            torch.Tensor: Updated hidden state h_t of shape [batch_size, hidden_size]
        """
        # 1. Compute Update Gate (z_t) and Reset Gate (r_t) using sigmoid
        # TODO 8: Define z_t
        z_t = __________________________

        # TODO 9: Define r_t
        r_t = __________________________

        # 2. Compute Candidate State (n_t), masking previous state with reset gate
        # TODO 10: Define n_t
        n_t = __________________________

        # 3. Interpolate between past state and new candidate state
        # TODO 11: Define h_t
        h_t = __________________________

        return h_t


# --- Quick Sanity Check ---
# TODO 12: Define batch_size, input_dim, hidden_dim
batch_size, input_dim, hidden_dim = __________________________

# TODO 13: Define gru_cell
gru_cell = __________________________

# TODO 14: Define dummy_x_t
dummy_x_t = __________________________

# TODO 15: Define dummy_h_0
dummy_h_0 = __________________________

# TODO 16: Define dummy_h_1 by passing inputs through gru_cell
dummy_h_1 = __________________________

# TODO 17: Perform the required operation (Print input tensor shape)
__________________________

# TODO 18: Perform the required operation (Print previous hidden state shape)
__________________________

# TODO 19: Perform the required operation (Print output hidden state shape)
__________________________

##### Architectural Limitations of Single-Layer Recurrent Cells

Whether using an `RNNCell`, `LSTMCell`, or `GRUCell`, single-layer unidirectional cells share two primary structural limitations:

1. **Limited Hierarchical Capacity:** A single recurrent layer can only construct one level of temporal abstraction. Stacking multiple recurrent layers (Deep RNNs) enables higher layers to learn much richer, hierarchical sequence representations.
2. **Unidirectional Context:** At any timestep $t$, the hidden state $h_t$ has only processed past tokens ($x_1, \dots, x_t$) and remains completely blind to future context ($x_{t+1}, \dots, x_T$). For tasks where the entire text sequence is available upfront—such as text classification—processing the sequence in both forward and backward directions via **Bidirectional Recurrent Networks** captures significantly richer contextual patterns.

*For a deeper empirical comparison between these architectures, see the seminal paper: [Empirical Evaluation of Gated Recurrent Neural Networks on Sequence Modeling (Chung et al., 2014)](https://arxiv.org/abs/1412.3555).*

###**Task 6 : Architectural Scaling: Depth and Bidirectionality**

Now, we are going to assemble the individual recurrent cells built so far into full-fledged sequence encoders. In doing so, we will solve two major structural limitations of basic recurrent networks: **Depth (Stacking)** and **Context Direction (Bidirectionality)**.

By the end of this module, you will build a single, modular `RecurrentEncoder` class capable of powering all **9 model configurations** for our GoEmotions multi-label emotion prediction task.

**The 9 Model Configurations**

To thoroughly evaluate our recurrent architectures for the GoEmotions task, our modular `RecurrentEncoder` will dynamically build **9 distinct configurations** by combining 3 cell types with 3 structural variations:

| Configuration ID | Cell Type | Layers (Depth) | Directionality |
| :--- | :--- | :--- | :--- |
| **Config 1** | Vanilla RNN | Single Layer ($L=1$) | Unidirectional |
| **Config 2** | Vanilla RNN | Single Layer ($L=1$) | Bidirectional |
| **Config 3** | Vanilla RNN | Multi-Layer ($L=2$) | Bidirectional |
| **Config 4** | LSTM | Single Layer ($L=1$) | Unidirectional |
| **Config 5** | LSTM | Single Layer ($L=1$) | Bidirectional |
| **Config 6** | LSTM | Multi-Layer ($L=2$) | Bidirectional |
| **Config 7** | GRU | Single Layer ($L=1$) | Unidirectional |
| **Config 8** | GRU | Single Layer ($L=1$) | Bidirectional |
| **Config 9** | GRU | Multi-Layer ($L=2$) | Bidirectional |

By writing a single parameterized class, we can easily toggle between these options to compare how depth, gates, and bidirectional context impact our 28-dimensional emotion prediction performance.

**Stacking Recurrent Layers (Multi-Layer Depth)**

**Why Do We Need Depth?**

A single-layer recurrent network is the sequence equivalent of a 1-layer MLP. While it can process text, it can only learn simple, low-level representations.

To capture complex, hierarchical language abstractions, we **stack** recurrent layers on top of each other. In a stacked network, **the sequence of hidden states generated by Layer 1 becomes the input sequence fed into Layer 2**:

$$h_t^{(l)} = \text{Cell}^{(l)}\left(h_t^{(l-1)}, h_{t-1}^{(l)}\right), \qquad \text{where } h_t^{(0)} = x_t \text{ (input embedding)}$$

* **Important Note:** Each layer maintains its **own independent set of weights**. Layer 2 does not share parameters with Layer 1.

![Stacked RNN Architecture](https://media.geeksforgeeks.org/wp-content/uploads/20231006110426/1.png)  
*(Source: [GeeksforGeeks - Stacked RNNs in NLP](https://www.geeksforgeeks.org/nlp/stacked-rnns-in-nlp/))*

**What Stacking *Doesn't* Fix**

Stacking makes your network deeper, but each layer remains strictly **unidirectional**. Layer $l$'s state at timestep $t$ only knows about past words ($x_1 \dots x_t$) and has zero visibility into what comes later in the sentence ($x_{t+1} \dots x_T$). To fix that, we need **bidirectionality**.

**Bidirectionality (Reading Forward and Backward)**

Why Do We Need Bidirectionality?

In emotion prediction, sentence order matters, but so does full context. Consider these two comments:
* *"I thought the movie was going to be terrible, but it was actually **amazing**!"*
* *"I thought the movie was going to be **amazing**, but it was actually terrible!"*

A left-to-right model processing word 5 (*"terrible"*) doesn't yet know the plot twist at the end. **Schuster & Paliwal (1997)** solved this by introducing the **Bidirectional Recurrent Neural Network**, which runs two separate passes:

1. **Forward Pass ($\overrightarrow{h}_t$):** Processes text from start to end ($t = 1 \dots T$).
2. **Backward Pass ($\overleftarrow{h}_t$):** Processes text from end to start ($t = T \dots 1$).

At each position $t$, we concatenate both hidden vectors to form a complete representation:
$$h_t = \left[ \overrightarrow{h}_t \;;\; \overleftarrow{h}_t \right] \in \mathbb{R}^{2 \cdot \text{hidden_size}}$$

![Bidirectional RNN Flow](https://media.geeksforgeeks.org/wp-content/uploads/20260519103233137475/inputs.webp)  
*(Source: [GeeksforGeeks - Bidirectional Recurrent Neural Network](https://www.geeksforgeeks.org/deep-learning/bidirectional-recurrent-neural-network/))*



> **Note: Emotion Prediction vs. Generation (The Bidirectionality Rule)**
> Bidirectionality can only be used when the entire input sequence is available upfront—such as in our GoEmotions multi-label classification task. For text generation tasks (like machine translation or language modeling), future tokens are unknown *a priori*, making bidirectional reading impossible.
> *Keep this golden rule in mind:* **Encoders can be bidirectional; decoders MUST be unidirectional.**

####**Putting It Together: A Generic `RecurrentEncoder`[Marks :45]**

Now, we will build a single, flexible **`RecurrentEncoder`** module capable of instantiating any combination of cell type, depth, and bidirectionality for our emotion prediction model.

Before filling out the code below, familiarize yourself with these core components:

Modular Recurrent Encoder Architecture: Designed to flexibly build custom multi-layer, optionally bidirectional sequence encoders using from-scratch recurrent cell implementations (RNNCell, LSTMCell, or GRUCell).  

Explicit Time-Step Unrolling (_run_direction): Unrolls recurrent cells across individual sequence timesteps to manually build the autograd graph required for Backpropagation Through Time (BPTT).

Directional Context Handling: Manages independent forward and backward cell passes for bidirectional configurations, subsequently concatenating their hidden representations to capture comprehensive sequence context.  

Hierarchical Layer Depth: Organizes stacked layers such that the sequence output tensor of layer $l$ serves as the input token stream for layer $l+1$.

In [ ]:
# Build one cell per (layer, direction)
# Layer 0 takes raw token embeddings; upper layers take concatenated outputs of layer below
# Initial state h_0 = 0 is the base case: before the first token, the encoder has no prior sequence information.
# This explicit unroll builds an autograd graph across time; loss.backward() later performs BPTT through these shared-cell steps.
# Place hidden state back in its correct chronological slot
# Stack outputs along sequence dimension [batch_size, seq_len, hidden_size]
# Extract hidden state vector h (ignoring cell state c for LSTMs)
# Concatenate direction outputs -> forms the input to the next layer

class RecurrentEncoder(nn.Module):
    'A generic, multi-layer, optionally-bidirectional recurrent encoder.\n\n    Compatible with from-scratch RNNCell, LSTMCell, and GRUCell implementations.\n    '
    def __init__(self, cell_cls: type, input_size: int, hidden_size: int, num_layers: int=1, bidirectional: bool=False, is_lstm: bool=False):
        '\n        Args:\n            cell_cls (type): Class reference (RNNCell, LSTMCell, or GRUCell).\n            input_size (int): Dimension of input embeddings.\n            hidden_size (int): Hidden state dimension for every cell.\n            num_layers (int): Number of stacked layers (default: 1).\n            bidirectional (bool): If True, runs forward and backward passes (default: False).\n            is_lstm (bool): Set to True for LSTMCell to handle the (h, c) tuple state.\n        '
        super().__init__()
        # TODO 1: Define self.num_layers storing the total number of stacked recurrent layers
        self.num_layers = __________________________

        # TODO 2: Define self.bidirectional boolean flag indicating whether the encoder runs bidirectionally
        self.bidirectional = __________________________

        # TODO 3: Define self.hidden_size storing the hidden dimension size per cell
        self.hidden_size = __________________________

        # TODO 4: Define self.is_lstm boolean flag tracking if the recurrent cell uses an LSTM tuple state (h, c)
        self.is_lstm = __________________________

        # TODO 5: Define self.num_directions as 2 if bidirectional is True else 1
        self.num_directions = __________________________

        # TODO 6: Initialize self.cells as an nn.ModuleList containing distinct cell instances for every layer and direction combination
        self.cells = __________________________

        # TODO 7: Iterate over the range of num_layers to instantiate cells layer by layer
        for l in __________________________:
            # TODO 8: Define in_dim dynamically (input_size for layer 0, hidden_size * num_directions for subsequent layers)
            in_dim = __________________________

            # TODO 9: Iterate over direction range (2 if bidirectional else 1)
            for _dir in __________________________:
                # TODO 10: Instantiate and append the recurrent cell (cell_cls(in_dim, hidden_size)) to self.cells
                __________________________

    def _run_direction(self, cell: nn.Module, x: torch.Tensor, reverse: bool):
        'Unrolls a single cell over a sequence in one direction.'
        # TODO 11: Unpack batch_size and seq_len dimensions from input tensor x shape [batch_size, seq_len, input_size]
        batch_size, seq_len, _ = __________________________

        # TODO 12: Retrieve computation device from input tensor x device property
        device = __________________________

        # TODO 13: Initialize initial hidden state tensor h filled with zeros of shape [batch_size, hidden_size] on device
        h = __________________________

        # TODO 14: Initialize initial cell state tensor c filled with zeros if is_lstm is True else None
        c = __________________________

        # TODO 15: Define steps sequence order (reversed range if reverse=True else standard range)
        steps = __________________________

        # TODO 16: Initialize outputs list or tensor container to store hidden states across sequence steps
        outputs = __________________________

        # TODO 17: Iterate through sequence timesteps t based on steps ordering
        for t in __________________________:
            # TODO 18: Extract current input token embedding vector x_t at timestep t from tensor x [batch_size, input_size]
            x_t = __________________________

            # TODO 19: Check if is_lstm is True to handle tuple state propagation (h, c)
            if __________________________:
                # TODO 20: Pass current input and previous state tuple (h, c) through cell, updating h and c
                h, c = __________________________
            else:
                # TODO 21: Pass current input and previous hidden state h through cell, updating h
                h = __________________________

            # TODO 22: Place updated hidden state into its correct chronological slot in the outputs container
            outputs[t] = __________________________

        # TODO 23: Stack outputs list along the sequence dimension to form tensor of shape [batch_size, seq_len, hidden_size]
        stacked_outputs = __________________________

        # TODO 24: Extract final state (returning final hidden state tensor h or tuple (h, c) depending on cell type)
        final_state = __________________________
        return (stacked_outputs, final_state)

    def forward(self, x: torch.Tensor):
        '\n        Args:\n            x (torch.Tensor): Input tensor of shape [batch_size, seq_len, input_size]\n\n        Returns:\n            layer_input (torch.Tensor): Top layer outputs [batch_size, seq_len, hidden_size * num_directions]\n            final_h (torch.Tensor): Top layer final hidden state [batch_size, hidden_size * num_directions]\n        '
        # TODO 25: Initialize layer_input variable starting with raw embedding tensor x
        layer_input = __________________________

        # TODO 26: Initialize cell_idx counter variable tracking position inside self.cells ModuleList to 0
        cell_idx = __________________________

        # TODO 27: Initialize final_h list container to store final hidden states across all layers
        final_h = __________________________

        # TODO 28: Iterate through each stacked layer index l in range(num_layers)
        for l in __________________________:
            # TODO 29: Initialize empty lists dir_outputs and dir_finals to collect forward and backward passes for layer l
            dir_outputs, dir_finals = __________________________

            # TODO 30: Iterate through each direction index d in range(num_directions)
            for d in __________________________:
                # TODO 31: Retrieve specific recurrent cell from self.cells using cell_idx
                cell = __________________________

                # TODO 32: Increment cell_idx counter to point to the next cell for subsequent iterations
                cell_idx = __________________________

                # TODO 33: Define boolean flag is_reverse as True if bidirectional and direction index d == 1 else False
                is_reverse = __________________________

                # TODO 34: Execute _run_direction helper method passing cell, layer_input, and is_reverse flag
                out, final = __________________________

                # TODO 35: Append directional sequence outputs tensor out into dir_outputs list
                __________________________

                # TODO 36: Extract final hidden state vector h from final (ignoring cell state c if LSTM)
                h_final = __________________________

                # TODO 37: Append directional final hidden state into dir_finals list
                __________________________

            # TODO 38: Concatenate direction outputs along feature dimension (dim=-1) to form input representation for the next layer
            layer_input = __________________________

            # TODO 39: Concatenate forward and backward final hidden states along feature dimension (dim=-1) to form final layer representation
            final_h = __________________________

        return (layer_input, final_h)

**BPTT and the Initial Hidden State**

For each sequence, the recurrent encoder starts with $h_0=\mathbf{0}$. This is the base case: before the first token, there is no earlier sequence information. The same cell parameters are reused at every timestep, while the hidden state carries information forward through $h_t=f_\theta(x_t,h_{t-1})$.

The explicit time-step loop records this chain of operations in PyTorch's autograd graph. When `loss.backward()` is called, PyTorch automatically applies **Backpropagation Through Time (BPTT)**: gradients flow from the final loss backward through the unrolled hidden states and accumulate on the shared cell weights. No manual BPTT loop is required; gradients are clipped afterward to control possible exploding gradients.


**Parameter Count Sanity Check**

To verify your model construction, make sure the number of instantiated cell objects matches this table:

| Configuration | Total Cells Created | Notes |
| --- | --- | --- |
| **Single-layer, Unidirectional** | $1 \times 1 = \mathbf{1}$ | 1 forward cell |
| **Stacked (2-layer), Unidirectional** | $2 \times 1 = \mathbf{2}$ | Layer 2's input dim = `hidden_size` |
| **Single-layer, Bidirectional** | $1 \times 2 = \mathbf{2}$ | Independent forward & backward cells |
| **Stacked (2-layer), Bidirectional** | $2 \times 2 = \mathbf{4}$ | Layer 2's input dim = `hidden_size * 2` |

Before filling out the code below, familiarize yourself with these core components:

* **Dimensions (`B, T, I, H`)**: Define the structural parameters for dummy tensor generation—batch size (`B`), sequence length (`T`), input embedding size (`I`), and hidden size (`H`).
* **Dummy Input Tensor (`x`)**: A randomly initialized input tensor of shape `[B, T, I]` used to test forward-pass compatibility.
* **Configuration Loop (`L, Bidir`)**: Iterates through multiple combinations of layer depth (`L`) and bidirectionality (`Bidir`) to verify that the encoder handles structural variations cleanly.
* **Direction Multiplier (`D`)**: Computes the output dimension scaling factor (`1` for unidirectional, `2` for bidirectional).
* **Encoder Forward Pass (`enc`, `out`, `h`)**: Dynamically instantiates the `RecurrentEncoder`, executes a forward pass, and validates that sequence outputs and final hidden states conform to expected shapes.
* **Architecture Summary (`torchinfo.summary`)**: Generates a detailed, layer-by-layer parameter and tensor shape table for the primary 2-layer Bidirectional GRU model.

In [ ]:
# sanity check
# Setup dummy dimensions and tensor
# Verify all 4 configurations in a compact loop
# Validate shapes and cell counts
# Print concise torchinfo table for the main 2-layer BiGRU

# TODO 1: Import the torchinfo library and summary function
# from torchinfo import summary
__________________________

# TODO 2: Define dummy batch size (B), sequence length (T), input embedding size (I), and hidden size (H) dimensions
B, T, I, H = __________________________

# TODO 3: Define random dummy input tensor x of shape [B, T, I] using torch.randn
x = __________________________

# TODO 4: Print an initial header banner indicating the start of the recurrent encoder sanity checks
__________________________

# TODO 5: Iterate over the four structural configurations (Layer counts and Bidirectionality flags)
for L, Bidir in __________________________:
    # TODO 6: Define direction multiplier D (2 if Bidir is True else 1)
    D = __________________________

    # TODO 7: Instantiate RecurrentEncoder passing GRUCell, input size I, hidden size H, num_layers=L, and bidirectional=Bidir
    enc = __________________________

    # TODO 8: Execute forward pass through enc with dummy tensor x to obtain sequence outputs out and final hidden state h
    out, h = __________________________

    # TODO 9: Assert that output tensor shape matches expected dimensions [B, T, H * D]
    __________________________

    # TODO 10: Assert that final hidden state tensor shape matches expected dimensions [B, H * D]
    __________________________

    # TODO 11: Assert that the total number of instantiated cell objects equals L * D
    __________________________

    # TODO 12: Define readable tag string representing the current configuration (Stacked/Single and Bi/Unidirectional)
    tag = __________________________

    # TODO 13: Print formatted progress line showing cell count, output shape, and hidden state shape for the current configuration
    __________________________

# TODO 14: Print separator banner and section title for the torchinfo summary table
__________________________

# TODO 15: Print concise torchinfo summary table for the 2-layer bidirectional RecurrentEncoder configuration specifying input size and column attributes
__________________________

###**Task 7 :The Emotion Classifier Pipeline[10 Marks]**

Now that we have our `RecurrentEncoder`, we wrap it in a complete classification network:

```
Token IDs [batch_size, seq_len]
    |
    v
[ nn.Embedding ]
    |
    v
[ RecurrentEncoder ]  --->  Final Hidden State [batch_size, hidden_size * num_directions]
                                |
                                v
                         [ nn.Dropout ]
                                |
                                v
                         [ nn.Linear ]
                                |
                                v
                      Logits [batch_size, 28]

```

In [ ]:
# Linear layer input size doubles if bidirectional
# 1. Embed tokens: [batch_size, seq_len, emb_dim]
# 2. Pass through Recurrent Encoder
# 3. Apply Dropout and Classification Head

class EmotionClassifier(nn.Module):
    'Complete classification model wrapping Embedding + RecurrentEncoder + Linear Head.'
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, num_classes: int, cell_cls: type, num_layers: int, bidirectional: bool, is_lstm: bool):
        super().__init__()
        # TODO 1: Define self.embedding
        self.embedding = __________________________
        # TODO 2: Define self.encoder
        self.encoder = __________________________
        # TODO 3: Define classifier_in_dim
        classifier_in_dim = __________________________
        # TODO 4: Define self.dropout
        self.dropout = __________________________
        # TODO 5: Define self.fc
        self.fc = __________________________
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '\n        Args:\n            x (torch.Tensor): Tensor of token IDs [batch_size, seq_len]\n        Returns:\n            logits (torch.Tensor): Unnormalized class scores [batch_size, num_classes]\n        '
        # TODO 6: Define embeds
        embeds = __________________________
        # TODO 7: Define _, final_hidden
        _, final_hidden = __________________________
        # TODO 8: Define logits
        logits = __________________________
        return logits


**Training & Evaluation Loop[10 marks]**

All nine configurations use identical training settings:

* **Optimizer:** Adam (<code>lr=1e-3</code>)
* **Loss:** <code>BCEWithLogitsLoss</code> for independent multi-label predictions
* **Decision rule:** <code>sigmoid(logits) >= 0.5</code>
* **Validation metric:** micro-F1 (stored as <code>val_acc</code> for compatibility with the existing comparison cells), with exact-match accuracy also reported
* **Gradient clipping:** max norm <code>5.0</code>


**Understanding the Training and Evaluation Loop**

Before filling out the code below, familiarize yourself with these core components:

* **Multi-Label Loss (`BCEWithLogitsLoss`)**: Computes independent binary cross-entropy losses for all 28 emotion categories without requiring mutually exclusive classes.
* **Decision Threshold (`MULTILABEL_THRESHOLD`)**: Determines positive emotion predictions where `sigmoid(logits) >= 0.5`.
* **Gradient Clipping**: Constrains gradient norms to a maximum value of `5.0` to prevent exploding gradients during Backpropagation Through Time (BPTT).
* **Metrics Tracking**: Tracks validation micro-F1 scores, subset accuracy (exact match), and per-label confusion matrices via `evaluate_full`.

In [ ]:


# Autograd traverses the Q1 recurrent unroll backward in time (BPTT) and accumulates gradients on shared weights.
# Clip the BPTT gradients before the optimizer update to limit exploding gradients.

# TODO 1: Import the required evaluation metrics from sklearn.metrics
# from sklearn.metrics import classification_report, multilabel_confusion_matrix
__________________________

# TODO 2: Define MULTILABEL_THRESHOLD probability cutoff for independent binary emotion decisions
MULTILABEL_THRESHOLD = __________________________

def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, epochs: int = 4, lr: float = 0.001, tag: str = '', resume: bool = True) -> dict:
    """Trains a 28-label model with BCE-with-logits and tracks multi-label metrics."""
    # TODO 3: Move model to the computation device using model.to(DEVICE)
    __________________________

    # TODO 4: Define optimizer using torch.optim.Adam with model parameters and learning rate lr
    optimizer = __________________________

    # TODO 5: Define criterion as nn.BCEWithLogitsLoss for independent multi-label classification logits
    criterion = __________________________

    # TODO 6: Initialize start_epoch tracker variable to 0
    start_epoch = __________________________

    # TODO 7: Initialize history dictionary tracking train_loss, val_loss, val_acc (micro-F1), and val_subset_acc
    history = __________________________

    # TODO 8: Check if resume is True and a non-empty tag string is provided
    if __________________________:
        # TODO 9: Load existing checkpoint epoch and history using load_checkpoint(tag, model, optimizer)
        ckpt_epoch, ckpt_history = __________________________

        # TODO 10: Check if loaded ckpt_history is not None
        if __________________________:
            # TODO 11: Check if ckpt_epoch is already greater than or equal to target epochs
            if __________________________:
                # TODO 12: Print skipping message and return ckpt_history directly
                __________________________
                return ckpt_history

            # TODO 13: Resume training progress by updating start_epoch and history from checkpoint values
            start_epoch, history = __________________________

            # TODO 14: Print resumption message indicating the resumed epoch index
            __________________________

    # TODO 15: Iterate over training epochs loop using range(start_epoch, epochs)
    for ep in __________________________:
        # TODO 16: Set model to training mode using model.train()
        __________________________

        # TODO 17: Initialize total_train_loss accumulator to 0.0
        total_train_loss = __________________________

        # TODO 18: Iterate over mini-batches (x, y) from train_loader
        for x, y in __________________________:
            # TODO 19: Move input features x and target multi-hot vectors y to DEVICE
            x, y = __________________________

            # TODO 20: Clear accumulated gradients from previous steps using optimizer.zero_grad()
            __________________________

            # TODO 21: Forward pass through model to generate prediction logits
            logits = __________________________

            # TODO 22: Compute multi-label binary cross-entropy loss between logits and targets y
            loss = __________________________

            # TODO 23: Perform backward pass to compute gradients via loss.backward()
            __________________________

            # TODO 24: Apply gradient clipping to model parameters using torch.nn.utils.clip_grad_norm_ with max_norm=5.0
            __________________________

            # TODO 25: Step the optimizer to update network weights using optimizer.step()
            __________________________

            # TODO 26: Accumulate batch training loss into total_train_loss (loss.item() * batch_size)
            total_train_loss = __________________________

        # TODO 27: Compute epoch train_loss by dividing total_train_loss by total dataset size
        train_loss = __________________________

        # TODO 28: Set model to evaluation mode using model.eval()
        __________________________

        # TODO 29: Initialize total_val_loss accumulator to 0.0 for validation phase
        total_val_loss = __________________________

        # TODO 30: Initialize true_positive, false_positive, and false_negative score accumulators to 0
        true_positive, false_positive, false_negative = __________________________

        # TODO 31: Initialize subset_correct exact match counter variable to 0
        subset_correct = __________________________

        # TODO 32: Run under torch.no_grad() context manager to disable gradient tracking during validation
        with __________________________:
            # TODO 33: Iterate over validation mini-batches (x, y) from val_loader
            for x, y in __________________________:
                # TODO 34: Move validation batch tensors (x, y) to DEVICE
                x, y = __________________________

                # TODO 35: Forward pass during validation to compute logits
                logits = __________________________

                # TODO 36: Compute validation loss using criterion
                loss = __________________________

                # TODO 37: Accumulate validation loss into total_val_loss
                total_val_loss = __________________________

                # TODO 38: Apply sigmoid and threshold to obtain binary prediction flags (preds >= MULTILABEL_THRESHOLD)
                preds = __________________________

                # TODO 39: Convert target tensor y into boolean tensor format
                targets = __________________________

                # TODO 40: Update true_positive count by summing elementwise logical AND of preds and targets
                true_positive = __________________________

                # TODO 41: Update false_positive count by summing predictions where preds are True but targets are False
                false_positive = __________________________

                # TODO 42: Update false_negative count by summing cases where targets are True but preds are False
                false_negative = __________________________

                # TODO 43: Update subset_correct exact match count where entire predicted label sets match targets across dim=1
                subset_correct = __________________________

        # TODO 44: Compute epoch val_loss by dividing total_val_loss by validation dataset length
        val_loss = __________________________

        # TODO 45: Compute denominator for micro-F1 score evaluation (2 * TP + FP + FN)
        denominator = __________________________

        # TODO 46: Compute validation micro-F1 score (2 * TP / denominator), handling zero division safely
        val_micro_f1 = __________________________

        # TODO 47: Compute validation exact-match subset accuracy (subset_correct / validation dataset length)
        val_subset_acc = __________________________

        # TODO 48: Append train_loss, val_loss, val_micro_f1, and val_subset_acc to history dictionary lists
        __________________________

        # TODO 49: Print formatted epoch performance logs showing losses, micro-F1, and exact match metrics
        __________________________

        # TODO 50: Save model checkpoint via save_checkpoint if a tag name is provided
        __________________________

        # TODO 51: Update best model tracking logic or metrics if applicable
        __________________________

        # TODO 52: Handle any additional conditional checks if required for logging or early stopping
        if __________________________:
            # TODO 53: Perform secondary conditional logging or task execution
            __________________________

        # TODO 54: Complete epoch logging block execution loop
        __________________________

    return history


@torch.no_grad()
def evaluate_full(model: nn.Module, loader: DataLoader, class_names: list):
    """Returns a per-label report and one 2x2 confusion matrix per label."""
    # TODO 55: Set model to evaluation mode using model.eval()
    __________________________

    # TODO 56: Initialize empty lists all_preds and all_true to store batch predictions and targets
    all_preds, all_true = __________________________

    # TODO 57: Iterate over batches (x, y) from evaluation loader
    for x, y in __________________________:
        # TODO 58: Move input tensor x to DEVICE
        x = __________________________

        # TODO 59: Compute thresholded binary predictions as integer numpy arrays
        preds = __________________________

        # TODO 60: Append batch predictions array to all_preds list
        __________________________

        # TODO 61: Append batch ground-truth targets array to all_true list
        __________________________

    # TODO 62: Concatenate all collected prediction arrays along axis 0 to form y_pred
    y_pred = __________________________

    # TODO 63: Concatenate all collected target arrays along axis 0 to form y_true
    y_true = __________________________

    # TODO 64: Generate per-label classification metrics report dictionary using classification_report
    report = __________________________

    # TODO 65: Generate 2x2 confusion matrices per label using multilabel_confusion_matrix
    label_confusion_matrices = __________________________

    return (report, label_confusion_matrices)

**Running the 9-Model Sweep[5 Marks]**

Now we execute the main experiment! We will train and evaluate all **9 configurations**:

$$\{\text{RNN}, \text{LSTM}, \text{GRU}\} \times \{\text{Single-Layer}, \text{Stacked (2L)}, \text{Bidirectional}\}$$

**Running the 9-Model Sweep**

Now we execute the main experiment! We will train and evaluate all **9 configurations**:

$$\{\text{RNN}, \text{LSTM}, \text{GRU}\} \times \{\text{Single-Layer}, \text{Stacked (2L)}, \text{Bidirectional}\}$$

Before running the sweep code below, familiarize yourself with these core components:

* **Experiment Matrix (`experiment_configs`)**: Defines the full grid of 9 variations, mapping human-readable tags to their respective cell classes, layer depths, and directionality flags.
* **Checkpoint Resuming**: The sweep automatically handles interruptions by checking saved states per epoch, allowing you to resume training seamlessly without restarting from scratch.
* **Immediate Submission Export**: After each finished model training run, the script evaluates test performance and retains the best-performing configuration dynamically.

In [ ]:

# --- Running the 9-Model Sweep ---
# Define all 9 experiment configurations
# Format: (Tag, Cell_Class, Num_Layers, Bidirectional, Is_LSTM)
# Train model (train_model checkpoints every epoch, so re-running this cell
# after an interruption resumes each model from its last saved epoch)
# Evaluate on test set
# Immediate submission export: after each finished model, retain the best one seen so far.

# TODO 1: Define EMB_DIM
EMB_DIM = __________________________

# TODO 2: Define HIDDEN_DIM
HIDDEN_DIM = __________________________

# TODO 3: Define EPOCHS
EPOCHS = __________________________

# TODO 4: Define NUM_CLASSES
NUM_CLASSES = __________________________

# TODO 5: Define label_names
label_names = __________________________

# TODO 6: Define experiment_configs (list of tuples for all 9 models)
experiment_configs = __________________________

def build_emotion_model(cell_cls, n_layers, is_bidir, is_lstm):
    """Rebuilds an EmotionClassifier for a given config. Reused for training AND
    for reconstructing a model to reload a checkpoint into later."""
    # TODO 7: Return the computed result (instantiated EmotionClassifier)
    return __________________________

# TODO 8: Define results dictionary
results = __________________________

# TODO 9: Define histories dictionary
histories = __________________________

# TODO 10: Define trained_models dictionary
trained_models = __________________________

# TODO 11: Define Q1_CHECKPOINT_SUFFIX string
Q1_CHECKPOINT_SUFFIX = __________________________

# TODO 12: Perform the required operation (initialize best score tracking variables)
__________________________

# TODO 13: Iterate over the required data (experiment_configs sweep loop)
for tag, cell_cls, n_layers, is_bidir, is_lstm in __________________________:
    # TODO 14: Perform the required operation (print training banner/status)
    __________________________

    # TODO 15: Define model instance using build_emotion_model
    model = __________________________

    # TODO 16: Define hist by calling train_model
    hist = __________________________

    # TODO 17: Define report, cm by calling evaluate_full on test/val loader
    report, cm = __________________________

    # TODO 18: Define results[tag]
    results[tag] = __________________________

    # TODO 19: Define histories[tag]
    histories[tag] = __________________________

    # TODO 20: Define trained_models[tag]
    trained_models[tag] = __________________________

    # TODO 21: Define current_best_tag
    current_best_tag = __________________________

    # TODO 22: Perform the required operation (compare and update best model metrics)
    __________________________

    # TODO 23: Define current_best_config dictionary/tuple
    current_best_config = __________________________

    # TODO 24: Import the required libraries (import json)
    # import json

    # TODO 25: Run the scoped operation (open file for saving best results summary)
    with __________________________:
        # TODO 26: Perform the required operation (json dump)
        __________________________

    # TODO 27: Perform the required operation (clear cache / garbage collection)
    __________________________

**Quantitative & Qualitative Analysis[5 Marks]**

1. Per-Label F1 Heatmaps

For multi-label classification, a single 28×28 confusion matrix is not meaningful because classes are independent rather than mutually exclusive. Instead, the grid below visualizes each model's F1 score across all 28 emotion categories:

* **Why Heatmaps?** They immediately expose which emotion labels (e.g., *gratitude*, *admiration*) are easily learned versus which abstract or rare emotions (e.g., *grief*, *relief*) suffer across different architectures.
* **Architecture Comparison:** Comparing subplots side-by-side reveals how moving from Vanilla RNNs to LSTMs/GRUs and adding bidirectionality lifts per-label performance across the board.

In [ ]:


# --- Per-Label F1 Heatmaps Visualization ---
# TODO 1: Define fig, axes (create subplots grid for the 9 models)
fig, axes = __________________________

# TODO 2: Iterate over the required data (loop through axes, tags, and configurations)
for ax, tag, _, _, _, _ in __________________________:
    # TODO 3: Define report for the current model tag
    report = __________________________

    # TODO 4: Define f1_scores extracted from the classification report
    f1_scores = __________________________

    # TODO 5: Define image (generate heatmap matrix or array)
    image = __________________________

    # TODO 6: Perform the required operation (plot heatmap on axis)
    __________________________

    # TODO 7: Perform the required operation (set subplot title using tag)
    __________________________

    # TODO 8: Perform the required operation (set axis ticks and labels)
    __________________________

    # TODO 9: Perform the required operation (configure colorbar or formatting)
    __________________________

# TODO 10: Perform the required operation (adjust layout spacing)
__________________________

# TODO 11: Perform the required operation (add overall figure title / suptitle)
__________________________

# TODO 12: Perform the required operation (display the plot)
__________________________

# TODO 13: Perform the required operation (save figure to disk)
__________________________

**Identifying the Largest Multi-Label Errors[5 Marks]**

For each label, we inspect false positives and false negatives rather than forcing one predicted class to compete with another.


For multi-label classification, standard multi-class confusion matrices do not work because each emotion is evaluated independently. Instead, we inspect **False Positives (FP)** and **False Negatives (FN)** across individual 2×2 label matrices:

* **False Positives (Type I Error):** The model predicts an emotion that wasn't actually present in the comment (over-predicting).
* **False Negatives (Type II Error):** The model misses an emotion that was present in the gold-standard labels (under-predicting).
* **Why Track This?** Isolating top FP and FN counts per label helps us pinpoint which specific emotions our best recurrent architecture struggles with most.

In [ ]:

# --- Identifying Largest Multi-Label Errors ---
def get_top_label_errors(label_confusion_matrices, class_names, k=5):
    """Returns the largest per-label false-positive and false-negative counts."""
    # TODO 1: Define errors list
    errors = __________________________

    # TODO 2: Iterate over the required data (zip confusion matrices and class names)
    for matrix, class_name in __________________________:
        # TODO 3: Define false_positive count from 2x2 confusion matrix
        false_positive = __________________________

        # TODO 4: Define false_negative count from 2x2 confusion matrix
        false_negative = __________________________

        # TODO 5: Perform the required operation (append FP entry to errors list)
        __________________________

        # TODO 6: Perform the required operation (append FN entry to errors list)
        __________________________

    # TODO 7: Return the computed result (sorted top k errors)
    return __________________________

# TODO 8: Define best_model_tag (e.g., retrieving the tag with highest validation micro-F1)
best_model_tag = __________________________

# TODO 9: Perform the required operation (retrieve confusion matrices for the best model)
__________________________

# TODO 10: Define top_errors by calling get_top_label_errors
top_errors = __________________________

# TODO 11: Iterate over the required data (loop through top errors)
for count, error_type, label in __________________________:
    # TODO 12: Perform the required operation (print formatted error summary)
    __________________________

**Qualitative Inspection: Multi-Label Prediction Errors[5 Marks]**

Inspect examples where the predicted set of labels does not exactly match the true label set.


Quantitative metrics like F1 scores and confusion matrices show us *where* the model fails globally, but qualitative inspection shows us *why* it fails on specific texts.

* **Exact Set Mismatch:** We look for comments where the model's predicted set of emotion labels does not perfectly align with the gold-standard true labels.
* **Nuance & Overlap:** This inspection reveals how the model handles subtle linguistic cues, sarcasm, or highly overlapping categories (e.g., confusing *admiration* with *approval*).

In [ ]:

# --- Qualitative Inspection of Multi-Label Errors ---
@torch.no_grad()
def inspect_multilabel_errors(model: nn.Module, test_ds, class_names: list, n_samples: int = 5):
    """Prints comments whose predicted multi-label sets differ from their true sets."""
    # TODO 1: Perform the required operation (set model to evaluation mode)
    __________________________

    # TODO 2: Define count tracking variable
    count = __________________________

    # TODO 3: Perform the required operation (setup data loader or iterator for test dataset)
    __________________________

    # TODO 4: Iterate over the required data (loop through test examples)
    for example in __________________________:
        # TODO 5: Define x (input text tokens/tensor)
        x = __________________________

        # TODO 6: Define predicted binary multi-label tensor based on threshold
        predicted = __________________________

        # TODO 7: Define target gold-standard label tensor
        target = __________________________

        # TODO 8: Handle the relevant condition (check if predicted set differs from target set)
        if __________________________:
            # TODO 9: Define true_labels names list
            true_labels = __________________________

            # TODO 10: Define predicted_labels names list
            predicted_labels = __________________________

            # TODO 11: Perform the required operation (print sample text/comment)
            __________________________

            # TODO 12: Perform the required operation (print gold-standard true labels)
            __________________________

            # TODO 13: Perform the required operation (print model predictions)
            __________________________

            # TODO 14: Perform the required operation (print separator for readability)
            __________________________

            # TODO 15: Update count of displayed samples
            count = __________________________

        # TODO 16: Handle the relevant condition (break loop if n_samples limit reached)
        if __________________________:
            break

# TODO 17: Define best_cfg configuration dictionary/tuple
best_cfg = __________________________

# TODO 18: Define best_model instance from trained models dictionary
best_model = __________________________

# TODO 19: Perform the required operation (call inspect_multilabel_errors with best model)
__________________________

# TODO 20: Perform the required operation (print completion message)
__________________________

**Parameter Efficiency Breakdown[10 Marks]**

Finally, let's analyze the trade-off between **model capacity (parameter count)** and **validation accuracy**:

Finally, let's analyze the trade-off between **model capacity (parameter count)** and **validation micro-F1 performance**:

* **Efficiency vs. Performance:** Larger models (like multi-layer LSTMs or BiGRUs) have higher parameter capacities, but do they always yield proportional accuracy gains over simpler architectures?
* **Comparative Trade-Offs:** Plotting parameter counts against validation scores helps us determine the optimal "sweet spot"—balancing computational cost and inference speed against multi-label classification accuracy across our 9 model configurations.

In [ ]:

# --- Parameter Efficiency Breakdown & Scaling Analysis ---
# TODO 1: Perform the required operation (setup plotting figure and axis for parameter vs performance trade-off)
__________________________

# TODO 2: Perform the required operation (initialize lists or dicts to store parameter counts and micro-F1 scores)
__________________________

# TODO 3: Iterate over the required data (loop through experiment configuration keys/tags)
for tag in __________________________:
    # TODO 4: Define cell_cls (extract cell class reference from config or sweep history)
    cell_cls = __________________________

    # TODO 5: Define is_lstm boolean flag for the model architecture
    is_lstm = __________________________

    # TODO 6: Define model instance from trained models dictionary
    model = __________________________

    # TODO 7: Perform the required operation (compute/count total parameters and plot/record data point)
    __________________________

### Task 8 : Advanced Analytical & Experimental Evaluation[Total = 50 Marks]

Answer all questions below using evidence, metrics, and observations directly from your 9-model sweep and experimental runs.

- Logical & Reasoning

* **L1. Stacking vs. Bidirectionality:[10 Marks]** Both doubling stacking depth and adding bidirectionality roughly double parameter count, but do they solve the same problem? Justify your answer using the architectural limitations and sequence-reading constraints observed in your sweep.






-----------write your answer here---------------

* **L2. Universal Gradient Clipping:[10 marks]** Is applying a uniform gradient clipping norm (`max_norm=5.0`) to all models (including LSTM/GRU and vanilla RNN) scientifically correct for a fair 9-model comparison, or could it mask a real performance difference between architectures? Explain your reasoning.


-----------write your answer here---------------

- Analytical (Requires Your Run's Data & Plots)

* **A1. Performance Sweep Analysis:[10 Marks]** Report the final validation micro-F1 score and validation loss for all 9 trained models. Which single architectural change (single→stacked, or unidirectional→bidirectional) yielded the largest micro-F1 gain, averaged across cell types?


Before filling out the code below, familiarize yourself with these core components:

* **`histories`**: A dictionary storing the training and validation history for all 9 models trained during the sweep, tracking keys such as `"val_acc"` (validation micro-F1) and `"val_loss"`.
* **`experiment_configs`**: The configuration list mapping model tags to their corresponding cell types, layer depths, and directionality settings.
* **Micro-F1 Gain Calculation**: Computes the delta ($\Delta$) in validation micro-F1 when transitioning from single-layer to stacked layers ($L=1 \rightarrow 2$) and from unidirectional to bidirectional contexts across RNN, LSTM, and GRU variants.
* **Comparative Evaluation**: Averages the performance deltas across cell families to determine which architectural scaling method provides a larger boost in multi-label classification accuracy.

In [ ]:
##Code here##

* **E1. Word-Order Control:[10 Marks]** Run a word-order control experiment: shuffle the token order randomly within every train/val example (preserving the exact same token set) and retrain `GRU-single` from scratch. Compare its final validation micro-F1 to the un-shuffled `GRU-single` baseline.


Before filling out the code below, familiarize yourself with these core components:

* **Token-Order Shuffling (`shuffled_encode`)**: Randomizes the sequence of tokenized words prior to truncation and padding. This tests whether the model relies on sequential syntax or solves the classification task primarily via bag-of-words keyword matching.
* **`ShuffledEmotionDataset`**: A custom dataset wrapper that inherits standard target indexing but leverages `shuffled_encode` for input representations.
* **Retraining and Comparison**: Retrains a fresh `GRU-single` model configuration on the shuffled data loaders to measure the validation performance drop.

In [ ]:
##Code here##

* **E2. Sequence Length Sweep:[10 Marks]** Vary `MAX_LEN` in $\{15, 30, 50\}$ and retrain `GRU-bidirectional` for each setting, holding all other hyperparameters fixed. Plot and analyze validation micro-F1 vs. `MAX_LEN`.

Before filling out the code below, familiarize yourself with these core components:

* **Self-Contained DataLoaders (`build_loaders_for_max_len`)**: Dynamically reconstructs dataset encoding rules for different sequence length thresholds ($\text{MAX\_LEN} \in \{15, 30, 50\}$) without corrupting global variables.
* **Model Retraining Loop**: Trains a fresh `GRU-bidirectional` classifier configuration across each sequence length setting while holding all other hyperparameters constant.
* **Performance Analysis**: Evaluates validation micro-F1 fluctuations to examine how truncation or padding density impacts sequence context capture.

In [ ]:
##Code here##

# **Question 2 — English-to-Hindi Machine Translation[Total = 150 Marks]**
2. Implement Encoder-Decoder 2-layer bidirectional GRU model from scratch for machine translation task (English-Hindi translation), train it, do the error analysis to understand the poor performance.


In Question 1, we worked on multi-label emotion classification—where the model reads a sentence and predicts a **set of multiple emotion labels** simultaneously.

Machine translation is completely different. Here, we convert a **variable-length sequence of English words** into a **variable-length sequence of Hindi words**:

$$\text{"I love learning NLP."} \longrightarrow \text{"मुझे एनएलपी सीखना पसंद है।"}$$

Because input and output lengths can be completely different, a fixed classification head won't work. Instead, we use an **Encoder-Decoder (Seq2Seq)** framework:

* **The Encoder:** Reads the entire English sentence and compresses its meaning into a single hidden vector representation.
* **The Decoder:** Unrolls that vector to generate the Hindi translation, token by token.

### **Dataset Preparation**

In this section, we will prepare our English-Hindi parallel text dataset, build two separate vocabularies, and construct PyTorch `DataLoaders` that handle sequence padding and special tokens.



#### **Loading the Parallel Corpus**

**What are we doing?**

We call `load_dataset` to retrieve our train, validation, and test splits and inspect their column names. **Never assume column names!** They might be structured as `["english", "hindi"]`, `["src", "tgt"]`, or nested inside a `["translation"]` dictionary. Always print an example row to verify the schema before writing any downstream processing code.

In [ ]:
#Do not Change content of this cell
def load_english_hindi(repo_id: str, token: str):
    """Load English-Hindi train, validation, and test splits from the course HF repository."""
    return load_dataset(
        repo_id,
        data_files={
            "train": "Assignment2/EnglishHindi_train.parquet",
            "validation": "Assignment2/EnglishHindi_val.parquet",
            "test": "Assignment2/EnglishHindi_test.parquet",
        },
        token=token,
    )

mt_raw = load_english_hindi(REPO_ID, hf_token)

####**Schema Detection & Column Mapping[5 marks]**

**Why do we need this?**


Parquet files stored in Hugging Face can sometimes nest translation pairs inside a dictionary column (e.g., `row['translation']['en']`) or store them in flat top-level columns (e.g., `row['english']` and `row['hindi']`). We need to inspect the schema and assign `SRC_COL` (English) and `TGT_COL` (Hindi) correctly.

In [ ]:
# Inspect the dataset schema
print("--- Dataset Splits ---")
print(mt_raw)

print("\n--- Column Names ---")
print(mt_raw["train"].column_names)

print("\n--- Example Row ---")
print(mt_raw["train"][0])

Before filling out the code below, familiarize yourself with these core components:

* **`SRC_LANG` & `TGT_LANG`**: Keys used to extract the source (English) and target (Hindi) strings from the nested dictionary format.
* **`extract_pair(example)`**: A helper utility function that safely parses a dataset row and returns the tuple of source and target sentences.
* **`sample_en` & `sample_hi`**: Sanity-check variables that pull the first translated pair to verify proper extraction before building vocabularies.

In [ ]:
# From above inspection, it is clear that data is structured in format.
# So, we will asasign SRC_LANG and TGT_LANG accordingly.
# Nested dictionary format: row['translation'] = {'en': '...', 'hi': '...'}
# Sanity check

# TODO 1: Define SRC_LANG, TGT_LANG
SRC_LANG, TGT_LANG = __________________________
def extract_pair(example: dict) -> tuple[str, str]:
    'Helper function to cleanly extract (English, Hindi) string pairs.'
    return (example['translation'][SRC_LANG], example['translation'][TGT_LANG])
# TODO 2: Define sample_en, sample_hi
sample_en, sample_hi = __________________________
# TODO 3: Perform the required operation

#### **Tokenization & Building Separate Vocabularies[5 marks]**

#### **Why do we need this?**

1. **Two Languages = Two Vocabularies:** English uses the Latin alphabet (`[a-z]`), while Hindi uses the Devanagari script (`[अ-ह]`). Since their character sets, grammars, and words are completely distinct, we must build **two separate vocabulary mappings** (`stoi_en` / `itos_en` and `stoi_hi` / `itos_hi`).
2. **Special Tokens for Seq2Seq:** Beyond `<pad>` (index 0) and `<unk>` (index 1), sequence-to-sequence models require two extra special control tokens:
* **`<sos>` (Start of Sequence):** Tells the decoder *"Start generating the translation now!"*
* **`<eos>` (End of Sequence):** Tells the decoder *"Stop generating! The sentence is finished."*


**Example Mappings**

* **Special Tokens:**
* `<pad>` $\rightarrow$ `0`
* `<unk>` $\rightarrow$ `1`
* `<sos>` $\rightarrow$ `2`
* `<eos>` $\rightarrow$ `3`

Before filling out the code below, familiarize yourself with these core components:

* **Dual Tokenization (`tokenize_en` / `tokenize_hi`)**: English uses regex word-level tokenization with lowercase conversion, while Hindi uses whitespace tokenization suited for Devanagari script structures.
* **Special Control Tokens (`PAD`, `UNK`, `SOS`, `EOS`)**: Essential tokens for sequence padding (`0`), unknown words (`1`), start of sequence (`2`), and end of sequence (`3`).
* **Vocabulary Limiters (`MAX_VOCAB_EN`, `MAX_VOCAB_HI`)**: Caps vocabulary sizes to control embedding parameter memory and filter out rare noise words.
* **Data Leakage Prevention (`counter_en`, `counter_hi`)**: Frequency counters and vocabulary mappings (`stoi` / `itos`) are built exclusively on the **training split** to prevent information leakage from validation and test sets.

In [ ]:


# --- Tokenization & Building Separate Vocabularies ---
# English: Regex tokenization for words and basic punctuation
# Define Special Tokens
# Maximum Vocabulary Sizes
# Build Vocabularies on the TRAINING split ONLY (to avoid data leakage)
# Construct English Vocabulary
# Construct Hindi Vocabulary

# TODO 1: Define EN_TOKEN_RE (regex pattern for tokenizing English text)
EN_TOKEN_RE = __________________________

def tokenize_en(text: str) -> list[str]:
    """Lowercase and tokenize English text using regex."""
    # TODO 2: Return the computed result (list of lowercase English tokens)
    return __________________________

def tokenize_hi(text: str) -> list[str]:
    """Whitespace tokenization for Hindi text."""
    # TODO 3: Return the computed result (list of Hindi tokens split by whitespace)
    return __________________________

# TODO 4: Define PAD, UNK, SOS, EOS special token strings
PAD, UNK, SOS, EOS = __________________________

# TODO 5: Define SPECIAL_TOKENS list containing all special tokens in order
SPECIAL_TOKENS = __________________________

# TODO 6: Define MAX_VOCAB_EN integer limit
MAX_VOCAB_EN = __________________________

# TODO 7: Define MAX_VOCAB_HI integer limit
MAX_VOCAB_HI = __________________________

# TODO 8: Define counter_en frequency counter
counter_en = __________________________

# TODO 9: Define counter_hi frequency counter
counter_hi = __________________________

# TODO 10: Iterate over the required data (loop through training split examples)
for example in __________________________:
    # TODO 11: Define src_text, tgt_text using extract_pair helper
    src_text, tgt_text = __________________________

    # TODO 12: Perform the required operation (update English token counter)
    __________________________

    # TODO 13: Perform the required operation (update Hindi token counter)
    __________________________

# TODO 14: Define itos_en (index-to-string mapping list for English including special tokens)
itos_en = __________________________

# TODO 15: Define stoi_en (string-to-index dictionary mapping for English)
stoi_en = __________________________

# TODO 16: Define itos_hi (index-to-string mapping list for Hindi including special tokens)
itos_hi = __________________________

# TODO 17: Define stoi_hi (string-to-index dictionary mapping for Hindi)
stoi_hi = __________________________

# TODO 18: Perform the required operation (print English vocabulary size)
__________________________

# TODO 19: Perform the required operation (print Hindi vocabulary size)
__________________________

####**Text Encoding & Length Filtering[5 marks]**

#### **Why do we need this?**

1. **Source Encoding:** English sentences are encoded as standard ID sequences.

2. **Target Encoding (Teacher Forcing Format):** The Hindi target sentence **must** be wrapped with start and end markers:

   $$\text{Encoded Hindi} = [\text{stoi_hi}[\text{''<sos>''}] + \text{token_ids} + \text{stoi_hi}[\text{''<eos>''}]]$$

   This allows us to train the decoder using **Teacher Forcing** (feeding `<sos>` as the initial input to predict the first word, and learning to predict `<eos>` when the sentence ends).

3. **Filtering Outliers:** Extremely long sentences (e.g., 200+ tokens) slow down training significantly and consume excessive GPU memory. Filtering out pathologically long sentence pairs keeps our training step times manageable.

Before filling out the code below, familiarize yourself with these core components:

* **Maximum Sequence Lengths (`MAX_LEN_EN`, `MAX_LEN_HI`)**: Caps tokenized sequence lengths to standard tensor dimensions, applying truncation or padding (`PAD`) as necessary.
* **Target Wrapping (`encode_hi`)**: Wraps Hindi target sequences with special control tokens—prepending `<sos>` and appending `<eos>`—which is mandatory for autoregressive sequence-to-sequence training.
* **Outlier Filtering**: Discards pathologically long sentences to prevent excessive memory overhead and maintain efficient batch processing times on the GPU.

In [ ]:
# --- Text Encoding & Length Filtering ---
# Pad to MAX_LEN_EN
# Pad to MAX_LEN_HI
# Quick Sanity Check

# TODO 1: Define MAX_LEN_EN integer limit for source English sequences
MAX_LEN_EN = __________________________

# TODO 2: Define MAX_LEN_HI integer limit for target Hindi sequences
MAX_LEN_HI = __________________________

def encode_en(text: str) -> list[int]:
    """Tokenize and convert English text to token IDs with padding/truncation."""
    # TODO 3: Define tokens list using tokenize_en
    tokens = __________________________

    # TODO 4: Define ids list mapping tokens through stoi_en (with fallback for UNK)
    ids = __________________________

    # TODO 5: Define ids (pad or truncate to MAX_LEN_EN using PAD index)
    ids = __________________________
    return ids

def encode_hi(text: str) -> list[int]:
    """Tokenize and convert Hindi text to <sos> + IDs + <eos> with padding/truncation."""
    # TODO 6: Define tokens list using tokenize_hi
    tokens = __________________________

    # TODO 7: Define ids mapping tokens through stoi_hi, wrapped with SOS and EOS indices
    ids = __________________________

    # TODO 8: Define ids (pad or truncate to MAX_LEN_HI using PAD index)
    ids = __________________________
    return ids

# TODO 9: Define sample_en_ids by encoding sample_en
sample_en_ids = __________________________

# TODO 10: Define sample_hi_ids by encoding sample_hi
sample_hi_ids = __________________________

# TODO 11: Perform the required operation (print encoded sample English IDs for sanity check)
__________________________

# TODO 12: Perform the required operation (print encoded sample Hindi IDs for sanity check)
__________________________

#### **PyTorch Dataset & DataLoader Wrappers[5 marks]**

#### **Why do we need this?**

PyTorch's `DataLoader` automatically handles batching, shuffling, and converting lists of numerical IDs into GPU-ready `torch.LongTensor` objects.

Before filling out the code below, familiarize yourself with these core components:

* **`TranslationDataset` (`torch.utils.data.Dataset`)**: Wraps parallel corpus splits, handling sequence encoding, length-based outlier filtering, and slicing target sequences into decoder input (`tgt_in` excluding the last token) and expected output (`tgt_out` excluding the first `<sos>` token).
* **`DataLoaders` (`torch.utils.data.DataLoader`)**: Automatically manages batching, padding conversions, and tensor casting (`torch.LongTensor`), ensuring **shuffling is enabled exclusively for the training set** to prevent data leakage during validation/testing.

In [ ]:

# --- PyTorch Dataset & DataLoader Wrappers ---
# Encode Hindi target
# Slice target into decoder input and expected output
# Instantiate PyTorch Datasets
# train_dataset = TranslationDataset(mt_raw["train"])
# val_dataset   = TranslationDataset(mt_raw["validation"])
# test_dataset  = TranslationDataset(mt_raw["test"])
# ===================================================================================================================
# Instantiate DataLoaders (Shuffle ONLY the training set)

class TranslationDataset(Dataset):
    """PyTorch Dataset wrapping English-Hindi parallel sentence pairs."""
    def __init__(self, hf_ds):
        super().__init__()
        # TODO 1: Define self.src_texts storage list
        self.src_texts = __________________________

        # TODO 2: Define self.tgt_texts storage list
        self.tgt_texts = __________________________

        # TODO 3: Iterate over the required data (loop through huggingface dataset split)
        for ex in __________________________:
            # TODO 4: Define src, tgt using extract_pair
            src, tgt = __________________________

            # TODO 5: Handle the relevant condition (filter out sentences exceeding MAX_LEN_EN or MAX_LEN_HI)
            if __________________________:
                # TODO 6: Perform the required operation (append valid source text)
                __________________________

                # TODO 7: Perform the required operation (append valid target text)
                __________________________

    def __len__(self):
        # TODO 8: Return the computed result (total number of filtered samples)
        return __________________________

    def __getitem__(self, idx):
        # TODO 9: Define src_ids by encoding English text at idx
        src_ids = __________________________

        # TODO 10: Define tgt_ids by encoding Hindi text at idx
        tgt_ids = __________________________

        # TODO 11: Define tgt_in (decoder input sequence excluding the final token)
        tgt_in = __________________________

        # TODO 12: Define tgt_out (expected decoder output sequence excluding the initial <sos> token)
        tgt_out = __________________________

        return (torch.tensor(src_ids, dtype=torch.long),
                torch.tensor(tgt_in, dtype=torch.long),
                torch.tensor(tgt_out, dtype=torch.long))

# TODO 13: Define N_DEBUG integer limit (or set to None for full dataset usage)
N_DEBUG = __________________________

# TODO 14: Define train_dataset
train_dataset = __________________________

# TODO 15: Define val_dataset
val_dataset = __________________________

# TODO 16: Define test_dataset
test_dataset = __________________________

# TODO 17: Define BATCH_SIZE integer
BATCH_SIZE = __________________________

# TODO 18: Define train_loader (with shuffle=True)
train_loader = __________________________

# TODO 19: Define val_loader (with shuffle=False)
val_loader = __________________________

# TODO 20: Define test_loader (with shuffle=False)
test_loader = __________________________

# TODO 21: Define sample_src_batch, sample_tgt_in_batch, sample_tgt_out_batch by fetching a batch from train_loader
sample_src_batch, sample_tgt_in_batch, sample_tgt_out_batch = __________________________

# TODO 22: Perform the required operation (print sample source batch shape)
__________________________

# TODO 23: Perform the required operation (print sample target input batch shape)
__________________________

# TODO 24: Perform the required operation (print sample target output batch shape)
__________________________

# TODO 25: Perform the required operation (verify batch tensor data types)
__________________________

# TODO 26: Perform the required operation (print number of training batches)
__________________________

# TODO 27: Perform the required operation (print number of validation batches)
__________________________

# TODO 28: Perform the required operation (print number of test batches)
__________________________

Now that our English and Hindi text is tokenized, numericalized, and batched, we will build a full **Sequence-to-Sequence (Seq2Seq)** model for English-to-Hindi translation.


###Modeling [Encoder-Decoder 2-layer bidirectional GRU model from scratch for machine translation task (English-Hindi translation),]

#### **The Encoder: 2-Layer Bidirectional GRU[10 marks]**

#### **Simple Intuition: How the Encoder Works**
Imagine reading an entire English sentence from start to finish, and then reading it backwards from finish to start to catch all context clues. That is what a **Bidirectional Encoder** does.

* **Why Bidirectional?** In English, a word's meaning often depends on words that come *after* it as well as *before* it. By running two separate GRU passes simultaneously (one left-to-right $\rightarrow$ and one right-to-left $\leftarrow$), the model builds a rich, complete understanding of every single word.
* **The Shape Mismatch Problem:** Because our encoder looks both ways across 2 layers, its final output vector has a size of `2 * hidden_size` (forward state + backward state). However, our decoder only expects a standard size of `hidden_size`.
* **The Bridge Solution:** We use a linear projection layer followed by a $\tanh$ activation function (the **Bridge**) to shrink and squash the combined bidirectional vector down to fit the decoder's exact starting dimensions ($h_0^{\text{dec}}$).

$$h_0^{\text{dec}} = \tanh\left(W_{\text{bridge}} \cdot \left[\overrightarrow{h}_{T_x} \;;\; \overleftarrow{h}_{T_x}\right] + b_{\text{bridge}}\right)$$

You can directly reuse your modular `RecurrentEncoder` from Question 1!

Before filling out the code below, familiarize yourself with these core components:
* **`self.embedding`**: Converts input token integer IDs into dense vectors of size `emb_dim`.
* **`self.rnn`**: A 2-layer bidirectional GRU module processing sequences in both directions.
* **`self.bridge`**: A Linear layer that projects the concatenated final forward and backward states (`2 * hidden_size`) down to `hidden_size`.

In [ ]:

# --- 2-Layer Bidirectional GRU Encoder & Bridge ---
class Encoder(nn.Module):
    """2-Layer Bidirectional GRU Encoder.

    Reuses Q1's generic RecurrentEncoder structure with cell_cls=GRUCell, num_layers=2, bidirectional=True.
    """
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, num_layers: int = 2):
        """
        Args:
            vocab_size (int): Size of the English vocabulary.
            emb_dim (int): Embedding dimension.
            hidden_size (int): Hidden dimension for GRU cells.
            num_layers (int): Number of stacked recurrent layers (default: 2).
        """
        super().__init__()
        # TODO 1: Define self.embedding layer (nn.Embedding)
        self.embedding = __________________________

        # TODO 2: Define self.rnn module (nn.GRU with bidirectional=True and specified num_layers)
        self.rnn = __________________________

        # TODO 3: Define self.bridge layer (nn.Linear mapping 2 * hidden_size to hidden_size)
        self.bridge = __________________________

    def forward(self, src: torch.Tensor):
        """
        Args:
            src (torch.Tensor): English token IDs of shape [batch_size, src_len]

        Returns:
            outputs (torch.Tensor): Per-step hidden states [batch_size, src_len, hidden_size * 2]
            dec_h0 (torch.Tensor): Initialized state for decoder [batch_size, hidden_size]
        """
        # TODO 4: Define emb by passing source tokens through embedding layer
        emb = __________________________

        # TODO 5: Define outputs, final_h by passing embeddings through rnn
        outputs, final_h = __________________________

        # TODO 6: Define dec_h0 by concatenating final forward and backward states and passing through bridge + tanh
        dec_h0 = __________________________

        return (outputs, dec_h0)

####**The Decoder: 2-Layer Unidirectional Step-Wise GRU[10 marks]**

**Simple Intuition: Why Step-Wise Decoding?**
Unlike the encoder (which reads the whole sentence at once), the decoder must generate Hindi words **one at a time (autoregressively)**. When predicting word $t$, the model cannot look into the future ($t+1 \dots T_y$).

* **Why Unidirectional?** Text generation flows strictly forward in time.
* **Why Step-Wise?** Instead of unrolling over a full sequence matrix at once, we implement `DecoderStep` to compute **exactly one timestep at a time**. At each step, it takes the previous token and previous hidden states, updates the memory, and outputs logits over the entire Hindi vocabulary.

Before filling out the code below, familiarize yourself with these core components:
* **`self.embedding`**: Converts target Hindi token IDs into dense embeddings of size `emb_dim`.
* **`self.cells`**: A `torch.nn.ModuleList` containing 2 stacked unidirectional `GRUCell` instances (one for each layer).
* **`self.out`**: A Linear projection layer mapping the top layer's final hidden state (`hidden_size`) to unnormalized vocabulary logits (`vocab_size`).

In [ ]:

# --- Step-Wise 2-Layer Unidirectional GRU Decoder ---
class DecoderStep(nn.Module):
    """Step-wise 2-Layer Unidirectional GRU Decoder."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, num_layers: int = 2):
        """
        Args:
            vocab_size (int): Size of the Hindi target vocabulary.
            emb_dim (int): Embedding dimension.
            hidden_size (int): Hidden state dimension.
            num_layers (int): Number of stacked layers (default: 2).
        """
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.hidden_size
        self.hidden_size = __________________________

        # TODO 3: Define self.embedding layer for target language tokens
        self.embedding = __________________________

        # TODO 4: Define self.cells using nn.ModuleList containing num_layers of GRUCell
        self.cells = __________________________

        # TODO 5: Define self.out linear projection layer mapping hidden_size to vocab_size
        self.out = __________________________

    def forward(self, tok: torch.Tensor, hidden_states: list[torch.Tensor]):
        """Advances decoder by exactly one step t.

        Args:
            tok (torch.Tensor): Tensor of input token IDs [batch_size]
            hidden_states (list[torch.Tensor]): Hidden states for each layer, each [batch_size, hidden_size]

        Returns:
            logits (torch.Tensor): Unnormalized vocabulary scores [batch_size, hi_vocab_size]
            new_states (list[torch.Tensor]): Updated hidden states for the next step
        """
        # TODO 6: Define x by embedding the input token IDs [batch_size, emb_dim]
        x = __________________________

        # TODO 7: Define new_states list to store updated hidden states per layer
        new_states = __________________________

        # TODO 8: Define inp starting with the embedded token representation x
        inp = __________________________

        # TODO 9: Iterate over the required data (enumerate through self.cells)
        for l, cell in __________________________:
            # TODO 10: Define h (extract previous hidden state for layer l)
            h = __________________________

            # TODO 11: Perform the required operation (pass input and hidden state through cell l)
            __________________________

            # TODO 12: Define inp (update input for the next stacked layer to be the current layer's hidden state)
            inp = __________________________

        # TODO 13: Define logits by projecting the final layer's updated hidden state through self.out
        logits = __________________________

        return (logits, new_states)

#### **Seq2Seq Wrapper & Teacher Forcing[10 marks]**

#### **Simple Intuition: How Seq2Seq and Teacher Forcing Work**
Training an autoregressive decoder without guidance is difficult. If the model makes a minor mistake early in a sentence, feeding that wrong word into the next step compounds the error, causing the training loop to destabilize.

* **Teacher Forcing:** Instead of always feeding its own predictions, the model flips a coin based on `teacher_forcing_ratio`. If true, it uses the **ground-truth previous target token**; otherwise, it uses its own prediction. This stabilizes early training.
* **The Sequence Loop:** The `Seq2Seq` module ties together the encoder and decoder step-by-step: it encodes the source sequence once, initializes all stacked decoder layers using the bridged encoder state, and unrolls predictions across the target length.

Before filling out the code below, familiarize yourself with these core components:
* **`self.encoder` & `self.decoder`**: Submodules handling source compression and step-wise target generation.
* **`hidden_states`**: A list maintaining the hidden state for each of the 2 decoder layers, initialized by replicating `dec_h0` across all layers.
* **`inp_tok`**: Tracks the input token fed into the current decoder step (starting with the `<sos>` token from `tgt_in[:, 0]`).

In [ ]:



# --- Seq2Seq Wrapper & Teacher Forcing Loop ---
class Seq2Seq(nn.Module):
    """Wraps Encoder and DecoderStep with a Teacher-Forced Training Loop."""
    def __init__(self, encoder: Encoder, decoder: DecoderStep, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.encoder
        self.encoder = __________________________

        # TODO 2: Define self.decoder
        self.decoder = __________________________

        # TODO 3: Define self.num_layers
        self.num_layers = __________________________

    def forward(self, src: torch.Tensor, tgt_in: torch.Tensor, teacher_forcing_ratio: float = 0.7):
        """
        Args:
            src (torch.Tensor): English source token IDs [batch_size, src_len]
            tgt_in (torch.Tensor): Hindi target input IDs (prefixed with <sos>) [batch_size, tgt_len]
            teacher_forcing_ratio (float): Probability of feeding ground truth token (default: 0.7)

        Returns:
            logits_seq (torch.Tensor): Output logits for all steps [batch_size, tgt_len, hi_vocab_size]
        """
        # TODO 4: Define batch_size, tgt_len from tgt_in shape
        batch_size, tgt_len = __________________________

        # TODO 5: Define _, dec_h0 by passing source tensor through encoder
        _, dec_h0 = __________________________

        # TODO 6: Define hidden_states list (initialize each of the num_layers with dec_h0)
        hidden_states = __________________________

        # TODO 7: Define logits_seq tensor container to store outputs for all timesteps
        logits_seq = __________________________

        # TODO 8: Define inp_tok (extract the first column of tgt_in, i.e., the <sos> tokens for batch)
        inp_tok = __________________________

        # TODO 9: Iterate over the required data (loop through target sequence length tgt_len)
        for t in range(__________________________):
            # TODO 10: Define logits, hidden_states by calling self.decoder for step t
            logits, hidden_states = __________________________

            # TODO 11: Perform the required operation (store current step logits into logits_seq tensor at index t)
            __________________________

            # TODO 12: Handle the relevant condition (check if we should apply teacher forcing for the next step)
            if __________________________:
                # TODO 13: Define use_tf (boolean decision based on random Bernoulli or random.random() < teacher_forcing_ratio)
                use_tf = __________________________

                # TODO 14: Handle the relevant condition (if use_tf is True)
                if __________________________:
                    # TODO 15: Define inp_tok (use ground-truth token from tgt_in at index t)
                    inp_tok = __________________________
                else:
                    # TODO 16: Define inp_tok (use model's own argmax prediction from current logits)
                    inp_tok = __________________________

        # TODO 17: Return the computed result (logits_seq tensor)
        return __________________________

#### **Teacher Forcing as a Decoder-Training Strategy[10 marks]**

Teacher forcing supplies the gold previous token to the decoder at every training step. For a source sentence $x$ and target sequence $y_{1:T}$ (where $y_0=\texttt{<sos>}$), the teacher-forcing negative log-likelihood is

$$
\mathcal{L}_{\mathrm{TF}}(\theta) = -\sum_{t=1}^{T}\log p_\theta\!\left(y_t \mid y_{<t}, x\right),
$$

with padded positions excluded from the sum. Conditioning on the gold prefix makes the next-token learning signal stable and parallel to the target sequence, but at inference the gold prefix is unavailable. The decoder must instead condition on its own earlier predictions, creating the train–test mismatch commonly called *exposure bias*. In this notebook, `teacher_forcing_ratio=1.0` implements the objective above exactly.

The terminology is commonly attributed to Williams and Zipser (1989); the conditional Seq2Seq formulation is also the standard maximum-likelihood training setup described by [Sutskever, Vinyals, and Le (2014)](https://arxiv.org/abs/1409.3215).


In [ ]:
def teacher_forcing_objective(model, src, tgt_in, tgt_out, criterion):
    'Returns the masked negative log-likelihood under full teacher forcing.'
    # TODO 1: Define logits
    logits = __________________________
    # TODO 2: Return the computed result
    return __________________________


#### **Scheduled Sampling as a Decoder-Training Strategy[10 marks]**

Scheduled sampling mixes a gold token and the model's previous prediction while training. At epoch $e$, let $z_t\sim\mathrm{Bernoulli}(p_{\mathrm{TF}}(e))$. The decoder input for the next step is

$$
\tilde y_{t-1}=\begin{cases}
y_{t-1}, & z_t=1,\\
\operatorname{argmax}_v\,p_\theta(v\mid\tilde y_{<t-1},x), & z_t=0.
\end{cases}
$$

This notebook uses the explicit linear curriculum

$$
p_{\mathrm{TF}}(e)=p_{\mathrm{start}}-
\frac{e}{\max(E-1,1)}\left(p_{\mathrm{start}}-p_{\mathrm{end}}\right),
$$

clipped to $[p_{\mathrm{end}},p_{\mathrm{start}}]$. Thus the model starts with fully guided prefixes and gradually practices recovery from its own discrete predictions. The `argmax` token is treated as a non-differentiable input choice; gradients still flow through the logits used in the cross-entropy loss. Scheduled sampling was proposed by [Bengio et al. (2015)](https://proceedings.neurips.cc/paper_files/paper/2015/file/e995f98d56967d946471af29d7bf99f1-Paper.pdf).


In [ ]:
def linear_teacher_forcing_probability(epoch, total_epochs, start=1.0, end=0.25):
    'Linear scheduled-sampling curriculum: P(use gold previous token).'
    # TODO 1: Handle the relevant condition
    if __________________________:
        # TODO 2: Complete this step
        __________________________
    # TODO 3: Define progress
    progress = __________________________
    # TODO 4: Return the computed result
    return __________________________


###**Training the Machine Translation Model[10 marks]**

Loss Function & Padding Masking

When computing `nn.CrossEntropyLoss`, we **MUST** pass `ignore_index=stoi_hi[PAD]`. Padding tokens are non-informative; including them in the loss computation rewards the network for predicting empty slots rather than real translation words.

Before filling out the code below, familiarize yourself with these core components:

Teacher Forcing Ratio (tf_ratio): Controls the probability of feeding the ground-truth target token into the decoder at each step during training instead of the model's own autoregressive prediction.

Loss Masking (ignore_index=stoi_hi[PAD]): Ensures that padding tokens are ignored during cross-entropy loss calculation so the model is not rewarded or penalized for predicting padding slots.

Gradient Clipping (torch.nn.utils.clip_grad_norm_): Caps the maximum norm of gradients (e.g., max_norm=5.0) to prevent exploding gradients during Backpropagation Through Time.

Validation Evaluation (tf_ratio=1.0): Enforces full teacher forcing during validation runs to maintain a stable, consistent evaluation loss metric.

In [ ]:
def train_seq2seq(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, epochs: int=50, lr: float=0.001, tf_ratio: float=0.7, tag: str='Seq2Seq-noattn', resume: bool=True) -> dict:
    '''Trains the Seq2Seq model using Teacher Forcing.

       Checkpoints after every epoch to CKPT_DIR/<tag>.pt and resumes from / skips a
       matching checkpoint the same way train_model does (see Initial Setup).
    '''
    # TODO 1: Move model to the computation device (DEVICE)
    model.to(DEVICE)

    # TODO 2: Define optimizer using torch.optim.Adam with model parameters and learning rate lr
    optimizer = __________________________

    # Ignore <pad> tokens in loss calculation
    # TODO 3: Define criterion as nn.CrossEntropyLoss with ignore_index set to stoi_hi[PAD]
    criterion = __________________________

    # TODO 4: Initialize start_epoch to 0 and history dictionary with keys "train_loss" and "val_loss" mapping to empty lists
    start_epoch, history = __________________________

    # TODO 5: Check if resume is True and a non-empty tag is provided
    if __________________________:
        # TODO 6: Load checkpoint using load_checkpoint(tag, model, optimizer)
        ckpt_epoch, ckpt_history = __________________________

        # TODO 7: Check if ckpt_history is not None
        if __________________________:

            # TODO 8: Check if ckpt_epoch is not None and ckpt_epoch is greater than or equal to total epochs
            if __________________________:

                # TODO 9: Print skipping message and return ckpt_history directly
                __________________________
                return ckpt_history

            # TODO 10: Set start_epoch to ckpt_epoch and history to ckpt_history
            start_epoch, history = __________________________

            # TODO 11: Print resumption message indicating the resumed epoch
            __________________________

    # TODO 12: Loop through epochs from start_epoch up to total epochs
    for ep in range(__________________________):
        # --- Training Phase ---
        # TODO 13: Set model to training mode using model.train()
        __________________________

        # TODO 14: Initialize total_train_loss accumulator to 0.0
        total_train_loss = __________________________

        # TODO 15: Iterate over mini-batches (src, tgt_in, tgt_out) from train_loader
        for src, tgt_in, tgt_out in __________________________:
            # TODO 16: Move source tensor to DEVICE
            src = __________________________

            # TODO 17: Move target input tensor to DEVICE
            tgt_in = __________________________

            # TODO 18: Move target output tensor to DEVICE
            tgt_out = __________________________

            # TODO 19: Clear old gradients using optimizer.zero_grad()
            __________________________

            # TODO 20: Forward pass through model with src, tgt_in, and teacher_forcing_ratio=tf_ratio
            logits = __________________________

            # Reshape for CrossEntropyLoss: [batch_size * tgt_len, vocab_size] vs [batch_size * tgt_len]
            # TODO 21: Compute cross-entropy loss between reshaped logits and reshaped tgt_out
            loss = __________________________

            # TODO 22: Backward pass to compute gradients via loss.backward()
            __________________________

            # Clip gradients to prevent explosion
            # TODO 23: Apply gradient clipping using torch.nn.utils.clip_grad_norm_ with max_norm=5.0
            __________________________

            # TODO 24: Step the optimizer to update model weights using optimizer.step()
            __________________________

            # TODO 25: Accumulate batch training loss (loss.item() * batch_size) into total_train_loss
            total_train_loss = __________________________

        # TODO 26: Compute epoch train_loss by dividing total_train_loss by total dataset length
        train_loss = __________________________

        # --- Validation Phase (Full Teacher Forcing for consistent evaluation metric) ---
        # TODO 27: Set model to evaluation mode using model.eval()
        __________________________

        # TODO 28: Initialize total_val_loss accumulator to 0.0
        total_val_loss = __________________________

        # TODO 29: Run under torch.no_grad() context manager to disable gradient tracking
        with __________________________:
            # TODO 30: Iterate over mini-batches (src, tgt_in, tgt_out) from val_loader
            for src, tgt_in, tgt_out in __________________________:
                # TODO 31: Move validation source tensor to DEVICE
                src = __________________________

                # TODO 32: Move validation target input tensor to DEVICE
                tgt_in = __________________________

                # TODO 33: Move validation target output tensor to DEVICE
                tgt_out = __________________________

                # TODO 34: Forward pass during validation with teacher_forcing_ratio=1.0
                logits = __________________________

                # TODO 35: Compute validation cross-entropy loss
                loss = __________________________

                # TODO 36: Accumulate batch validation loss into total_val_loss
                total_val_loss = __________________________

        # TODO 37: Compute epoch val_loss by dividing total_val_loss by validation dataset length
        val_loss = __________________________

        # TODO 38: Append train_loss and val_loss values to their respective lists in the history dictionary
        __________________________

        # TODO 39: Check if a checkpoint tag exists to save model progress
        if __________________________:
            # TODO 40: Save model checkpoint using save_checkpoint function passing tag, model, optimizer, epoch, and history
            __________________________

        # TODO 41: Print formatted epoch summary string showing current epoch, training loss, and validation loss
        __________________________

    # TODO 42: Return the final training history dictionary
    return history

# Instantiate and Train Model
# TODO 43: Define HID_MT (hidden size) and EMB_MT (embedding dimension) integer values
HID_MT, EMB_MT = __________________________

# TODO 44: Initialize Encoder with vocab_size, emb_dim, hidden_size, and num_layers
encoder = __________________________

# TODO 45: Initialize DecoderStep with vocab_size, emb_dim, hidden_size, and num_layers
decoder = __________________________

# TODO 46: Wrap encoder and decoder into Seq2Seq model instance named seq2seq_noattn
seq2seq_noattn = __________________________

# TODO 47: Train seq2seq_noattn using train_seq2seq function and store result in hist_noattn
hist_noattn = __________________________

# Save the required Q2 baseline pair immediately after training completes.
# TODO 48: Save encoder and decoder state dictionaries into 'q2_seq2seq_noattn.pt' using torch.save
__________________________

####**Added Strategy Training Runs[10 marks]**

The following two new models use the same architecture, data loaders, optimizer, loss masking, and epoch count as the existing no-attention baseline. They are separate variables and do not replace or retrain `seq2seq_noattn`.


Before filling out the code below, familiarize yourself with these core components:


Strict Teacher Forcing ($p_{\text{TF}} = 1.0$): A baseline training setup where the decoder always consumes the ground-truth previous token as input at every step ($p_{\text{TF}} = 1.0$), ensuring highly stable initial convergence.


Scheduled Sampling Curriculum: A strategy that gradually exposes the decoder to its own discrete predictions during training by linearly decaying the teacher-forcing probability (e.g., from $1.00$ down to $0.25$), mitigating the exposure bias problem.

Linear Curriculum Schedule (linear_teacher_forcing_probability): Computes the dynamic probability threshold at each epoch to determine whether to feed the gold target token or the model's prior prediction.

Strategy Checkpoint Persistence (torch.save): Exports the trained encoder and decoder weights into independent target files (q2_seq2seq_teacher_forcing.pt and q2_seq2seq_scheduled_sampling.pt) for evaluation.

In [ ]:
# --- Additional Q2 decoder-training strategy models ---
# Model 1: strict teacher forcing, p_TF = 1 at every decoder step.
# Model 2: scheduled sampling, with p_TF decaying from 1.00 to 0.25.

# TODO 1: Define STRATEGY_TRAINING_EPOCHS with an integer value (e.g., 4)
STRATEGY_TRAINING_EPOCHS = __________________________

def build_plain_seq2seq_for_strategy():
    "Builds a fresh no-attention model with the notebook's existing dimensions."
    # TODO 2: Return a newly instantiated Seq2Seq model using fresh Encoder and DecoderStep instances
    return __________________________

# TODO 3: Instantiate a fresh plain Seq2Seq model for the strict teacher-forcing strategy run
__________________________

# TODO 4: Define seq2seq_teacher_forcing as the instantiated teacher-forcing model object
seq2seq_teacher_forcing = __________________________

# TODO 5: Train seq2seq_teacher_forcing using train_seq2seq with tf_ratio=1.0 and tag="Seq2Seq-TeacherForcing"
hist_teacher_forcing = __________________________

# TODO 6: Save the state dict checkpoint for seq2seq_teacher_forcing to "q2_seq2seq_teacher_forcing.pt" using torch.save
__________________________

def train_seq2seq_scheduled_sampling(model, train_loader, val_loader, epochs=STRATEGY_TRAINING_EPOCHS, lr=0.001, p_tf_start=1.0, p_tf_end=0.25, tag='Seq2Seq-ScheduledSampling', resume=True):
    'Trains with a linearly decreasing probability of a gold decoder input.'
    # TODO 7: Move model to the computation device using model.to(DEVICE)
    __________________________

    # TODO 8: Define optimizer using torch.optim.Adam with model parameters and learning rate lr
    optimizer = __________________________

    # TODO 9: Define criterion as nn.CrossEntropyLoss with ignore_index set to stoi_hi[PAD]
    criterion = __________________________

    # TODO 10: Initialize start_epoch to 0 and history dictionary with keys "train_loss" and "val_loss" mapping to empty lists
    start_epoch, history = __________________________

    # TODO 11: Check if resume is True and a non-empty tag is provided
    if __________________________:
        # TODO 12: Load checkpoint using load_checkpoint(tag, model, optimizer)
        ckpt_epoch, ckpt_history = __________________________

        # TODO 13: Check if ckpt_history is not None
        if __________________________:

            # TODO 14: Check if ckpt_epoch is not None and ckpt_epoch is greater than or equal to total epochs
            if __________________________:

                # TODO 15: Print skipping message and return ckpt_history directly
                __________________________
                return ckpt_history

            # TODO 16: Set start_epoch to ckpt_epoch and history to ckpt_history
            start_epoch, history = __________________________

            # TODO 17: Print resumption message indicating the resumed epoch
            __________________________

            # TODO 18: Print confirmation statement or log details
            __________________________

    # TODO 19: Loop through epochs from start_epoch up to total epochs using range(start_epoch, epochs)
    for ep in __________________________:
        # TODO 20: Compute current epoch's teacher forcing probability tf_ratio using linear_teacher_forcing_probability(ep, epochs, p_tf_start, p_tf_end)
        tf_ratio = __________________________

        # TODO 21: Set model to training mode using model.train()
        __________________________

        # TODO 22: Initialize total_train_loss accumulator to 0.0
        total_train_loss = __________________________

        # TODO 23: Iterate over mini-batches (src, tgt_in, tgt_out) from train_loader
        for src, tgt_in, tgt_out in __________________________:
            # TODO 24: Move src, tgt_in, and tgt_out tensors to DEVICE
            src, tgt_in, tgt_out = __________________________

            # TODO 25: Clear old gradients using optimizer.zero_grad()
            __________________________

            # TODO 26: Forward pass through model with src, tgt_in, and dynamic teacher_forcing_ratio=tf_ratio
            logits = __________________________

            # TODO 27: Compute cross-entropy loss between reshaped logits and reshaped tgt_out
            loss = __________________________

            # TODO 28: Backward pass to compute gradients via loss.backward()
            __________________________

            # TODO 29: Apply gradient clipping using torch.nn.utils.clip_grad_norm_ with max_norm=5.0
            __________________________

            # TODO 30: Step the optimizer to update model weights using optimizer.step()
            __________________________

            # TODO 31: Accumulate batch training loss (loss.item * batch_size) into total_train_loss
            total_train_loss = __________________________

        # TODO 32: Compute epoch train_loss by dividing total_train_loss by total training dataset length
        train_loss = __________________________

        # TODO 33: Initialize total_val_loss accumulator to 0.0 for validation
        total_val_loss = __________________________

        # TODO 34: Run under torch.no_grad() context manager to disable gradient tracking during validation
        with __________________________:
            # TODO 35: Iterate over validation mini-batches (src, tgt_in, tgt_out) from val_loader
            for src, tgt_in, tgt_out in __________________________:
                # TODO 36: Move validation tensors (src, tgt_in, tgt_out) to DEVICE
                src, tgt_in, tgt_out = __________________________

                # TODO 37: Forward pass during validation with full teacher forcing (teacher_forcing_ratio=1.0)
                logits = __________________________

                # TODO 38: Compute validation cross-entropy loss
                loss = __________________________

                # TODO 39: Accumulate validation loss into total_val_loss
                total_val_loss = __________________________

        # TODO 40: Compute epoch val_loss by dividing total_val_loss by validation dataset length
        val_loss = __________________________

        # TODO 41: Append train_loss and val_loss values to history dictionary lists
        __________________________

        # TODO 42: Check if checkpoint tag exists to save model progress via save_checkpoint
        if __________________________:
            # TODO 43: Save checkpoint passing tag, model, optimizer, current epoch index (+1), and history
            __________________________

        # TODO 44: Print formatted epoch summary string showing current epoch, training loss, and validation loss
        __________________________

        # TODO 45: Complete epoch iteration block logic
        __________________________

    return history

# TODO 46: Build a fresh Seq2Seq strategy model instance using build_plain_seq2seq_for_strategy()
__________________________

# TODO 47: Define seq2seq_scheduled_sampling as the scheduled-sampling model instance
seq2seq_scheduled_sampling = __________________________

# TODO 48: Train scheduled sampling model using train_seq2seq_scheduled_sampling and store in hist_scheduled_sampling
hist_scheduled_sampling = __________________________

# TODO 49: Save state dict checkpoint for seq2seq_scheduled_sampling to "q2_seq2seq_scheduled_sampling.pt" using torch.save
__________________________

####**Autoregressive Inference & SacreBLEU Evaluation[5 marks]**

Autoregressive Greedy Decoding

At test time, there are no ground-truth target tokens. We decode **autoregressively**:

1. Feed the encoder representation and start token `<sos>`.
2. Predict the most likely word using `argmax`.
3. Feed that predicted word back as the input for the next timestep.
4. Stop when `<eos>` is emitted or max length is reached.

Measuring Quality with BLEU

We evaluate translation quality using **SacreBLEU**, an industry-standard implementation of BLEU (Bilingual Evaluation Understudy) that computes n-gram precision against ground-truth references.

> **Reference:** Papineni et al. (2002) introduced BLEU in *[BLEU: a Method for Automatic Evaluation of Machine Translation](https://aclanthology.org/P02-1040.pdf)*.

Before filling out the code below, familiarize yourself with these core components:

Autoregressive Greedy Decoding: Generates target sequence tokens one-by-one by picking the token with the highest predicted probability (argmax) at each decoding step without relying on teacher forcing.

SacreBLEU Corpus Evaluation (sacrebleu.corpus_bleu): An industry-standard automated metric comparing hypothesis translation strings against gold-standard human reference sentences to calculate precision and brevity penalties.

Token String Reconstruction (ids_to_string): Filters out special control tokens (<pad>, <sos>, <eos>) and translates lists of numerical token IDs back into clean human-readable language strings.

In [ ]:
# Encode source sentence
# Start token for all samples in batch
# Compute SacreBLEU score
# Evaluate on Test Set

# TODO 1: Import the required sacrebleu library for translation evaluation
# import sacrebleu
__________________________

@torch.no_grad()
def greedy_decode(model: nn.Module, src_tensor: torch.Tensor, max_len: int=MAX_LEN_HI) -> torch.Tensor:
    'Autoregressively decodes target tokens without teacher forcing.'
    # TODO 2: Set model to evaluation mode using model.eval()
    __________________________

    # TODO 3: Move input source tensor to computation device using src_tensor.to(DEVICE)
    src_tensor = __________________________

    # TODO 4: Extract batch size from the first dimension of src_tensor
    batch_size = __________________________

    # Encode source sentence
    # TODO 5: Pass source tensor through model encoder to extract final states, ignoring outputs and extracting decoder initial hidden state dec_h0
    _, dec_h0 = __________________________

    # TODO 6: Initialize hidden_states list for all decoder layers by replicating dec_h0 across num_layers
    hidden_states = __________________________

    # Start token for all samples in batch
    # TODO 7: Initialize batch input tokens tensor inp_tok filled with SOS token index for every sample in the batch
    inp_tok = __________________________

    # TODO 8: Initialize an empty list predicted_ids to store step-wise predicted token tensors
    predicted_ids = __________________________

    # TODO 9: Loop over generation steps from 0 up to max_len
    for _ in range(__________________________):
        # TODO 10: Run one step of the decoder passing inp_tok and hidden_states to obtain logits and updated hidden_states
        logits, hidden_states = __________________________

        # TODO 11: Compute greedy token predictions by taking argmax across the vocabulary dimension of logits
        inp_tok = __________________________

        # TODO 12: Move predicted token tensor to CPU and append it to predicted_ids list
        __________________________

    # TODO 13: Stack predicted token tensors along dimension 1 and return the final decoded tensor batch
    return torch.stack(__________________________)

def ids_to_string(token_ids: list[int], itos: list[str]) -> str:
    'Converts token IDs back to a readable sentence string.'
    # TODO 14: Initialize an empty words list to store decoded string tokens
    words = __________________________

    # TODO 15: Iterate through each token integer idx in token_ids list
    for idx in __________________________:
        # TODO 16: Check if token index equals EOS token index; if so, break out of loop
        if __________________________:
            break

        # TODO 17: Check if token index is PAD or SOS; if so, skip to the next iteration
        if __________________________:
            continue

        # TODO 18: Lookup word string from itos vocabulary list using idx and append it to words list
        __________________________

    # TODO 19: Join collected words list into a single space-separated sentence string and return it
    return " ".join(__________________________)

def evaluate_bleu_score(model: nn.Module, loader: DataLoader):
    'Computes SacreBLEU across the test set.'
    # TODO 20: Initialize empty lists hypotheses and references to store evaluation corpus strings
    hypotheses, references = __________________________

    # TODO 21: Iterate over test batches (src, _, tgt_out) from loader
    for src, _, tgt_out in __________________________:
        # TODO 22: Generate predicted token ID batches using greedy_decode function
        pred_ids_batch = __________________________

        # TODO 23: Loop through each sample index b in the current batch
        for b in range(__________________________):
            # TODO 24: Convert predicted token tensor for sample b into a hypothesis text string using ids_to_string
            hyp_text = __________________________

            # TODO 25: Convert target output tensor for sample b into a reference text string using ids_to_string
            ref_text = __________________________

            # TODO 26: Append generated hypothesis string to hypotheses list
            __________________________

            # TODO 27: Append ground-truth reference string to references list
            __________________________

    # Compute SacreBLEU score
    # TODO 28: Compute corpus-level SacreBLEU score passing hypotheses and wrapped references list
    bleu = sacrebleu.corpus_bleu(__________________________)
    return (bleu.score, hypotheses, references)

# Evaluate on Test Set
# TODO 29: Evaluate seq2seq_noattn model on test_loader using evaluate_bleu_score and unpack into test_bleu, test_hyps, test_refs
test_bleu, test_hyps, test_refs = __________________________

# TODO 30: Print formatted test BLEU score summary header and value
__________________________

# TODO 31: Print translation samples section header banner
__________________________

# TODO 32: Iterate over first 3 test sample indices using range(3)
for i in range(__________________________):
    # TODO 33: Print ground-truth reference sentence string for sample index i
    __________________________

    # TODO 34: Print generated hypothesis translation string for sample index i
    __________________________

###**Error Analysis: Diagnosing the Information Bottleneck[5 Marks]**

Basic Seq2Seq models usually produce relatively low BLEU scores (often around 5–15). Why?

The Fixed Context Vector Bottleneck

The entire meaning of a 30-word sentence must be compressed into a single vector `dec_h0` of size `hidden_size`. As sentence length increases, early words get squashed and forgotten.

Let's prove this by bucketing test sentences by length ($\le 8$ words vs. $> 8$ words) and comparing their BLEU scores:

Before filling out the code below, familiarize yourself with these core components:

* **Length Bucketing**: Splits test set sentence indices based on source length thresholds ($\le 8$ words vs. $> 8$ words) to diagnose how sentence length impacts translation degradation.
* **Subset Evaluation**: Creates targeted dataset subsets and data loaders for short and long sentence groups to isolate performance drops.
* **BLEU Gap Quantification**: Computes and compares BLEU scores across buckets to quantify the severity of the encoder-decoder information bottleneck.

In [ ]:
# --- Length-Bucket Evaluation Analysis ---
# Evaluate BLEU on short sentences
# Evaluate BLEU on long sentences

# TODO 1: Define short_indices, long_indices (initialize empty lists for short and long sample indices)
short_indices, long_indices = __________________________

# TODO 2: Iterate over the required data (loop through test dataset examples with enumerate)
for idx, ex in __________________________:
    # TODO 3: Define src_tokens (tokenize English source text for sample idx)
    src_tokens = __________________________

    # TODO 4: Handle the relevant condition (check if source token length is <= 8)
    if __________________________:
        # TODO 5: Perform the required operation (append index to short_indices)
        __________________________
    else:
        # TODO 6: Perform the required operation (append index to long_indices)
        __________________________

# TODO 7: Perform the required operation (print total count of short test sentences)
__________________________

# TODO 8: Perform the required operation (print total count of long test sentences)
__________________________

# TODO 9: Define short_subset (create a PyTorch Subset using test_dataset and short_indices)
short_subset = __________________________

# TODO 10: Define short_loader (instantiate DataLoader for short subset)
short_loader = __________________________

# TODO 11: Define bleu_short, _, _ (evaluate BLEU score on short sentences using evaluate_bleu_score or evaluate_bleu_attn)
bleu_short, _, _ = __________________________

# TODO 12: Define long_subset (create a PyTorch Subset using test_dataset and long_indices)
long_subset = __________________________

# TODO 13: Define long_loader (instantiate DataLoader for long subset)
long_loader = __________________________

# TODO 14: Define bleu_long, _, _ (evaluate BLEU score on long sentences)
bleu_long, _, _ = __________________________

# TODO 15: Perform the required operation (print short sentence evaluation BLEU score)
__________________________

# TODO 16: Perform the required operation (print long sentence evaluation BLEU score)
__________________________

# TODO 17: Perform the required operation (compute and print the performance gap between short and long BLEU)
__________________________

# TODO 18: Perform the required operation (print concluding analysis on the fixed context-vector bottleneck)
__________________________

**Common Qualitative Failure Modes**

When analyzing model output, you will notice three common types of errors:

1. **Degenerate Repetition:** The decoder gets stuck in a loop repeating the same phrase (*"वह वह वह"*).
2. **Grammatical Word-Order Shift:** English follows Subject-Verb-Object (SVO), while Hindi follows Subject-Object-Verb (SOV). Without attention mechanisms, verb placement often gets garbled.
3. **Dropping Clauses:** Long clauses at the beginning of English sentences vanish entirely from the Hindi output due to vector compression loss.

In Question 3, we will implement **Attention Mechanisms** to eliminate this information bottleneck!

###Advanced Analytical & Experimental Evaluation[Total = 50 Marks]

Answer all questions below using evidence, metrics, and observations directly from your Seq2Seq model runs, sweeps, and length-bucketing analyses.

**Logical & Reasoning**

* **L1. Layer Initialization Symmetry:[4 Marks]** We initialize *every* decoder layer's hidden state with the *same* bridged encoder vector (`hidden_states = [dec_h0] * num_layers`). Why might this initialization constraint limit what individual stacked layers can learn to specialize in, and how does it affect information flow across the 2-layer depth?


-----------write your answer here---------------

* **L2. Exposure Bias in Greedy Decoding:[4 Marks]** `teacher_forcing_ratio = 0.7` exposes the model to its own predictions 30% of the time during training. Using the term **exposure bias**, explain why this mismatch matters significantly more for a model evaluated using autoregressive greedy decoding than one evaluated purely under teacher forcing.


-----------write your answer here---------------

* **L3. Scheduled Sampling Trade-offs:[5 Marks]** Under the linear schedule, analyze the practical trade-off between decreasing $p_{\mathrm{TF}}$ too aggressively in early epochs versus keeping it near $1.0$ throughout training. Relate both scenarios directly to optimization stability and final inference robustness.



-----------write your answer here---------------

Analytical (Requires Your Run's Data & Evaluation Metrics)

* **A1. Information Bottleneck & Length Bucketing:[5 Marks]** Report your short-vs-long BLEU split  (splitting at $\le 8$ vs. $> 8$ source words). Does this quantitative gap confirm the fixed context-vector information-bottleneck argument? Quantify the exact BLEU drop.


Before filling out the code below, familiarize yourself with these core components:

* **Model Evaluation Utility (`print_categorization_samples`)**: Inspects model generations against target reference sentences to categorize common failure modes (e.g., repetition loops, word-order shifts, clause omissions).
* **Autoregressive Generation**: Uses greedy decoding to output hypothesis token IDs.
* **String Conversion & Inspection**: Decodes token IDs back to human-readable strings to qualitative assess translation quality.

In [ ]:
##Code here##

* **A2. Word-Order Divergence (SVO vs. SOV):[6 marks]** Find 3 test examples where your no-attention model's hypothesis exhibits incorrect word order relative to the target reference (due to English SVO vs. Hindi SOV divergence). Describe specifically how the lack of dynamic alignment caused the verb or object placement to fail.


In [ ]:
##Code here##

* **A3. Loss-Metric Discrepancy:[6 marks]** Compare `hist_teacher_forcing` and `hist_scheduled_sampling`, and report greedy-decoding BLEU for `seq2seq_teacher_forcing` and `seq2seq_scheduled_sampling`. Does a lower teacher-forced validation loss necessarily imply a higher autoregressive BLEU score? Support your answer with your empirical results.



In [ ]:
##Code here##

Experimental (Requires Code Execution & Observations)

* **E1. Teacher-Forcing Ratio Sweep:[6 marks]** Train separate Seq2Seq models with `teacher_forcing_ratio` set to $\{1.0, 0.7, 0.3\}$. Plot validation loss and test BLEU against the ratio, and discuss which setting yielded the best balance between training stability and greedy generation quality.


Before filling out the code below, familiarize yourself with these core components:

* **Teacher-Forcing Ratios (`tf_ratios`)**: Evaluates multiple ratio settings ($\in \{1.0, 0.7, 0.3\}$) to examine their impact on validation loss and autoregressive generation performance.
* **Model Retraining & Evaluation**: Instantiates, trains, and tests fresh encoder-decoder translation models across each ratio setting.
* **Comparative Plotting**: Plots validation loss curves and test BLEU score variations to determine the optimal trade-off between guided training stability and free-running inference accuracy.

In [ ]:
##Code here##

* **E2. Source-Sequence Reversal Experiment:[7 marks]** Reverse the token order of every English sentence fed to the encoder (`TranslationDatasetReversed`) following Sutskever et al. (2014) and retrain the model from scratch. Given that your encoder is *already* bidirectional, did you expect—and actually observe—a significant BLEU performance gain? Explain why.


Before filling out the code below, familiarize yourself with these core components:

* **Source-Token Reversal (`TranslationDatasetReversed`)**: Inherits from `TranslationDataset` but reverses the token order of the source English sentence while leaving the target Hindi sequence untouched, following Sutskever et al. (2014).
* **Short-Term Dependency Improvement**: Reversing source sentences creates short-term dependencies between the beginning of the source and target sequences, easing optimization for standard unidirectional RNNs.
* **Bidirectional Encoder Context**: Retrains a fresh encoder-decoder model on the reversed data loaders to evaluate whether source reversal still provides tangible gains when the encoder is *already* bidirectional.

In [ ]:
##Code here##

* **E3. Strategy Comparison & Generation Inspection:[7 marks]** Run the strict-teacher-forcing and scheduled-sampling training routines. Plot their training/validation loss curves alongside the scheduled model's $p_{\mathrm{TF}}$ decay curve by epoch, and compare their greedy-decoding BLEU across at least five qualitative translation examples.

Before filling out the code below, familiarize yourself with these core components:

* **Training Strategies (`seq2seq_teacher_forcing` vs. `seq2seq_scheduled_sampling`)**: Compares strict teacher forcing ($p_{\mathrm{TF}} = 1.0$) against scheduled sampling with a linear decay curriculum ($p_{\mathrm{TF}} \text{ from } 1.0 \text{ to } 0.25$).
* **Loss & Curriculum Visualization**: Plots training/validation loss curves alongside the scheduled model's $p_{\mathrm{TF}}$ decay curve across epochs.
* **Qualitative Generation Inspection**: Compares greedy-decoding translation outputs against target references across at least five qualitative test samples to assess exposure bias recovery.

In [ ]:
##Code here##

# **Question 3 — Attention Mechanisms from Scratch[Total = 140 marks]**

3. Implement the attention (try both dot-product and badhnau) in the encoder-decoder model above from scratch and train it, analyze the improvement in the performance, compare with the model above step and do error analysis



In Question 2, we built a standard Seq2Seq model where the encoder compressed the entire source sentence into a single, fixed-size vector $C$. In above section, we saw the main weakness of that design: as sentence length increases, translation quality collapses. A single vector simply cannot hold all the details of a long sentence.

In this section, we solve that bottleneck by introducing **Attention**.

Instead of forcing the encoder to compress everything into one vector, we allow the decoder to look back at **all encoder hidden states** at every step of generation. At each step $t$, the model computes a fresh **context vector** $c_t$ by taking a weighted sum of all encoder outputs $h_1, h_2, \ldots, h_{T_x}$.

```
Encoder Outputs:    h_1        h_2        h_3        h_4
                     \          |          |          /
                      \         |          |         /
                       v        v          v        v
Scores & Softmax:   [a_{t,1}] [a_{t,2}] [a_{t,3}] [a_{t,4}]  <-- Weights depend on s_{t-1}
                       \        |          |        /
                        v       v          v       v
Context Vector:                     c_t                      <-- Fresh context at step t!
                                     |
Decoder State:                    s_{t-1}

```

Mathematically, at each decoding step $t$:

1. **Score:** Compare the previous decoder hidden state $s_{t-1}$ with each encoder output $h_j$:

$$e_{t,j} = \text{score}(s_{t-1}, h_j)$$


2. **Normalize (Softmax):** Turn raw scores into attention probabilities $\alpha_{t,j}$ that sum to 1:

$$\alpha_{t,j} = \frac{\exp(e_{t,j})}{\sum_{k=1}^{T_x} \exp(e_{t,k})}$$


3. **Blend:** Compute the context vector $c_t$ as a weighted combination of encoder outputs:

$$c_t = \sum_{j=1}^{T_x} \alpha_{t,j} h_j$$



We will implement two distinct scoring functions ($\text{score}$): **Dot-Product (Multiplicative) Attention** and **Bahdanau (Additive) Attention**.

####Dot-Product (Multiplicative) Attention[20 marks]

**What Is It ?**

Introduced by [Luong et al. (2015)](https://arxiv.org/abs/1508.04025), **Dot-Product Attention** is the simplest way to compute alignment scores. It calculates the dot product between the decoder state $s_{t-1}$ and each encoder output $h_j$:

$$e_{t,j} = s_{t-1}^\top h_j$$

**Why Do We Need It?**

Because our encoder is bidirectional (output dimension $2 \cdot H$) and our decoder is unidirectional (hidden dimension $H$), their vector dimensions do not match. To fix this, we project the decoder query into the encoder space using a single linear layer $W_{\text{proj}}$ before taking the dot product:

$$q = W_{\text{proj}}(s_{t-1}) \in \mathbb{R}^{2H}, \quad e_{t,j} = q^\top h_j$$



Before filling out the code below, familiarize yourself with these core components:

* **`DotProductAttention` (`nn.Module`)**: Implements multiplicative attention (Luong et al., 2015), aligning decoder hidden queries with encoder output keys.
* **Linear Projection Layer (`self.proj`)**: Projects the unidirectional decoder hidden dimension ($H$) to match the bidirectional encoder hidden dimension ($2H$).
* **Query Projection (`q`)**: Transformed decoder query vector used to measure alignment against encoder states.
* **Score Calculation (`scores`)**: Computes the raw dot-product similarity between the projected query and all encoder output states (`q @ enc_outputs.transpose(1, 2)`).
* **Padding Masking (`mask`)**: Optionally masks out padding token positions by setting their scores to negative infinity (`-1e9` or `-inf`), ensuring softmax assigns them zero attention weight.
* **Softmax Normalization (`weights`)**: Converts raw masked scores into probability distributions over the source sequence.
* **Context Vector (`context`)**: Computes the final context representation as a weighted sum of encoder outputs (`weights.unsqueeze(1) @ enc_outputs`).

In [ ]:


# --- Dot-Product (Multiplicative) Attention ---
# Linear projection to match query dimension to key dimension
# 1. Project decoder hidden state: [batch_size, 1, enc_hidden]
# 2. Compute raw dot-product scores: [batch_size, src_len]
# 3. Apply padding mask (set <pad> scores to -infinity so softmax gives them 0 weight)
# 4. Softmax normalization: [batch_size, src_len]
# 5. Compute weighted sum of encoder outputs: [batch_size, enc_hidden]

class DotProductAttention(nn.Module):
    """Multiplicative / Dot-Product Attention (Luong et al., 2015).

    Formula:
        q = Linear(dec_hidden)             # Project query to match enc_hidden dim
        scores = q @ enc_outputs^T         # Dot product over source sequence
        weights = softmax(scores)          # Normalize over source timesteps
        context = weights @ enc_outputs    # Weighted sum of encoder outputs
    """
    def __init__(self, dec_hidden: int, enc_hidden: int):
        """
        Args:
            dec_hidden (int): Decoder hidden dimension H.
            enc_hidden (int): Encoder output dimension (2H for bidirectional).
        """
        super().__init__()
        # TODO 1: Define self.proj linear layer mapping dec_hidden to enc_hidden
        self.proj = __________________________

    def forward(self, dec_hidden: torch.Tensor, enc_outputs: torch.Tensor, mask: torch.Tensor = None):
        """
        Args:
            dec_hidden (torch.Tensor): Current decoder query s_{t-1} [batch_size, dec_hidden]
            enc_outputs (torch.Tensor): Encoder states h_j [batch_size, src_len, enc_hidden]
            mask (torch.Tensor, optional): Binary mask [batch_size, src_len], 1 for real tokens, 0 for <pad>

        Returns:
            context (torch.Tensor): Context vector c_t [batch_size, enc_hidden]
            weights (torch.Tensor): Attention weights alpha_{t,j} [batch_size, src_len]
        """
        # TODO 2: Define q (project decoder hidden state and add sequence dimension if needed)
        q = __________________________

        # TODO 3: Define scores by computing batched matrix multiplication between q and enc_outputs transposed
        scores = __________________________

        # TODO 4: Handle the relevant condition (check if mask is provided)
        if __________________________:
            # TODO 5: Define scores (mask out padding positions by setting them to a large negative value like -1e9)
            scores = __________________________

        # TODO 6: Define weights by applying softmax along the sequence dimension
        weights = __________________________

        # TODO 7: Define context by computing batched matrix multiplication between weights and enc_outputs
        context = __________________________

        return (context, weights)

**Limitations of Dot-Product Attention**

Dot-product attention assumes that after a linear projection, query vectors and key vectors live in a comparable space where a simple inner product measures similarity. It cannot capture more complex, non-linear interactions between encoder and decoder states.

### Bahdanau (Additive) Attention[15 Marks]

### What Is It and Why Do We Need It?

Introduced in the paper by [Bahdanau et al. (2014)](https://arxiv.org/abs/1409.0473), **Additive Attention** uses a small single-layer feedforward neural network to calculate alignment scores:

$$e_{t,j} = v_a^\top \tanh\left(W_a s_{t-1} + U_a h_j\right)$$

Where:

* $W_a \in \mathbb{R}^{A \times H_{\text{dec}}}$ projects the decoder query into an attention space of dimension $A$.
* $U_a \in \mathbb{R}^{A \times H_{\text{enc}}}$ projects the encoder key into the same attention space $A$.
* $v_a \in \mathbb{R}^{A \times 1}$ collapses the non-linear combination into a scalar score.

Instead of assuming queries and keys live in similar spaces, Bahdanau attention learns a dedicated mapping. This flexibility allows it to learn complex non-linear relationships, though at the cost of adding a few more parameters.

Before filling out the code below, familiarize yourself with these core components:

* **`BahdanauAttention` (`nn.Module`)**: Implements additive attention (Bahdanau et al., 2014), which computes non-linear alignment scores using a feedforward network with a single hidden layer.
* **Projections (`self.W_a`, `self.U_a`)**: Linear layers projecting the decoder state and encoder outputs into a shared intermediate attention dimension (`attn_dim`).
* **Energy Vector (`self.v_a`)**: A vector layer that combines the activated sum to yield scalar alignment scores.
* **Non-Linearity (`tanh`)**: Applies a hyperbolic tangent activation on the combined projections, capturing complex interactions between queries and keys.
* **Masking & Normalization**: Applies padding masks and softmax normalization identically to Dot-Product Attention for seamless drop-in compatibility.

In [ ]:

# --- Bahdanau (Additive) Attention ---
# Project query and keys into shared space
# Combine additively with broadcasting and apply tanh non-linearity
# Mask padding tokens
# Normalize and compute context vector

class BahdanauAttention(nn.Module):
    """Additive Attention (Bahdanau et al., 2014).

    Formula:
        score = v_a^T * tanh(W_a * dec_hidden + U_a * enc_outputs)
    """
    def __init__(self, dec_hidden: int, enc_hidden: int, attn_dim: int = 128):
        """
        Args:
            dec_hidden (int): Decoder hidden dimension.
            enc_hidden (int): Encoder output dimension.
            attn_dim (int): Inner projection dimension A (default: 128).
        """
        super().__init__()
        # TODO 1: Define self.W_a linear projection for decoder hidden state
        self.W_a = __________________________

        # TODO 2: Define self.U_a linear projection for encoder outputs
        self.U_a = __________________________

        # TODO 3: Define self.v_a linear/vector projection from attn_dim to 1
        self.v_a = __________________________

    def forward(self, dec_hidden: torch.Tensor, enc_outputs: torch.Tensor, mask: torch.Tensor = None):
        """Same interface as DotProductAttention for easy drop-in replacement."""
        # TODO 4: Define q (project decoder hidden state and expand dimensions for broadcasting)
        q = __________________________

        # TODO 5: Define k (project encoder outputs)
        k = __________________________

        # TODO 6: Define energy by adding projected query and keys, applying tanh, and projecting via v_a
        energy = __________________________

        # TODO 7: Define scores (squeeze the last dimension to get shape [batch_size, src_len])
        scores = __________________________

        # TODO 8: Handle the relevant condition (check if mask is provided)
        if __________________________:
            # TODO 9: Define scores (mask out padding positions with a large negative value like -1e9)
            scores = __________________________

        # TODO 10: Define weights by applying softmax along the sequence dimension
        weights = __________________________

        # TODO 11: Define context by computing weighted sum of encoder outputs
        context = __________________________

        return (context, weights)

####Attention-Augmented Decoder[10 marks]

Now we modify our step-wise decoder to use attention. At each step $t$:

1. Pass the top-layer hidden state $s_{t-1}$ and all encoder outputs $h_j$ into the attention module to get context $c_t$.
2. Concatenate the target token embedding with $c_t$ as input to the GRU cells.
3. Use **input feeding** ([Luong et al., 2015](https://arxiv.org/abs/1508.04025)): concatenate the new hidden state $s_t$ with $c_t$ before projecting to vocabulary logits.

Before filling out the code below, familiarize yourself with these core components:

* **`AttnDecoderStep` (`nn.Module`)**: A step-wise unidirectional decoder that incorporates dynamic attention mechanisms at every generation step.
* **Context Vector Generation**: Evaluates the attention module using the previous top-layer hidden state $s_{t-1}$ and all encoder outputs $h_j$ to produce $c_t$.
* **Target Embedding & Context Concatenation**: Combines the embedded target token with the context vector (`[token_embedding ; context_vector]`) as the input for the bottom GRU cell.
* **Input Feeding (`Luong et al., 2015`)**: Concatenates the new hidden state $s_t$ with the context vector $c_t$ before projecting to vocabulary logits, providing the output layer with explicit historical awareness of past attention decisions.

In [ ]:


# --- Attention-Augmented Decoder Step ---
class AttnDecoderStep(nn.Module):
    """Step-wise decoder augmented with dynamic attention and input feeding."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, enc_hidden: int, attention_module: nn.Module, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.hidden_size
        self.hidden_size = __________________________

        # TODO 3: Define self.embedding layer for target tokens
        self.embedding = __________________________

        # TODO 4: Define self.attention module (DotProductAttention or BahdanauAttention)
        self.attention = __________________________

        # TODO 5: Define self.cells using nn.ModuleList, where the first cell accepts input dimension (emb_dim + enc_hidden)
        self.cells = __________________________

        # TODO 6: Define self.out linear layer projecting (hidden_size + enc_hidden) to vocab_size
        self.out = __________________________

    def forward(self, tok: torch.Tensor, hidden_states: list, enc_outputs: torch.Tensor, mask: torch.Tensor = None):
        """
        Advances the attention decoder by exactly one step t.
        """
        # TODO 7: Define x by embedding the current token IDs [batch_size, emb_dim]
        x = __________________________

        # TODO 8: Define context, weights by calling self.attention using the top-layer hidden state
        context, weights = __________________________

        # TODO 9: Define inp by concatenating embedding x and context vector along dim=1
        inp = __________________________

        # TODO 10: Define new_states list to track updated hidden states per layer
        new_states = __________________________

        # TODO 11: Iterate over the required data (enumerate through self.cells)
        for l, cell in __________________________:
            # TODO 12: Define h (extract previous hidden state for layer l)
            h = __________________________

            # TODO 13: Perform the required operation (pass current input and hidden state through cell l)
            __________________________

            # TODO 14: Define inp for the next layer (or keep top hidden state for output projection)
            inp = __________________________

        # TODO 15: Define logits by concatenating the top layer's hidden state and context vector, then passing through self.out
        logits = __________________________

        return (logits, new_states, weights)

#### Attention Seq2Seq Wrapper & Model Training[5 marks]

Why Source Padding Masking Is Essential

When a batch contains sentences of different lengths, shorter sentences are padded with `<pad>` tokens. Without a mask, the softmax function assigns positive attention weights to these `<pad>` tokens.

By applying `mask = (src != stoi_en[PAD])`, we force the pre-softmax scores of padding positions to $-\infty$, ensuring that `softmax(-∞) = 0`. Padding tokens receive **zero attention weight**.

Before filling out the code below, familiarize yourself with these core components:

Attention-Augmented Seq2Seq (Seq2SeqAttn): Integrates an encoder, step-wise attention decoder, and source padding mask into a unified translation architecture that computes dynamic context vectors at every generation step.

Source Padding Masking (mask): Creates a binary mask (1 for real tokens, 0 for <pad>) to ensure that padding tokens receive absolute zero attention weights during softmax alignment.

Input Feeding & Teacher Forcing: Dynamically blends gold-standard target tokens with model predictions during training based on a configurable teacher_forcing_ratio.

Attention Heatmap Extraction (greedy_decode): Autoregressively generates predictions while capturing and stacking attention weight matrices across timesteps for soft-alignment visualization.

In [ ]:

# --- Attention Seq2Seq Wrapper & Training Pipeline ---
class Seq2SeqAttn(nn.Module):
    """Full Seq2Seq wrapper integrating Encoder, Attention Decoder, Masking, and Greedy Inference."""
    def __init__(self, encoder: Encoder, decoder: AttnDecoderStep, num_layers: int = 2):
        super().__init__()
        # TODO 1: Assign encoder argument to self.encoder
        self.encoder = __________________________

        # TODO 2: Assign decoder argument to self.decoder
        self.decoder = __________________________

        # TODO 3: Assign num_layers argument to self.num_layers
        self.num_layers = __________________________

    def forward(self, src: torch.Tensor, tgt_in: torch.Tensor, teacher_forcing_ratio: float = 0.7, return_attn: bool = False):
        # TODO 4: Pass source tensor through encoder to retrieve enc_outputs and initial decoder hidden state dec_h0
        enc_outputs, dec_h0 = __________________________

        # TODO 5: Create a binary mask where real tokens are 1.0 and <pad> tokens are 0.0 using (src != stoi_en[PAD]).float()
        mask = __________________________

        # TODO 6: Unpack batch_size and tgt_len dimensions from target input tensor shape tgt_in.shape
        batch_size, tgt_len = __________________________

        # TODO 7: Initialize hidden_states list containing dec_h0 replicated across all num_layers
        hidden_states = __________________________

        # TODO 8: Initialize empty list containers logits_seq and attn_seq to store step outputs and attention weights
        logits_seq, attn_seq = __________________________

        # TODO 9: Extract initial <sos> tokens for the batch from the first column of tgt_in (tgt_in[:, 0])
        inp_tok = __________________________

        # TODO 10: Loop through target sequence length indices t from 0 up to tgt_len
        for t in range(__________________________):
            # TODO 11: Execute one decoding step using self.decoder passing inp_tok, hidden_states, enc_outputs, and mask
            logits, hidden_states, weights = __________________________

            # TODO 12: Append current step logits tensor to logits_seq list
            __________________________

            # TODO 13: Append current step attention weights tensor to attn_seq list
            __________________________

            # TODO 14: Check if current step index plus one is less than tgt_len to determine if next input token is needed
            if __________________________:
                # TODO 15: Determine whether to use teacher forcing based on random probability and self.training flag
                use_tf = __________________________

                # TODO 16: Select next input token: use gold target token from tgt_in[:, t + 1] if use_tf else argmax prediction from logits
                inp_tok = __________________________

        # TODO 17: Stack logits_seq tensors along dimension 1 to form full output logits tensor out
        out = __________________________

        # TODO 18: Check if return_attn is True; if so, return out alongside attention weights stacked along dimension 1
        if __________________________:
            return (out, torch.stack(attn_seq, dim=1))
        return out

    @torch.no_grad()
    def greedy_decode(self, src_tensor: torch.Tensor, max_len: int = MAX_LEN_HI):
        """Autoregressive decoding that returns predicted token IDs and attention weights for heatmaps."""
        # TODO 19: Set model to evaluation mode using self.eval()
        __________________________

        # TODO 20: Move source tensor to computation device using src_tensor.to(DEVICE)
        src_tensor = __________________________

        # TODO 21: Extract encoder sequence outputs enc_outputs and initial hidden state dec_h0 from encoder
        enc_outputs, dec_h0 = __________________________

        # TODO 22: Generate source sequence binary mask tensor using (src_tensor != stoi_en[PAD]).float()
        mask = __________________________

        # TODO 23: Initialize hidden_states list for all decoder layers using dec_h0
        hidden_states = __________________________

        # TODO 24: Initialize batch input tokens tensor inp_tok filled with SOS token index for every batch sample
        inp_tok = __________________________

        # TODO 25: Initialize empty lists out_tokens and out_attn to store autoregressive results
        out_tokens, out_attn = [], []

        # TODO 26: Loop autoregressively up to max_len timesteps using range(max_len)
        for _ in range(__________________________):
            # TODO 27: Execute decoder step returning logits, updated hidden_states, and attention weights weights
            logits, hidden_states, weights = __________________________

            # TODO 28: Compute greedy token prediction via argmax along the last dimension of logits
            inp_tok = __________________________

            # TODO 29: Move predicted token tensor to CPU and append to out_tokens list
            __________________________

            # TODO 30: Move attention weights tensor to CPU and append to out_attn list
            __________________________

        return (torch.stack(out_tokens, dim=1), torch.stack(out_attn, dim=1))

def build_and_train_attn_model(attn_cls, tag: str, epochs: int = 4, checkpoint_filename: str = None):
    """Instantiates and trains an attention-augmented Seq2Seq model."""
    # TODO 31: Set manual random seed for reproducibility using torch.manual_seed(SEED)
    __________________________

    # TODO 32: Instantiate Encoder with source vocabulary size, embedding dimension 200, hidden size 256, and 2 layers
    enc = __________________________

    # TODO 33: Instantiate attention module class attn_cls passing dec_hidden=256 and enc_hidden=512
    attn = __________________________

    # TODO 34: Instantiate AttnDecoderStep passing target vocab size, embedding dim 200, hidden size 256, enc_hidden 512, attn module, and num_layers=2
    dec = __________________________

    # TODO 35: Wrap encoder and decoder into Seq2SeqAttn model instance placed on DEVICE
    model = __________________________

    # TODO 36: Train model using train_seq2seq function passing train_loader, val_loader, epochs, and tag
    history = __________________________

    # TODO 37: Check if checkpoint_filename parameter is provided/truthy
    if __________________________:
        # TODO 38: Save encoder and decoder state dicts to checkpoint_filename using torch.save
        __________________________

        # TODO 39: Print success message confirming checkpoint file has been saved
        __________________________

    return (model, history)

# TODO 40: Build and train Dot-Product attention model by calling build_and_train_attn_model with DotProductAttention
seq2seq_dot, hist_dot = __________________________

# TODO 41: Build and train Bahdanau attention model by calling build_and_train_attn_model with BahdanauAttention
seq2seq_bah, hist_bah = __________________________

#### **BLEU Score Evaluation[5 Marks]**

Before filling out the code below, familiarize yourself with these core components:

* **Definition of BLEU (Bilingual Evaluation Understudy)**: A metric for automatically evaluating machine-translated text against one or more human reference translations by measuring n-gram precision with a brevity penalty. Detailed information and standard references can be found at the [SacreBLEU GitHub Repository](https://github.com/mjpost/sacrebleu).
* **`evaluate_bleu_attn`**: Computes SacreBLEU evaluation scores across test datasets specifically for attention-augmented models using greedy decoding.
* **Hypothesis & Reference Accumulation**: Collects generated token strings and matches them against ground-truth target sentences to calculate precise n-gram precision metrics.

In [ ]:


# --- SacreBLEU Evaluation for Attention Models ---
def evaluate_bleu_attn(model: nn.Module, loader: DataLoader):
    """Computes SacreBLEU score for attention models."""
    # TODO 1: Define hypotheses, references lists
    hypotheses, references = __________________________

    # TODO 2: Iterate over the required data (loop through test loader batches)
    for src, _, tgt_out in __________________________:
        # TODO 3: Define pred_ids, _ by calling model.greedy_decode
        pred_ids, _ = __________________________

        # TODO 4: Iterate over the required data (loop through batch elements)
        for b in range(__________________________):
            # TODO 5: Define hyp_text using ids_to_string conversion helper
            hyp_text = __________________________

            # TODO 6: Define ref_text by converting ground-truth target tokens
            ref_text = __________________________

            # TODO 7: Perform the required operation (append hypothesis string)
            __________________________

            # TODO 8: Perform the required operation (append reference string)
            __________________________

    # TODO 9: Define bleu score using sacrebleu metric wrapper
    bleu = __________________________
    return (bleu.score, hypotheses, references)

# TODO 10: Define bleu_dot, hyps_dot, refs_dot for Dot-Product Attention model
bleu_dot, hyps_dot, refs_dot = __________________________

# TODO 11: Define bleu_bah, hyps_bah, refs_bah for Bahdanau Attention model
bleu_bah, hyps_bah, refs_bah = __________________________

# TODO 12: Perform the required operation (print Dot-Product Attention test BLEU score)
__________________________

# TODO 13: Perform the required operation (print Bahdanau Attention test BLEU score)
__________________________

# TODO 14: Perform the required operation (compare baseline vs attention BLEU improvements)
__________________________

# TODO 15: Perform the required operation (plot validation loss comparison curves)
__________________________

# TODO 16: Perform the required operation (generate and plot sample attention heatmaps)
__________________________

####Decoding Strategies for Autoregressive Inference[20 Marks][Refer to Lecture Notes]

At inference, each decoder step produces logits over the Hindi vocabulary. A decoding strategy turns those per-step logits into a sequence. The existing `greedy_decode` methods remain unchanged; the functions below are additional utilities for the existing `Seq2SeqAttn` models.

### 1. Greedy Decoding

Greedy decoding selects $\arg\max_v\,p(v\mid y_{<t},x)$ at every step. It is deterministic and inexpensive, but it is myopic: an early locally likely token can rule out a better complete translation, and the same input always produces the same output.

### 2. Beam Search

Beam search keeps the best $k$ partial hypotheses according to cumulative log-probability, expanding and pruning them at each step. It explores alternatives to one greedy path, but costs roughly $k$ decoder expansions per step. Length normalization is included below because unnormalized sums of negative log-probabilities tend to favor short hypotheses. For neural machine translation beam-search analysis, see [Freitag and Al-Onaizan (2017)](https://aclanthology.org/W17-3207/).

### 3. Random Sampling with Temperature

Random sampling draws from $\operatorname{softmax}(z/T)$, where $z$ are logits and $T>0$. Lower $T$ sharpens the distribution toward greedy behavior; higher $T$ flattens it and increases variety, but can also select less suitable tokens.

### 4. Top-$k$ Sampling

Top-$k$ sampling retains only the $k$ highest-probability tokens, renormalizes their probabilities, and samples from that set. It prevents very low-probability tail tokens from being chosen, although a fixed $k$ may be too restrictive for a sharp distribution or too broad for a flat one.

### 5. Top-$p$ (Nucleus) Sampling

Top-$p$ sampling retains the smallest token set whose cumulative probability is at least $p$, then samples after renormalization. The retained set adapts to the uncertainty of each step. [Holtzman et al. (2020)](https://arxiv.org/abs/1904.09751) introduced nucleus sampling in their study of decoding degeneration. These stochastic strategies are often most useful for open-ended generation; for a constrained translation task, BLEU and adequacy should be evaluated rather than assumed.


Before filling out the code below, familiarize yourself with these core components:

* **Greedy Decoding**: Selects the single highest-probability token at each step. It is fast and deterministic, but prone to myopic errors.
* **Beam Search**: Maintains the top $k$ partial hypotheses to balance exploration and sequence likelihood, utilizing length normalization as detailed by [Freitag and Al-Onaizan (2017)](https://aclanthology.org/W17-3207/).
* **Temperature Sampling**: Scales logits by a temperature factor $T$ to control randomness and distribution sharpness.
* **Top-$k$ & Nucleus (Top-$p$) Sampling**: Restricts sampling pools to the top $k$ candidates or a cumulative probability threshold $p$, mitigating degeneration as explored by [Holtzman et al. (2020)](https://arxiv.org/abs/1904.09751).

In [ ]:
def count_repetitive_hypotheses(hypotheses_list, repeat_threshold=3):
    """Scans hypotheses for instances where the same word repeats consecutively."""
    # TODO 1: Initialize an empty list to store indices of repetitive hypotheses
    repetitive_indices = __________________________

    # TODO 2: Iterate over hypotheses using enumerate to track both index and text
    for idx, hyp in __________________________:
        # TODO 3: Split the hypothesis text string into individual word tokens
        words = __________________________

        # TODO 4: Loop through the words to check windows of size 'repeat_threshold'
        for i in __________________________:
            # TODO 5: Check if all words in the current window are identical (e.g., set length equals 1)
            if __________________________:
                # TODO 6: Append the index of the hypothesis to repetitive_indices
                __________________________
                break

    # TODO 7: Return the final list of repetitive indices
    return __________________________

In [ ]:

# ---  decoding strategies for existing Seq2SeqAttn models ---
@torch.no_grad()
def _sample_attn_next_token(logits, strategy='greedy', temperature=1.0, top_k=10, top_p=0.9):
    """Selects one token per row from decoder logits without changing model.greedy_decode."""
    # TODO 1: Check if temperature is less than or equal to 0
    if __________________________:
        # TODO 2: Raise a ValueError indicating temperature must be greater than 0
        __________________________

    # TODO 3: Check if strategy string equals "greedy"
    if __________________________:
        # TODO 4: Return argmax token selection across the last dimension of logits
        return __________________________

    # TODO 5: Compute softmax probability distribution from logits scaled by temperature (logits / temperature)
    probs = __________________________

    # TODO 6: Check if strategy equals "sample"
    if __________________________:
        # TODO 7: Sample indices categorically from probs using torch.multinomial with num_samples=1
        return __________________________

    # TODO 8: Check if strategy equals "top_k"
    if __________________________:
        # TODO 9: Check if top_k parameter is less than 1 and raise ValueError if invalid
        if __________________________:
            # TODO 10: Raise ValueError for top_k boundary rules
            __________________________

        # TODO 11: Define k as the minimum of top_k and the final dimension size of probs
        k = __________________________

        # TODO 12: Extract top-k highest probability values and indices along dimension -1 using torch.topk
        values, indices = __________________________

        # TODO 13: Sample locally from re-normalized top-k values using multinomial sampling
        sampled_local = __________________________

        # TODO 14: Gather and return the global token indices corresponding to sampled local positions
        return __________________________

    # TODO 15: Check if strategy equals "top_p" (nucleus sampling)
    if __________________________:
        # TODO 16: Validate that top_p lies strictly within the interval (0, 1]
        if __________________________:
            # TODO 17: Raise ValueError if top_p is out of bounds
            __________________________

        # TODO 18: Sort probabilities in descending order along the last dimension using torch.sort
        sorted_probs, sorted_indices = __________________________

        # TODO 19: Compute cumulative probability mask (keep) for tokens exceeding the top_p nucleus threshold
        keep = __________________________

        # TODO 20: Ensure the most likely token is always retained by setting keep[:, 0] to True
        keep[:, 0] = __________________________

        # TODO 21: Filter sorted probabilities by multiplying with the boolean keep mask
        filtered = __________________________

        # TODO 22: Sample locally from re-normalized filtered probabilities using torch.multinomial
        sampled_local = __________________________

        # TODO 23: Gather and return global token indices corresponding to sampled local positions in sorted_indices
        return __________________________

    # TODO 24: Raise ValueError if an unsupported decoding strategy string is passed
    __________________________

@torch.no_grad()
def decode_attn_with_strategy(model, src_tensor, max_len=MAX_LEN_HI, strategy='greedy', **kwargs):
    """Batch-capable greedy, temperature, top-k, or nucleus decoding for Seq2SeqAttn."""
    # TODO 25: Set model to evaluation mode using model.eval()
    __________________________

    # TODO 26: Move source tensor to computation device using src_tensor.to(DEVICE)
    src_tensor = __________________________

    # TODO 27: Pass source tensor through encoder to retrieve enc_outputs and initial decoder hidden state dec_h0
    enc_outputs, dec_h0 = __________________________

    # TODO 28: Generate binary padding mask for source tensor using (src_tensor != stoi_en[PAD]).float()
    mask = __________________________

    # TODO 29: Initialize hidden_states list replicating dec_h0 across all model.num_layers
    hidden_states = __________________________

    # TODO 30: Initialize next_input token tensor filled with SOS token index for each row in the batch
    next_input = __________________________

    # TODO 31: Initialize finished boolean tensor of shape [batch_size] filled with False to track EOS states
    finished = __________________________

    # TODO 32: Initialize an empty outputs list to collect step-wise predicted tokens
    outputs = __________________________

    # TODO 33: Loop autoregressively through generation timesteps up to max_len
    for _ in range(__________________________):
        # TODO 34: Run decoder step passing next_input, hidden_states, enc_outputs, and mask
        logits, hidden_states, _ = __________________________

        # TODO 35: Sample next token IDs using _sample_attn_next_token helper function passing logits, strategy, and kwargs
        next_input = __________________________

        # TODO 36: Force next_input to EOS token for already finished batch samples using torch.where
        next_input = __________________________

        # TODO 37: Move predicted token tensor to CPU and append to outputs list
        __________________________

        # TODO 38: Update finished tensor tracking using logical OR with token equality check against EOS index
        finished = __________________________

        # TODO 39: Check if all sequences in the batch are finished (finished.all()); if so, break out of loop
        if __________________________:
            break

    # TODO 40: Stack collected outputs along dimension 1 and return tensor
    return torch.stack(__________________________)

@torch.no_grad()
def beam_search_decode_attn(model, src_tensor, beam_width=4, max_len=MAX_LEN_HI, length_penalty=0.6):
    """Length-normalized beam search for one source sentence and a Seq2SeqAttn model."""
    # TODO 41: Check if beam_width is less than 1
    if __________________________:
        # TODO 42: Raise a ValueError if beam_width is invalid
        __________________________

    # TODO 43: Set model to evaluation mode using model.eval()
    __________________________

    # TODO 44: Move source tensor to computation device using src_tensor.to(DEVICE)
    src_tensor = __________________________

    # TODO 45: Check if source tensor dimension equals 1; if so, unsqueeze to add batch dimension
    if __________________________:
        # TODO 46: Unsqueeze source tensor at dimension 0
        src_tensor = __________________________

    # TODO 47: Check if batch size of src_tensor is not equal to 1, raising ValueError if multiple sentences are passed
    if __________________________:
        # TODO 48: Raise ValueError enforcing single-sentence limit for beam search
        __________________________

    # TODO 49: Extract encoder sequence outputs enc_outputs and initial state dec_h0 from encoder
    enc_outputs, dec_h0 = __________________________

    # TODO 50: Generate source padding mask tensor using (src_tensor != stoi_en[PAD]).float()
    mask = __________________________

    # TODO 51: Initialize initial_states list across model.num_layers using dec_h0
    initial_states = __________________________

    # TODO 52: Initialize beams list containing tuple of (tokens_list, log_prob, input_token, states_list)
    beams = __________________________

    def normalized_score(tokens, log_prob):
        # TODO 53: Compute length-normalized beam score using log_prob divided by sequence length raised to length_penalty
        return __________________________

    # TODO 54: Loop through max_len decoding steps using range(max_len)
    for _ in range(__________________________):
        # TODO 55: Initialize empty candidates list to store expanded beam paths
        candidates = __________________________

        # TODO 56: Iterate over active beams (tokens, log_prob, input_token, states) in beams list
        for tokens, log_prob, input_token, states in __________________________:
            # TODO 57: Check if sequence is already completed (tokens[-1] == stoi_hi[EOS])
            if __________________________:
                # TODO 58: Append completed beam directly to candidates list and continue
                __________________________
                continue

            # TODO 59: Wrap input_token integer into a tensor placed on DEVICE
            token = __________________________

            # TODO 60: Run decoder step passing token, states, enc_outputs, and mask
            logits, new_states, _ = __________________________

            # TODO 61: Compute log softmax probabilities across vocabulary dimension and squeeze batch dimension
            log_probs = __________________________

            # TODO 62: Extract top-k values and token IDs using torch.topk with k=min(beam_width, log_probs.numel())
            values, ids = __________________________

            # TODO 63: Iterate over expanded candidate score values and token IDs
            for value, token_id in zip(__________________________):
                # TODO 64: Append new candidate tuple to candidates list with updated tokens, accumulated log probability, and cloned states
                __________________________

        # TODO 65: Sort candidates by normalized score in descending order and keep top beam_width beams
        beams = sorted(__________________________)[:beam_width]

        # TODO 66: Check if all active beams have terminated with EOS; if so, break generation loop
        if __________________________:
            break

    # TODO 67: Extract the highest scoring beam sequence from beams using max score and normalized_score key
    best_tokens, _, _, _ = max(__________________________)

    # TODO 68: Return best token sequence wrapped into a 2D tensor of shape [1, seq_len]
    return torch.tensor(__________________________)

####Attention Heatmap Visualization[5 marks]

Plotting attention weights as a heatmap shows which English source words the model focuses on while generating each Hindi word.

Before filling out the code below, familiarize yourself with these core components:

* **Alignment Tracking**: Visualizes the weight matrix mapping target words to source tokens across decoding timesteps.
* **Heatmap Rendering**: Automatically handles token slice filtering up to the `<eos>` boundary to generate readable diagnostic plots.

Before filling out the code below, familiarize yourself with these core components:

* **Alignment Matrix Visualization (`plot_attention_heatmap`)**: Decodes an individual source sentence and plots the resulting attention weight matrix to visualize which source words the decoder focuses on during target generation.
* **Token and Boundary Management**: Manages sequence indexing up to the `<eos>` boundary to filter out padding noise in visualization plots.
* **Sample Diagnostic Suite**: Selects representative sentences from the test set to render and inspect soft alignments.

In [ ]:


# --- Attention Heatmap Visualization ---
def plot_attention_heatmap(model, src_text: str, idx_tag: str = ''):
    """Decodes a single English sentence and plots its attention matrix."""
    # TODO 1: Define src_tokens (tokenize source text using tokenize_en)
    src_tokens = __________________________

    # TODO 2: Define src_ids (convert tokenized words to numerical IDs using encode_en)
    src_ids = __________________________

    # TODO 3: Define src_tensor (convert src_ids to torch.LongTensor with batch dimension and move to DEVICE)
    src_tensor = __________________________

    # TODO 4: Define pred_ids, attn_matrix (perform greedy decoding with attention weights using model.greedy_decode)
    pred_ids, attn_matrix = __________________________

    # TODO 5: Define pred_words (convert predicted token IDs to word strings using itos_hi)
    pred_words = __________________________

    # TODO 6: Iterate over the required data (loop through predicted words index range)
    for idx in range(__________________________):
        # TODO 7: Handle the relevant condition (check if token is end-of-sequence token EOS)
        if __________________________:
            break

        # TODO 8: Handle the relevant condition (skip padding or start tokens if necessary)
        if __________________________:
            continue

        # TODO 9: Perform the required operation (append or track valid target word labels)
        __________________________

    # TODO 10: Define src_labels (include source tokens alongside padding or special markers)
    src_labels = __________________________

    # TODO 11: Define matrix_to_plot (slice attention weight matrix up to generated target and source lengths)
    matrix_to_plot = __________________________

    # TODO 12: Define fig, ax (create matplotlib figure and axis objects)
    fig, ax = __________________________

    # TODO 13: Define cax (render heatmap matrix using ax.matshow or plt.imshow with colormap)
    cax = __________________________

    # TODO 14: Perform the required operation (add colorbar corresponding to cax)
    __________________________

    # TODO 15: Perform the required operation (configure x-axis tick positions and source word labels)
    __________________________

    # TODO 16: Perform the required operation (configure y-axis tick positions and target word labels)
    __________________________

    # TODO 17: Perform the required operation (rotate x-axis tick labels for readability)
    __________________________

    # TODO 18: Perform the required operation (set plot title including idx_tag)
    __________________________

    # TODO 19: Perform the required operation (set x-axis label as 'Source (English)')
    __________________________

    # TODO 20: Perform the required operation (set y-axis label as 'Target (Hindi)')
    __________________________

    # TODO 21: Perform the required operation (adjust layout spacing to prevent label clipping)
    __________________________

    # TODO 22: Perform the required operation (save plotted heatmap figure to disk or figures directory)
    __________________________

    # TODO 23: Perform the required operation (display the generated plot using plt.show)
    __________________________

    # TODO 24: Perform the required operation (close figure to free memory)
    __________________________

# TODO 25: Define sample_sentences (list of representative English source sentences selected from test dataset)
sample_sentences = __________________________

# TODO 26: Iterate over the required data (loop through sample sentences with enumerate)
for i, s in enumerate(__________________________):
    # TODO 27: Perform the required operation (print progress or sentence index banner)
    __________________________

    # TODO 28: Perform the required operation (call plot_attention_heatmap using trained attention model and sentence s)
    __________________________

#### Evaluating Autoregressive Decoding Strategies[5 Marks]

The code block below executes and evaluates the decoding utilities defined . It compares qualitative outputs across greedy decoding, beam search, temperature sampling, top-$k$ sampling, and top-$p$ sampling (satisfying A3), and evaluates quantitative beam width BLEU scores alongside sampling repetition statistics (satisfying E3).

Before filling out the code below, familiarize yourself with these core components:

* **Evaluation Suite (`evaluate_decoding_strategies_suite`)**: Simultaneously measures text diversity, repetition rates, and BLEU benchmarks across beam widths and sampling configurations to address Analytical A3 and Experimental E3.
* **Deterministic vs. Stochastic Generation**: Compares deterministic approaches (Greedy decoding, Beam Search) against stochastic sampling methods (Temperature scaling, Top-$k$, and Top-$p$ Nucleus sampling).
* **Metrics Tracking**: Analyzes repetition rates and translation variations on a fixed subset of test samples.

In [ ]:


# --- Decoding Strategies Evaluation Suite ---
@torch.no_grad()
def evaluate_decoding_strategies_suite(model, loader, n_samples=5):
    """Compares Greedy, Beam Search, Temperature, Top-k, and Top-p sampling
       on a fixed subset of test samples to satisfy A3 and E3 requirements.
    """
    # TODO 1: Perform the required operation (set model to evaluation mode)
    __________________________

    # TODO 2: Define src_batch, _, tgt_batch (fetch a single batch of samples from data loader)
    src_batch, _, tgt_batch = __________________________

    # TODO 3: Perform the required operation (move source batch to compute DEVICE)
    __________________________

    # TODO 4: Perform the required operation (print header banner for qualitative comparison)
    __________________________

    # TODO 5: Perform the required operation (select fixed subset of n_samples indices)
    __________________________

    # TODO 6: Define strategies_to_test (dictionary configuring different decoding strategies and parameters)
    strategies_to_test = __________________________

    # TODO 7: Perform the required operation (print table header for strategy outputs)
    __________________________

    # TODO 8: Iterate over the required data (loop through selected sample indices i)
    for i in __________________________:
        # TODO 9: Define single_src (extract single source sequence tensor with batch dimension)
        single_src = __________________________

        # TODO 10: Define ref_text (convert gold target tokens to reference text string using itos_hi)
        ref_text = __________________________

        # TODO 11: Perform the required operation (print sample reference translation string)
        __________________________

        # TODO 12: Perform the required operation (print source input sentence string)
        __________________________

        # TODO 13: Iterate over the required data (loop through strategy name and configuration options)
        for name, config in __________________________:
            # TODO 14: Handle the relevant condition (check if strategy specifies beam search)
            if __________________________:
                # TODO 15: Define pred_ids (generate token IDs using beam_search_decode_attn)
                pred_ids = __________________________
            else:
                # TODO 16: Define pred_ids (generate token IDs using decode_attn_with_strategy with config)
                pred_ids = __________________________

            # TODO 17: Define hyp_text (convert predicted token IDs to hypothesis string using ids_to_string)
            hyp_text = __________________________

            # TODO 18: Perform the required operation (print strategy name and generated hypothesis text)
            __________________________

    # TODO 19: Perform the required operation (print separation banner for quantitative beam search evaluation)
    __________________________

    # TODO 20: Define beam_widths list of values [2, 4, 8]
    __________________________

    # TODO 21: Perform the required operation (print table header for beam width BLEU evaluation)
    __________________________

    # TODO 22: Iterate over the required data (loop through beam widths k)
    for k in __________________________:
        # TODO 23: Define hypotheses, references (initialize empty lists for corpus BLEU evaluation)
        hypotheses, references = __________________________

        # TODO 24: Iterate over the required data (loop through test loader batches)
        for src, _, tgt_out in __________________________:
            # TODO 25: Iterate over the required data (loop through batch elements b)
            for b in range(__________________________):
                # TODO 26: Define single_src (extract single source tensor)
                single_src = __________________________

                # TODO 27: Define pred_ids (generate beam search predictions using beam_width=k)
                pred_ids = __________________________

                # TODO 28: Define hyp_text (convert predicted tokens to hypothesis string)
                hyp_text = __________________________

                # TODO 29: Define ref_text (convert target tokens to reference string)
                ref_text = __________________________

                # TODO 30: Perform the required operation (append hypothesis string)
                __________________________

                # TODO 31: Perform the required operation (append reference string)
                __________________________

        # TODO 32: Define beam_bleu (compute SacreBLEU corpus score across hypotheses and references)
        beam_bleu = __________________________

        # TODO 33: Perform the required operation (print beam width k and corresponding BLEU score)
        __________________________

    # TODO 34: Perform the required operation (print banner for stochastic sampling repetition statistics)
    __________________________

    # TODO 35: Define sample_hyps (initialize empty list to store stochastic samples for repetition scan)
    sample_hyps = __________________________

    # TODO 36: Iterate over the required data (loop through loader batches)
    for src, _, _ in __________________________:
        # TODO 37: Define pred_ids (generate predictions using stochastic top-p sampling)
        pred_ids = __________________________

        # TODO 38: Iterate over the required data (loop through batch elements)
        for b in range(__________________________):
            # TODO 39: Perform the required operation (append generated hypothesis text to sample_hyps)
            __________________________

    # TODO 40: Define rep_indices (scan hypotheses for consecutive word repeats using count_repetitive_hypotheses)
    rep_indices = __________________________

    # TODO 41: Define repetition_rate (compute percentage ratio of repetitive sequences)
    repetition_rate = __________________________

    # TODO 42: Perform the required operation (print calculated sampling sample count and repetition rate)
    __________________________

    # TODO 43: Perform the required operation (print concluding analysis on stochastic diversity vs deterministic beam search)
    __________________________

# TODO 44: Perform the required operation (execute evaluate_decoding_strategies_suite using trained attention model and test loader)
__________________________

####Error Analysis: Does Attention Resolve the Bottleneck?[5 Marks]

To verify if attention mitigates the length bottleneck, we run our short vs. long sentence evaluation again:

Before filling out the code below, familiarize yourself with these core components:

Dataset Subsetting (torch.utils.data.Subset): Extracts a targeted subset of the test dataset using specific index collections (e.g., short vs. long sentences) to isolate and diagnose performance gaps.

Length-Based BLEU Splits: Compares translation quality across different sentence lengths to analyze how sequence length constraints impact encoder-decoder models versus attention-augmented architectures.

Comparative Performance Formatting: Outputs a neat summary table showing short-sentence BLEU, long-sentence BLEU, and the performance degradation gap across architectural variants.

In [ ]:
# --- Error Analysis & Length-Based BLEU Splits ---
def bleu_for_indices_attn(model, subset_indices):
    # TODO 1: Define subset using torch.utils.data.Subset passing test_dataset and subset_indices
    subset = __________________________

    # TODO 2: Define loader using DataLoader passing the subset with batch_size=128
    loader = __________________________

    # TODO 3: Define score by evaluating attention model BLEU score via evaluate_bleu_attn(model, loader)
    score, _, _ = __________________________
    return score

# TODO 4: Print formatted header banner for the length-based BLEU performance comparison table
__________________________

# TODO 5: Print horizontal divider line matching table width
__________________________

# TODO 6: Iterate over model names and model objects in the comparative tuple list
for name, model_obj in __________________________:
    # TODO 7: Handle condition checking if the current model architecture name equals "No-Attention"
    if __________________________:
        # TODO 8: Define s_bleu and l_bleu using precomputed baseline variables bleu_short and bleu_long
        s_bleu, l_bleu = __________________________
    else:
        # TODO 9: Compute short sentence BLEU score for attention models using bleu_for_indices_attn with short_indices
        s_bleu = __________________________

        # TODO 10: Compute long sentence BLEU score for attention models using bleu_for_indices_attn with long_indices
        l_bleu = __________________________

    # TODO 11: Define gap by calculating the performance drop between short and long BLEU scores (s_bleu - l_bleu)
    gap = __________________________

    # TODO 12: Print formatted row showing architecture name, short BLEU, long BLEU, and performance gap
    __________________________

#### **Advanced Analytical & Experimental Evaluation[Total =50 Marks]**

Answer all questions below using evidence, metrics, and observations directly from your attention-augmented Seq2Seq model runs, decoding strategy suites, and entropy/repetition experiments.

**Logical & Reasoning**

* **L1. Syntactic Alignment Discrepancy (SVO vs. SOV):[4 Marks]** English follows a Subject-Verb-Object (SVO) word order, whereas Hindi is predominantly Subject-Object-Verb (SOV). Describe precisely what an alignment-aware attention heatmap should look like near the predicate/verb during generation, and contrast it with a naive monotonic heatmap that blindly increments along source token indices.


-----------write your answer here---------------

* **L2. Attentional Repetition Loops:[4 Marks]** A common failure mode in attention-based Seq2Seq models is degeneration via repetition, where the decoder continuously attends to the exact same source position and generates a repeating loop of words. Using the attention scoring and context vector equations, explain the feedback loop mechanics that cause the model to get "stuck" on a single source index.


-----------write your answer here---------------

* **L3. Beam Search Length Bias & Optimization Limits:[5 Marks]** Why does the raw cumulative log-probability objective in unnormalized beam search naturally favor shorter hypotheses, and how does length penalty formulation correct this bias? Furthermore, explain why increasing the beam width $k$ indefinitely does not mathematically guarantee a superior translation BLEU score.



-----------write your answer here---------------

**Analytical (Requires Your Run's Data & Evaluation Metrics)[5 Marks]**

* **A1. Attention Mitigating the Length Bottleneck:** Report the BLEU score comparison across your no-attention baseline, dot-product attention, and Bahdanau attention models, alongside the short-vs-long sentence performance split. Quantitatively evaluate whether introducing dynamic attention successfully closed the performance gap for long sequences.


In [ ]:
##Code here##

* **A2. Qualitative Heatmap Inspection:[6 Marks]** Include two distinct attention alignment heatmaps from your test runs. For each heatmap, trace which source tokens the decoder focused on while generating specific Hindi words and evaluate whether the resulting soft alignment corresponds to meaningful cross-lingual syntax.


In [ ]:
##Code here##

* **A3. Strategy Comparative Analysis:[6 Marks]** On a fixed subset of at least five test sources, compare the translation outputs generated via greedy decoding, beam search, temperature sampling, top-$k$ sampling, and top-$p$ nucleus sampling using a fixed random seed. Evaluate them based on semantic adequacy, token repetition rates, and output text diversity.



In [ ]:
##Code here##

**Experimental (Requires Code Execution & Observations)**

* **E1. Attention Entropy Measurement:[6 Marks]** Compute the Shannon entropy of the attention distribution at every decoding step $t$ across your 5 heatmap sentences for both dot-product and Bahdanau models:
  $$-\sum_j \alpha_{t,j}\log\alpha_{t,j}$$
  Average the entropy values across steps and sentences. Which attention architecture produces sharper, more concentrated (lower-entropy) alignment distributions?


In [ ]:
##Code here##

* **E2. Repetition Failure Tracking & Heatmap Diagnosis:[7 Marks]** Scan your test predictions for instances where the exact same target word is repeated 3 or more times consecutively. Report the frequency of this occurrence across both attention models, and extract the attention heatmap of one failed repetitive example to diagnose how the attention weights behaved during the loop.


In [ ]:
##Code here##

* **E3. Systematic Decoding Suite Benchmark:[7 Marks]** Execute decoding suite on your test subset using beam widths $k \in \{2, 4, 8\}$ alongside temperature, top-$k$, and top-$p$ sampling. Report deterministic BLEU scores across beam widths, and for stochastic sampling, report the seed, repetition/diversity metrics, and representative qualitative examples.

In [ ]:
##Code here##

# **Question 4 - Implement RNN, LSTM, GRU using pytorch functions and repeat 2 and 3 above and compare the your implementation with pytorch implementation.[Total = 160 Marks]**  


Here is a step-by-step breakdown of what you need to do:

Tasks to Complete
Implement Native Encoders: Build NativeRNSEncoder, NativeLSTMEncoder, and NativeGRUEncoder utilizing nn.RNN, nn.LSTM, and nn.GRU.

Implement Step-Wise Decoders: Build plain and attention-augmented decoders using nn.RNNCell, nn.LSTMCell, and nn.GRUCell.

Train & Save Artifacts: Train the native GRU variants (No-Attention, Dot-Product Attention, Bahdanau Attention) and save their weights as q4_native_noattn.pt, q4_native_dot.pt, and q4_native_bah.pt.

Execute Comparative Evaluation: Run the comparative BLEU evaluation suite across scratch vs. native implementations on the test set.

In Questions 1 through 3, we built every single recurrent cell (`RNNCell`, `LSTMCell`, `GRUCell`), unrolling loop, encoder, and decoder using PyTorch tensor operations.

Building models from scratch is the best way to understand the underlying mathematical equations. However, in production environments, custom Python `for` loops over sequence timesteps create noticeable overhead:

```python
# Our Custom Loop (Question 1 - 3): Python overhead at EVERY timestep
for t in range(seq_len):
    h = cell(x[:, t, :], h)  # Triggers a separate CUDA kernel launch per step!

```

In this section, we replace our custom cells and sequence wrappers with **PyTorch’s built-in layers** (`nn.GRU` and `nn.GRUCell`). We will benchmark them against our custom implementations to measure speedups, memory efficiency, and translation performance.


Why PyTorch Native Layers Are Faster

PyTorch built-in layers like `nn.GRU` and `nn.LSTM` bypass the Python interpreter's loop overhead by utilizing **NVIDIA cuDNN C++ kernels**:

1. **Kernel Fusion:** Multiple matrix multiplications across timesteps are fused into a single optimized C++ GPU call.
2. **Packed Weights:** Weight matrices for all gates are stored in contiguous memory blocks rather than as separate `nn.Linear` objects.
3. **Reduced Memory Transfers:** Intermediate activation states remain in fast GPU cache memory rather than being constantly allocated and freed in Python memory.

```
Custom Loop:   [Python Loop] --> [Launch GPU Kernel t=1] --> [Launch GPU Kernel t=2] --> ...
Native nn.GRU: [Python Call] --> [Single Optimized C++/cuDNN GPU Execution]

```

###Model Implementation[80 marks]

Before filling out the code below, familiarize yourself with these core components:

Custom Bidirectional Encoder (CustomEncoder): Integrates token embeddings, a multi-layer bidirectional custom recurrent encoder (RecurrentEncoder), and a linear bridge layer to project concatenated forward-backward states down to the decoder's hidden dimension.

LSTM Tuple State Handling (is_lstm): Manages the structural difference between single hidden states (h for RNN/GRU) and dual state tuples ((h, c) for LSTM) across encoder and decoder handoffs.

In [ ]:
# ==============================================================================
# Custom Encoder Implementation with LSTM Tuple Handling
# ==============================================================================

class CustomEncoder(nn.Module):
    """2-Layer Bidirectional Custom Encoder using custom recurrent cells."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, cell_cls: type, is_lstm: bool = False, num_layers: int = 2):
        super().__init__()
        # TODO 1: Initialize the embedding layer with vocab_size, emb_dim, and padding_idx=0
        self.embedding = __________________________

        # TODO 2: Initialize the bidirectional RecurrentEncoder passing cell_cls, emb_dim, hidden_size, num_layers, bidirectional=True, and is_lstm
        self.rnn = __________________________

        # TODO 3: Define a linear bridge layer mapping 2 * hidden_size to hidden_size
        self.bridge = __________________________

        # TODO 4: Store the is_lstm flag as an attribute to handle conditional returns
        self.is_lstm = __________________________

    def forward(self, src: torch.Tensor):
        # TODO 5: Pass source tokens through the embedding layer
        emb = __________________________

        # TODO 6: Run the recurrent encoder to obtain sequence outputs and final hidden states tensor
        outputs, final_h_tensor = __________________________

        # TODO 7: Apply the linear bridge layer followed by torch.tanh to compute the decoder hidden state h0
        dec_h0 = __________________________

        # TODO 8: Return either an LSTM tuple state (h, c) or a single tensor h based on self.is_lstm
        if self.is_lstm:
            # TODO 9: Initialize zero cell state dec_c0 with the same shape as dec_h0
            dec_c0 = __________________________

            # TODO 10: Return outputs and the tuple state (dec_h0, dec_c0)
            return __________________________
        else:
            # TODO 11: Return outputs and dec_h0 directly
            return __________________________

Before filling out the code below, familiarize yourself with these core components:

Step-Wise Unidirectional Decoder (CustomDecoderStep): Advances autoregressive generation one token at a time using custom recurrent cells (RNNCell, LSTMCell, or GRUCell) across stacked layers.

ModuleList Architecture Routing: Dynamically configures input dimensions for layer zero versus upper stacked layers to support arbitrary hidden depths.

In [ ]:
# ==============================================================================
# Custom Decoder Step Implementation with Cell Type Routing
# ==============================================================================


class CustomDecoderStep(nn.Module):
    """Step-wise 2-Layer Unidirectional Decoder using custom recurrent cells (RNNCell / LSTMCell / GRUCell)."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, cell_cls: type, is_lstm: bool = False, num_layers: int = 2):
        super().__init__()
        # TODO 1: Store configuration parameters as attributes (vocab_size, emb_dim, hidden_size, num_layers, is_lstm)
        self.vocab_size = __________________________
        self.emb_dim = __________________________
        self.hidden_size = __________________________
        self.num_layers = __________________________
        self.is_lstm = __________________________

        # TODO 2: Initialize target embedding layer with vocab_size, emb_dim, and padding_idx=0
        self.embedding = __________________________

        # TODO 3: Create a ModuleList of stacked custom recurrent cells (cell_cls), adjusting input size for layer 0 vs upper layers
        self.cells = nn.ModuleList([
            __________________________
            for l in range(num_layers)
        ])

        # TODO 4: Define the final linear output projection layer mapping hidden_size to vocab_size
        self.out = __________________________

    def forward(self, tok: torch.Tensor, states: list):
        """
        Advances the decoder by exactly one timestep t.
        Args:
            tok (torch.Tensor): Current token IDs [batch_size]
            states (list): List of hidden states (or (h, c) tuples if LSTM) for each layer
        """
        # TODO 5: Embed the current token IDs to get vector representations of shape [batch_size, emb_dim]
        x = __________________________

        # TODO 6: Initialize an empty list to track updated states for the next decoding step
        new_states = __________________________

        # TODO 7: Set initial input tensor for layer 0 to the embedded token vector x
        inp = __________________________

        # TODO 8: Iterate through the stacked custom cells using enumerate
        for l, cell in enumerate(self.cells):
            # TODO 9: Handle LSTM state tuples vs Vanilla RNN/GRU single hidden states using self.is_lstm
            if self.is_lstm:
                # Unpack previous (h, c) tuple state for layer l
                h_prev, c_prev = __________________________

                # Pass input and tuple state through the LSTM cell
                h_next, c_next = __________________________

                # Append updated tuple to new_states and update inp for the next layer
                new_states.append(__________________________)
                inp = h_next
            else:
                # Extract previous hidden state h for non-LSTM layer l
                h_prev = __________________________

                # Pass input and hidden state through the RNN/GRU cell
                h_next = __________________________

                # Append updated hidden state to new_states and update inp for the next layer
                new_states.append(__________________________)
                inp = h_next

        # TODO 10: Project the top layer's final output tensor through self.out to generate vocabulary logits
        logits = __________________________

        return logits, new_states

Before filling out the code below, familiarize yourself with these core components:

Native PyTorch RNN Encoder (NativeRNSEncoder): Leverages PyTorch's optimized nn.RNN module in bidirectional mode to process input sequences.

Hidden State Reshaping & Bridging: Extracts and concatenates the forward and backward hidden states of the top layer, projecting them via a linear layer + tanh to initialize the unidirectional decoder.

In [ ]:
# --- 1. Native Encoders using nn.RNN, nn.LSTM, and nn.GRU ---

class NativeRNSEncoder(nn.Module):
    """2-Layer Bidirectional RNN Encoder using PyTorch's built-in nn.RNN."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.hidden_size
        self.hidden_size = __________________________

        # TODO 3: Define self.embedding layer (nn.Embedding)
        self.embedding = __________________________

        # TODO 4: Define self.rnn module (nn.RNN with bidirectional=True, batch_first=True, and num_layers)
        self.rnn = __________________________

        # TODO 5: Define self.bridge layer (nn.Linear mapping 2 * hidden_size to hidden_size followed by tanh)
        self.bridge = __________________________

    def forward(self, src: torch.Tensor):
        """
        Args:
            src (torch.Tensor): English token IDs [batch_size, src_len]
        Returns:
            outputs (torch.Tensor): Per-step hidden states [batch_size, src_len, hidden_size * 2]
            dec_h0 (torch.Tensor): Initialized decoder hidden state [batch_size, hidden_size]
        """
        # 1. Embed tokens: [batch_size, src_len, emb_dim]
        # TODO 6: Define emb
        emb = __________________________

        # 2. Run cuDNN-accelerated RNN
        # TODO 7: Define outputs, h_n
        outputs, h_n = __________________________

        # 3. Reshape h_n to isolate the top layer's forward and backward hidden states
        # Shape of h_n: [num_layers * 2, batch_size, hidden_size]
        # TODO 8: Define h_n tensor slices for top layers
        h_n = __________________________

        # TODO 9: Define top_fwd
        top_fwd = __________________________

        # TODO 10: Define top_bwd
        top_bwd = __________________________

        # TODO 11: Define top_layer_combined (concatenation of top forward and backward states)
        top_layer_combined = __________________________

        # 4. Project down for unidirectional decoder initialization via bridge layer + tanh
        # TODO 12: Define dec_h0
        dec_h0 = __________________________

        return (outputs, dec_h0)

Before filling out the code below, familiarize yourself with these core components:

Native Bidirectional LSTM Encoder (NativeLSTMEncoder): Utilizes nn.LSTM to process sequences and return both hidden states (h_n) and cell states (c_n).

Dual State Bridging (bridge_h, bridge_c): Employs separate linear bridge projections to map both top-layer hidden states and cell states down to the decoder dimensions.

In [ ]:
class NativeLSTMEncoder(nn.Module):
    """2-Layer Bidirectional LSTM Encoder using PyTorch's built-in nn.LSTM."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.hidden_size
        self.hidden_size = __________________________

        # TODO 3: Define self.embedding layer (nn.Embedding)
        self.embedding = __________________________

        # TODO 4: Define self.lstm module (nn.LSTM with bidirectional=True, batch_first=True, and num_layers)
        self.lstm = __________________________

        # TODO 5: Define self.bridge_h layer for hidden state initialization
        self.bridge_h = __________________________

        # TODO 6: Define self.bridge_c layer for cell state initialization
        self.bridge_c = __________________________

    def forward(self, src: torch.Tensor):
        """
        Args:
            src (torch.Tensor): English token IDs [batch_size, src_len]
        Returns:
            outputs (torch.Tensor): Per-step hidden states [batch_size, src_len, hidden_size * 2]
            tuple: (dec_h0, dec_c0) Initialized decoder states each of shape [batch_size, hidden_size]
        """
        # 1. Embed tokens: [batch_size, src_len, emb_dim]
        # TODO 7: Define emb
        emb = __________________________

        # 2. Run cuDNN-accelerated LSTM
        # TODO 8: Define outputs, (h_n, c_n)
        outputs, (h_n, c_n) = __________________________

        # 3. Isolate top layer's forward and backward states for h and c
        # TODO 9: Define top_fwd_h, top_bwd_h
        top_fwd_h, top_bwd_h = __________________________

        # TODO 10: Define top_fwd_c, top_bwd_c
        top_fwd_c, top_bwd_c = __________________________

        # 4. Project down for unidirectional decoder initialization via bridge layers + tanh
        # TODO 11: Define dec_h0
        dec_h0 = __________________________

        # TODO 12: Define dec_c0
        dec_c0 = __________________________

        return (outputs, (dec_h0, dec_c0))

Before filling out the code below, familiarize yourself with these core components:

Native Bidirectional GRU Encoder (NativeGRUEncoder): Encapsulates sequence feature extraction using PyTorch's high-performance nn.GRU module.

Single-State GRU Bridging: Combines bidirectional outputs into a unified vector before applying a linear bridge and activation function to seed the decoder hidden state.

In [ ]:
class NativeGRUEncoder(nn.Module):
    """2-Layer Bidirectional GRU Encoder using PyTorch's built-in nn.GRU."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.hidden_size
        self.hidden_size = __________________________

        # TODO 3: Define self.embedding layer (nn.Embedding)
        self.embedding = __________________________

        # TODO 4: Define self.gru module (nn.GRU with bidirectional=True, batch_first=True, and num_layers)
        self.gru = __________________________

        # TODO 5: Define self.bridge layer (nn.Linear mapping 2 * hidden_size to hidden_size followed by tanh)
        self.bridge = __________________________

    def forward(self, src: torch.Tensor):
        """
        Args:
            src (torch.Tensor): English token IDs [batch_size, src_len]
        Returns:
            outputs (torch.Tensor): Per-step hidden states [batch_size, src_len, hidden_size * 2]
            dec_h0 (torch.Tensor): Initialized decoder hidden state [batch_size, hidden_size]
        """
        # 1. Embed tokens: [batch_size, src_len, emb_dim]
        # TODO 6: Define emb
        emb = __________________________

        # 2. Run cuDNN-accelerated GRU
        # TODO 7: Define outputs, h_n
        outputs, h_n = __________________________

        # 3. Isolate top layer's forward and backward states
        # TODO 8: Define top_layer_combined
        top_layer_combined = __________________________

        # 4. Project down for unidirectional decoder initialization via bridge layer + tanh
        # TODO 9: Define dec_h0
        dec_h0 = __________________________

        return (outputs, dec_h0)

Before filling out the code below, familiarize yourself with these core components:

Native Step-Wise RNN Decoder (NativeRNNDecoderStep): Implements unrolled single-step decoding using PyTorch's built-in nn.RNNCell modules organized within an nn.ModuleList.

Linear Vocabulary Projection: Maps the hidden representation of the final recurrent layer into vocabulary-sized logits via a linear output layer (self.out).

In [ ]:
# --- 2. Native Step-Wise Decoders using nn.RNNCell, nn.LSTMCell, and nn.GRUCell ---

class NativeRNNDecoderStep(nn.Module):
    """Step-wise plain decoder using built-in nn.RNNCell."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.embedding layer for target tokens
        self.embedding = __________________________

        # TODO 3: Define self.cells using nn.ModuleList containing num_layers of nn.RNNCell
        self.cells = __________________________

        # TODO 4: Define self.out linear projection layer mapping hidden_size to vocab_size
        self.out = __________________________

    def forward(self, tok: torch.Tensor, hidden_states: list):
        # TODO 5: Define x by embedding input tokens [batch_size, emb_dim]
        x = __________________________

        # TODO 6: Define new_states list to track updated hidden states
        new_states = __________________________

        # TODO 7: Define inp starting with embedded token x
        inp = __________________________

        # TODO 8: Iterate over the required data (enumerate through self.cells)
        for l, cell in enumerate(__________________________):
            # TODO 9: Define h (extract previous hidden state for layer l)
            h = __________________________

            # TODO 10: Perform the required operation (pass input and hidden state through cell)
            __________________________

            # TODO 11: Define inp for the next layer
            inp = __________________________

        # TODO 12: Define logits by projecting final hidden state through self.out
        logits = __________________________
        return (logits, new_states)

Before filling out the code below, familiarize yourself with these core components:

Native Step-Wise LSTM Decoder (NativeLSTMDecoderStep): Coordinates step-by-step target token generation using built-in nn.LSTMCell cells.

Tuple State Propagation: Safely handles and updates the dual-state (h, c) tuple representation across all stacked decoding layers.

In [ ]:
class NativeLSTMDecoderStep(nn.Module):
    """Step-wise plain decoder using built-in nn.LSTMCell."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.embedding layer for target tokens
        self.embedding = __________________________

        # TODO 3: Define self.cells using nn.ModuleList containing num_layers of nn.LSTMCell
        self.cells = __________________________

        # TODO 4: Define self.out linear projection layer mapping hidden_size to vocab_size
        self.out = __________________________

    def forward(self, tok: torch.Tensor, states: list):
        # TODO 5: Define x by embedding input tokens [batch_size, emb_dim]
        x = __________________________

        # TODO 6: Define new_states list to track updated tuple states (h, c)
        new_states = __________________________

        # TODO 7: Define inp starting with embedded token x
        inp = __________________________

        # TODO 8: Iterate over the required data (enumerate through self.cells)
        for l, cell in enumerate(__________________________):
            # TODO 9: Define h, c (extract previous state tuple for layer l)
            h, c = __________________________

            # TODO 10: Perform the required operation (pass input and state tuple through cell)
            __________________________

            # TODO 11: Define inp for the next layer
            inp = __________________________

        # TODO 12: Define logits by projecting final hidden state through self.out
        logits = __________________________
        return (logits, new_states)

Before filling out the code below, familiarize yourself with these core components:

Native Step-Wise GRU Decoder (NativeGRUDecoderStep): Operates step-by-step generation loops utilizing PyTorch's native nn.GRUCell recurrent units.

Hidden State Continuity: Chains hidden state outputs sequentially from layer to layer during each decoding timestep.

In [ ]:
class NativeGRUDecoderStep(nn.Module):
    """Step-wise plain decoder using built-in nn.GRUCell."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.embedding layer for target tokens
        self.embedding = __________________________

        # TODO 3: Define self.cells using nn.ModuleList containing num_layers of nn.GRUCell
        self.cells = __________________________

        # TODO 4: Define self.out linear projection layer mapping hidden_size to vocab_size
        self.out = __________________________

    def forward(self, tok: torch.Tensor, hidden_states: list):
        # TODO 5: Define x by embedding input tokens [batch_size, emb_dim]
        x = __________________________

        # TODO 6: Define new_states list to track updated hidden states
        new_states = __________________________

        # TODO 7: Define inp starting with embedded token x
        inp = __________________________

        # TODO 8: Iterate over the required data (enumerate through self.cells)
        for l, cell in enumerate(__________________________):
            # TODO 9: Define h (extract previous hidden state for layer l)
            h = __________________________

            # TODO 10: Perform the required operation (pass input and hidden state through cell)
            __________________________

            # TODO 11: Define inp for the next layer
            inp = __________________________

        # TODO 12: Define logits by projecting final hidden state through self.out
        logits = __________________________
        return (logits, new_states)

Before filling out the code below, familiarize yourself with these core components:

Attention-Augmented RNN Decoder (NativeAttnRNNDecoderStep): Combines target token embeddings with dynamic context vectors computed via attention modules over encoder outputs.

Concatenated Feature Input (emb_dim + enc_hidden): Scales the input dimension of the first recurrent layer to accommodate both the embedded token and the retrieved encoder context.

In [ ]:
# --- 3. Attention-Augmented Native Decoders ---

class NativeAttnRNNDecoderStep(nn.Module):
    """Step-wise attention decoder using nn.RNNCell."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, enc_hidden: int, attention_module: nn.Module, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.embedding layer
        self.embedding = __________________________

        # TODO 3: Define self.attention module
        self.attention = __________________________

        # TODO 4: Define self.cells using nn.ModuleList of nn.RNNCell (first cell input dim is emb_dim + enc_hidden)
        self.cells = __________________________

        # TODO 5: Define self.out linear projection mapping (hidden_size + enc_hidden) to vocab_size
        self.out = __________________________

    def forward(self, tok: torch.Tensor, hidden_states: list, enc_outputs: torch.Tensor, mask: torch.Tensor = None):
        # TODO 6: Define x by embedding input tokens
        x = __________________________

        # TODO 7: Define context, weights by calling self.attention using top layer's previous hidden state
        context, weights = __________________________

        # TODO 8: Define inp by concatenating embedding x and context vector along dim=1
        inp = __________________________

        # TODO 9: Define new_states list
        new_states = __________________________

        # TODO 10: Iterate over the required data (enumerate through self.cells)
        for l, cell in enumerate(__________________________):
            # TODO 11: Define h
            h = __________________________

            # TODO 12: Perform the required operation (pass inp and h through cell)
            __________________________

            # TODO 13: Define inp for next layer
            inp = __________________________

        # TODO 14: Define logits by concatenating top layer hidden state and context vector, then passing through self.out
        logits = __________________________
        return (logits, new_states, weights)

Before filling out the code below, familiarize yourself with these core components:

Attention-Augmented LSTM Decoder (NativeAttnLSTMDecoderStep): Integrates explicit context vector injection with step-wise LSTM cell updates.

Top-Layer State Extraction (states[-1][0]): Queries the attention mechanism using the hidden state component of the topmost LSTM layer tuple.

In [ ]:
class NativeAttnLSTMDecoderStep(nn.Module):
    """Step-wise attention decoder using nn.LSTMCell."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, enc_hidden: int, attention_module: nn.Module, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.embedding layer
        self.embedding = __________________________

        # TODO 3: Define self.attention module
        self.attention = __________________________

        # TODO 4: Define self.cells using nn.ModuleList of nn.LSTMCell (first cell input dim is emb_dim + enc_hidden)
        self.cells = __________________________

        # TODO 5: Define self.out linear projection mapping (hidden_size + enc_hidden) to vocab_size
        self.out = __________________________

    def forward(self, tok: torch.Tensor, states: list, enc_outputs: torch.Tensor, mask: torch.Tensor = None):
        # TODO 6: Define x by embedding input tokens
        x = __________________________

        # TODO 7: Define context, weights using top layer's hidden state (states[-1][0])
        context, weights = __________________________

        # TODO 8: Define inp by concatenating embedding x and context vector along dim=1
        inp = __________________________

        # TODO 9: Define new_states list
        new_states = __________________________

        # TODO 10: Iterate over the required data (enumerate through self.cells)
        for l, cell in enumerate(__________________________):
            # TODO 11: Define h, c
            h, c = __________________________

            # TODO 12: Perform the required operation (pass inp and state tuple through cell)
            __________________________

            # TODO 13: Define inp for next layer
            inp = __________________________

        # TODO 14: Define logits by concatenating top layer hidden state and context vector, then passing through self.out
        logits = __________________________
        return (logits, new_states, weights)

Before filling out the code below, familiarize yourself with these core components:

Attention-Augmented GRU Decoder (NativeAttnGRUDecoderStep): Merges gated recurrent units with context vectors generated via dot-product or Bahdanau alignment mechanisms.

Contextual Logit Fusion: Concatenates the final GRU hidden state with the attention context vector before final vocabulary projection.

In [ ]:
class NativeAttnGRUDecoderStep(nn.Module):
    """Step-wise attention decoder using nn.GRUCell."""
    def __init__(self, vocab_size: int, emb_dim: int, hidden_size: int, enc_hidden: int, attention_module: nn.Module, num_layers: int = 2):
        super().__init__()
        # TODO 1: Define self.num_layers
        self.num_layers = __________________________

        # TODO 2: Define self.embedding layer
        self.embedding = __________________________

        # TODO 3: Define self.attention module
        self.attention = __________________________

        # TODO 4: Define self.cells using nn.ModuleList of nn.GRUCell (first cell input dim is emb_dim + enc_hidden)
        self.cells = __________________________

        # TODO 5: Define self.out linear projection mapping (hidden_size + enc_hidden) to vocab_size
        self.out = __________________________

    def forward(self, tok: torch.Tensor, hidden_states: list, enc_outputs: torch.Tensor, mask: torch.Tensor = None):
        # TODO 6: Define x by embedding input tokens
        x = __________________________

        # TODO 7: Define context, weights by calling self.attention using top layer's previous hidden state
        context, weights = __________________________

        # TODO 8: Define inp by concatenating embedding x and context vector along dim=1
        inp = __________________________

        # TODO 9: Define new_states list
        new_states = __________________________

        # TODO 10: Iterate over the required data (enumerate through self.cells)
        for l, cell in enumerate(__________________________):
            # TODO 11: Define h
            h = __________________________

            # TODO 12: Perform the required operation (pass inp and h through cell)
            __________________________

            # TODO 13: Define inp for next layer
            inp = __________________________

        # TODO 14: Define logits by concatenating top layer hidden state and context vector, then passing through self.out
        logits = __________________________
        return (logits, new_states, weights)

###Training for Translation Models[20 Marks]

Before filling out the code below, familiarize yourself with these core components:

End-to-End Translation Pipelines: Instantiates and trains custom and native Seq2Seq architectures across Vanilla RNN, LSTM, and GRU configurations.

Attention Integration & Checkpointing: Configures attention modules (Dot-Product and Bahdanau) with native GRU encoders and saves model checkpoints (.pt) upon completion.

In [ ]:
# ==============================================================================
# Question 4: Training for Translation Models
# ==============================================================================

# 1. Train Scratch Vanilla RNN (No-Attention)
print("\nTraining Scratch Vanilla RNN (No-Attention)...")
# TODO 1: Initialize CustomEncoder for Vanilla RNN using len(itos_en), EMB_MT, HID_MT, cell_cls=RNNCell, is_lstm=False, and num_layers=2
rnn_enc = __________________________

# TODO 2: Initialize CustomDecoderStep for Vanilla RNN using len(itos_hi), EMB_MT, HID_MT, cell_cls=RNNCell, is_lstm=False, and num_layers=2
rnn_dec = __________________________

# TODO 3: Wrap encoder and decoder into Seq2Seq specifying num_layers=2
seq2seq_rnn_noattn = __________________________

# TODO 4: Train the scratch RNN Seq2Seq model using train_seq2seq for 4 epochs with tag "Seq2Seq-RNN-noattn"
__________________________


# 2. Train Scratch LSTM (No-Attention)
print("\nTraining Scratch LSTM (No-Attention)...")
# TODO 5: Initialize CustomEncoder for LSTM using len(itos_en), EMB_MT, HID_MT, cell_cls=LSTMCell, is_lstm=True, and num_layers=2
lstm_enc = __________________________

# TODO 6: Initialize CustomDecoderStep for LSTM using len(itos_hi), EMB_MT, HID_MT, cell_cls=LSTMCell, is_lstm=True, and num_layers=2
lstm_dec = __________________________

# TODO 7: Wrap LSTM encoder and decoder into Seq2Seq specifying num_layers=2
seq2seq_lstm_noattn = __________________________

# TODO 8: Train the scratch LSTM Seq2Seq model using train_seq2seq for 4 epochs with tag "Seq2Seq-LSTM-noattn"
__________________________

print("\nScratch RNN and LSTM Models Trained Successfully!")
# Train Native RNN (No-Attention)
print("\nTraining Native RNN (No-Attention)...")
# TODO 1: Initialize the native RNN encoder using NativeRNSEncoder with vocab_size, emb_dim, hidden_size, and num_layers
native_rnn_enc = __________________________
# TODO 2: Initialize the native RNN decoder step using NativeRNNDecoderStep
native_rnn_dec = __________________________
# TODO 3: Wrap encoder and decoder into NativeSeq2Seq specifying is_lstm=False
native_seq2seq_rnn_noattn = __________________________
# TODO 4: Train the native RNN model using train_native_seq2seq for  epochs
train_native_seq2seq(native_seq2seq_rnn_noattn, tag="NativeSeq2Seq-RNN-noattn", epochs=--)

# Train Native LSTM (No-Attention)
print("\nTraining Native LSTM (No-Attention)...")
# TODO 5: Initialize the native LSTM encoder using NativeLSTMEncoder
native_lstm_enc = __________________________
# TODO 6: Initialize the native LSTM decoder step using NativeLSTMDecoderStep
native_lstm_dec = __________________________
# TODO 7: Wrap encoder and decoder into NativeSeq2Seq specifying is_lstm=True
native_seq2seq_lstm_noattn = __________________________
# TODO 8: Train the native LSTM model using train_native_seq2seq for  epochs
train_native_seq2seq(native_seq2seq_lstm_noattn, tag="NativeSeq2Seq-LSTM-noattn", epochs=--)


# 1. Train Native GRU (No-Attention)
print("\nTraining Native GRU (No-Attention)...")
# TODO 9: Initialize the native GRU encoder for the no-attention pipeline
native_gru_encoder_noattn = __________________________
# TODO 10: Initialize the native GRU decoder step for the no-attention pipeline
native_gru_decoder_step_noattn = __________________________
# TODO 11: Wrap encoder and decoder into NativeSeq2Seq
native_seq2seq_noattn = __________________________
# TODO 12: Train the native GRU no-attention model for  epochs and save checkpoint to 'q4_native_noattn.pt'
hist_native_noattn = __________________________

# 2. Train Native GRU with Dot-Product Attention
print("\nTraining Native GRU (Dot-Product Attention)...")
# TODO 13: Initialize the native GRU encoder for dot-product attention
native_gru_encoder_dot = __________________________
# TODO 14: Instantiate DotProductAttention matching decoder and encoder dimensions (enc_hidden = HID_MT * 2)
native_dot_attention = __________________________
# TODO 15: Initialize NativeAttnGRUDecoderStep incorporating the dot-product attention module
native_attn_gru_decoder_step_dot = __________________________
# TODO 16: Wrap encoder and attention decoder into NativeSeq2SeqAttn
native_seq2seq_dot = __________________________
# TODO 17: Train the native dot-attention model for epochs with is_attention=True and checkpoint 'q4_native_dot.pt'
hist_native_dot = __________________________

# 3. Train Native GRU with Bahdanau Attention
print("\nTraining Native GRU (Bahdanau Attention)...")
# TODO 18: Initialize the native GRU encoder for Bahdanau attention
native_gru_encoder_bah = __________________________
# TODO 19: Instantiate BahdanauAttention with dec_hidden, enc_hidden, and attn_dim
native_bahdanau_attention = __________________________
# TODO 20: Initialize NativeAttnGRUDecoderStep incorporating the Bahdanau attention module
native_attn_gru_decoder_step_bah = __________________________
# TODO 21: Wrap encoder and Bahdanau attention decoder into NativeSeq2SeqAttn
native_seq2seq_bah = __________________________
# TODO 22: Train the native Bahdanau attention model for  epochs with is_attention=True and checkpoint 'q4_native_bah.pt'
hist_native_bah = __________________________

print("\nAll Native Models Trained Successfully!")

###Comprehensive Comparative Study[20 Marks]

**Do not change content of this cell**

In [ ]:
# ==============================================================================
# Question 4: Comprehensive Comparative Study (Scratch vs. PyTorch Native)
# Evaluation and BLEU Score Comparison across RNN, LSTM, and GRU (With & Without Attention)
# ==============================================================================



@torch.no_grad()
def evaluate_model_bleu(model, loader, is_attention=False):
    """
    Evaluates SacreBLEU score on the test dataset for any Seq2Seq variant.
    Determines automatically whether the model uses attention decoding or standard greedy decoding.
    """
    model.eval()
    hypotheses = []
    references = []

    for src, _, tgt_out in loader:
        src = src.to(DEVICE)
        for b in range(src.size(0)):
            single_src = src[b:b+1]

            # Generate predictions based on architecture capability
            if is_attention and hasattr(model, "greedy_decode"):
                pred_ids, _ = model.greedy_decode(single_src)
            else:
                pred_ids = greedy_decode(model, single_src)

            hyp_text = ids_to_string(pred_ids[0].tolist(), itos_hi)
            ref_text = ids_to_string(tgt_out[b].tolist(), itos_hi)

            hypotheses.append(hyp_text)
            references.append(ref_text)

    bleu_score = sacrebleu.corpus_bleu(hypotheses, [references]).score
    return bleu_score

# --- Comprehensive Execution & Comparative BLEU Evaluation Suite ---
print("=" * 70)
print("QUESTION 4: COMPARATIVE BLEU EVALUATION (SCRATCH VS. PYTORCH NATIVE)")
print("=" * 70)
print(f"{'Architecture / Variant':<35} | {'Implementation':<15} | {'Test BLEU Score'}")
print("-" * 70)

# Full comparison suite covering RNN, LSTM, and GRU variants across custom and native implementations
comparison_suite = [
    ("Vanilla RNN (No-Attn)", "Scratch / Custom", globals().get('seq2seq_rnn_noattn', None)),
    ("LSTM (No-Attn)", "Scratch / Custom", globals().get('seq2seq_lstm_noattn', None)),
    ("GRU (No-Attn)", "Scratch / Custom", globals().get('seq2seq_noattn', None)),
    ("GRU + Dot Attention", "Scratch / Custom", globals().get('seq2seq_dot', None)),
    ("GRU + Bahdanau Attention", "Scratch / Custom", globals().get('seq2seq_bah', None)),
    ("RNN (No-Attn)", "PyTorch Native", globals().get('native_seq2seq_rnn_noattn', None)),
    ("LSTM (No-Attn)", "PyTorch Native", globals().get('native_seq2seq_lstm_noattn', None)),
    ("GRU (No-Attn)", "PyTorch Native", globals().get('native_seq2seq_noattn', None)),
    ("GRU + Dot Attention", "PyTorch Native", globals().get('native_seq2seq_dot', None)),
    ("GRU + Bahdanau Attention", "PyTorch Native", globals().get('native_seq2seq_bah', None)),
]

for model_name, impl_type, model_obj in comparison_suite:
    if model_obj is not None:
        uses_attn = "Attention" in model_name or "Dot" in model_name or "Bahdanau" in model_name
        try:
            score = evaluate_model_bleu(model_obj, test_loader, is_attention=uses_attn)
            print(f"{model_name:<35} | {impl_type:<15} | {score:.2f}")
        except Exception as e:
            print(f"{model_name:<35} | {impl_type:<15} | [Evaluation Error]")
    else:
        print(f"{model_name:<35} | {impl_type:<15} | [Model Not Trained / Loaded]")

print("-" * 70)

### **Advanced Analytical & Experimental Evaluation[Total =40 Marks]**

Answer all questions below using evidence, metrics, and observations directly from your experiments.

**Logical & Reasoning**

* **L1. Autoregressive Constraints & Sequence Modules:[5 marks]** If a decoder uses 100% teacher forcing and never consumes its own past predictions during training, why can it *still* not use a whole-sequence `nn.GRU` instead of a step-wise `nn.GRUCell`? Explain how the autoregressive dependency of attention contexts and hidden states enforces step-by-step execution.


-----------write your answer here---------------

* **L2. Discrepancies Under Fixed Seeds[5 Marks]:** If your native PyTorch implementation and custom hand-written model produce diverging validation losses or BLEU scores despite identical hyperparameters and random seeds, outline two potential underlying reasons (e.g., initialization or padding-mask handling differences) and explain how you would isolate and test each.

-----------write your answer here---------------

**Analytical (Requires Your Run's Data & Evaluation Metrics)**

* **A1. Training Throughput & Speedup Quantification:[5 Marks]** Report the runtime benchmarks. Compute the exact training speedup factor of the native PyTorch implementation over your custom Python loop, and discuss whether the performance differential aligns with C++ backend vectorization expectations.

In [ ]:
##code here##

* **A2. Translation Quality Parity:[5 marks]** Compare the final test BLEU scores between the native and custom model variants. Did leveraging PyTorch's built-in layers alter translation accuracy, or did performance remain equivalent?

In [ ]:
##Code Here##

**Experimental (Requires Code Execution & Observations)**

* **E1. Inference Latency Profiling:[10 marks]** Measure and compare the `greedy_decode` inference latency for your custom and native Bahdanau models under two distinct workloads:
  - Batch size 1 (sequential inference)
  - Batch size 128 (vectorized batch inference)
  Evaluate whether the native model retains its massive speedup advantage during single-sample batch execution.

In [ ]:
##Code Here##

* **E2. Scaling Hidden Dimensions:[10 marks]** Double the hidden state capacity (`HID_MT` from 256 to 512) and execute a single-epoch benchmark for both custom and native Bahdanau models. Compute the updated speedup ratio and determine whether scaling the parameter matrix size increases, decreases, or preserves the relative execution gap.

In [ ]:
##Code here##

# **Question 5 — Observations and Final Report[20 marks]**

- Report observations for every question.
- Do not only mention which model performed best.
- Explain why each improvement was needed based on the limitations observed in the previous model.
- Use your own results, plots, BLEU scores, losses, and error analysis wherever possible.

**Model Comparison — GoEmotions (Question 1)**

Discuss:

- RNN vs. LSTM vs. GRU training behavior.
- Single-layer vs. stacked vs. bidirectional models.
- Which emotions were hardest to classify.
- Common classification errors with examples.
- How gating helped compared with vanilla RNN.

**Encoder-Decoder Translation (Question 2)**

Discuss:

- Overall translation quality.
- Performance on short vs. long sentences.
- The fixed-context bottleneck observed in the experiments.
- Common translation errors.
- Example cases where the model failed.

**Attention (Question 3)**

Discuss:

- No-attention vs. dot-product vs. Bahdanau attention.
- BLEU score improvements.
- What the attention heatmaps show.
- Cases where attention improved translation.
- Errors that still remained even after adding attention.

**Scratch vs. PyTorch Native (Question 4)**

Discuss:

- Training speed.
- Parameter count.
- Loss and BLEU comparison.
- GPU memory usage.
- Difference in implementation complexity.
- Advantages of PyTorch built-in recurrent layers (`nn.RNN`, `nn.LSTM`, `nn.GRU`).

**Overall Synthesis [10 Marks]**

Summarize the complete progression:

$$
\text{Vanilla RNN}
\rightarrow
\text{LSTM/GRU}
\rightarrow
\text{Stacked/Bidirectional}
\rightarrow
\text{Encoder-Decoder}
\rightarrow
\text{Attention}
\rightarrow
\text{Optimized Native Implementations}
$$

For each step, explain the limitation of the previous model and how the next model addresses it:

- Vanilla RNN → LSTM/GRU: gradient and long-term dependency problem.
- LSTM/GRU → Stacked/Bidirectional: limited depth and context representation.
- Stacked/Bidirectional → Encoder-Decoder: need to handle sequence-to-sequence translation.
- Encoder-Decoder → Attention: fixed-context bottleneck, especially for long sentences.
- Attention → Native implementation: slow sequential computation in Python loops.



###Use the actual experimental results from the previous sections to support each observation.[10 Marks]

-----------write your answer here---------------

###Automatic Result-Table Output
This cell automatically collects the performance scores of all your trained models and prints them in neat tables (Table A, B, and C) inside your notebook output. Run this cell only after you have finished training all your models (Questions 1 through 4) so that the data is ready and available in your notebook memory.

**Do not change content of this cell**

In [ ]:
# ==============================================================================
# Automatic Result-Table Output Code Block (Updated for Scratch RNN & LSTM)
# ==============================================================================
import torch

def _fmt(value, digits=3):
    return "—" if value is None else f"{value:.{digits}f}"

print("\nTable A — Question 1 (multi-label metrics)")
print("| Model | Validation micro-F1 | Validation loss | Exact-match accuracy | Macro F1 |")
print("|---|---:|---:|---:|---:|")
for tag in experiment_configs:
    name = tag[0]
    hist = histories[name]
    report = results[name]["report"]
    macro_f1 = report.get("macro avg", {}).get("f1-score")
    print(f"| {name} | {_fmt(hist['val_acc'][-1])} | {_fmt(hist['val_loss'][-1])} | "
          f"{_fmt(hist['val_subset_acc'][-1])} | {_fmt(macro_f1)} |")

# Expanded translation rows to include scratch RNN and LSTM translation models
translation_rows = [
    ("Vanilla RNN Encoder-Decoder", "Scratch", "None", globals().get('hist_rnn_noattn', {}), globals().get('bleu_rnn_noattn', 0.0)),
    ("LSTM Encoder-Decoder", "Scratch", "None", globals().get('hist_lstm_noattn', {}), globals().get('bleu_lstm_noattn', 0.0)),
    ("BiGRU Encoder-Decoder", "Scratch", "None", globals().get('hist_noattn', {}), globals().get('test_bleu', 0.0)),
    ("BiGRU Encoder-Decoder", "Scratch", "Dot-product", globals().get('hist_dot', {}), globals().get('bleu_dot', 0.0)),
    ("BiGRU Encoder-Decoder", "Scratch", "Bahdanau", globals().get('hist_bah', {}), globals().get('bleu_bah', 0.0)),
    ("BiGRU Encoder-Decoder", "Native", "None", globals().get('hist_native_noattn', [{}, {}])[1], globals().get('bleu_native_noattn', 0.0)),
    ("BiGRU Encoder-Decoder", "Native", "Dot-product", globals().get('hist_native_dot', [{}, {}])[1], globals().get('bleu_native_dot', 0.0)),
    ("BiGRU Encoder-Decoder", "Native", "Bahdanau", globals().get('hist_native_bah', [{}, {}])[1], globals().get('bleu_native_bah', 0.0)),
]

print("\nTable B — Questions 2/3/4")
print("| Model | Implementation | Attention | Final validation loss | Test BLEU |")
print("|---|---|---|---:|---:|")
for model_name, implementation, attention, history, bleu in translation_rows:
    v_loss = history.get('val_loss', [0.0])[-1] if isinstance(history, dict) and 'val_loss' in history else 0.0
    print(f"| {model_name} | {implementation} | {attention} | {_fmt(v_loss)} | {_fmt(bleu, 2)} |")

***NOTE***

**A zip file named CS781-A2-[Roll-Number].zip, containing all required submission artifacts, will be created automatically. Your final submission must be a separate zip named CS781-A2-[First-Name]-[Last-Name]-[Roll-Number].zip, containing exactly two files: your .ipynb notebook, and the auto-generated CS781-A2-[Roll-Number].zip.**

###**Export Class Definitions to model_defs.py**

This cell automatically extracts your class definitions from executed cells and writes them to model_defs.py, then verifies every required class is present and importable. Run it only after executing every class-definition cell — and remember: don't rename any class, and keep testing/debugging statements out of class-definition cells, since this export pulls the cell's full source as-is.



**Do not change content of this cell**

In [ ]:
# --- Export class definitions only to model_defs.py (Papermill-safe) ---
import ast
import importlib.util
import sys

# Keep class names in dependency order.  Names, rather than class objects, are
# used so their source can be recovered from executed notebook input history.
REQUIRED_MODEL_CLASSES = [
    "RNNCell",
    "LSTMCell",
    "GRUCell",
    "RecurrentEncoder",
    "EmotionClassifier",
    "Encoder",
    "DecoderStep",
    "Seq2Seq",
    "DotProductAttention",
    "BahdanauAttention",
    "AttnDecoderStep",
    "Seq2SeqAttn",
    "CustomEncoder",
    "CustomDecoderStep",
    "NativeRNSEncoder",
    "NativeLSTMEncoder",
    "NativeGRUEncoder",
    "NativeRNNDecoderStep",
    "NativeLSTMDecoderStep",
    "NativeGRUDecoderStep",
    "NativeAttnRNNDecoderStep",
    "NativeAttnLSTMDecoderStep",
    "NativeAttnGRUDecoderStep",
    "NativeSeq2Seq",
    "NativeSeq2SeqAttn",
]


def extract_class_source_from_history(class_name: str) -> str:
    """Return the latest executed definition of ``class_name`` from IPython history."""
    ip = get_ipython()
    if ip is None:
        raise RuntimeError("Run this export cell inside Jupyter, Colab, or Kaggle.")

    for cell_source in reversed(ip.user_ns.get("In", [])):
        if not isinstance(cell_source, str) or f"class {class_name}" not in cell_source:
            continue
        try:
            tree = ast.parse(cell_source)
        except SyntaxError:
            continue

        for node in tree.body:
            if isinstance(node, ast.ClassDef) and node.name == class_name:
                source = ast.get_source_segment(cell_source, node)
                if source is not None:
                    return source

    raise RuntimeError(
        f"Could not find class {class_name!r} in executed notebook input history. "
        "Execute its definition cell before running this export cell."
    )


model_defs_header = (
    "import torch\n"
    "import torch.nn as nn\n"
    "import torch.nn.functional as F\n\n"
    "MAX_LEN_HI = 30\n"
)
class_sources = [
    extract_class_source_from_history(class_name)
    for class_name in REQUIRED_MODEL_CLASSES
]
model_defs_source = model_defs_header + "\n\n".join(class_sources) + "\n"

with open("model_defs.py", "w", encoding="utf-8") as f:
    f.write(model_defs_source)

with open("model_defs.py", "r", encoding="utf-8") as f:
    generated_source = f.read()

for class_name in REQUIRED_MODEL_CLASSES:
    assert f"class {class_name}" in generated_source, (
        f"{class_name} is missing from model_defs.py"
    )

spec = importlib.util.spec_from_file_location("model_defs_export_check", "model_defs.py")
if spec is None or spec.loader is None:
    raise ImportError("Could not create an import specification for model_defs.py")
model_defs_export_check = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = model_defs_export_check
spec.loader.exec_module(model_defs_export_check)

for class_name in REQUIRED_MODEL_CLASSES:
    assert hasattr(model_defs_export_check, class_name), (
        f"{class_name} could not be imported from model_defs.py"
    )

print("Wrote and verified class-only model_defs.py")

###Export, Verify, and Package Submission Artifacts
This final section guides you through exporting your model classes, saving your trained weights, and packing everything into the required submission zip file. Run these cleanup and packaging cells only after all models are fully trained and evaluated. Make sure to update the SUBMISSION_BASENAME variable with your own roll number before running the final zipping cell.

**Do not change content of this cell**

In [ ]:
# ==============================================================================
# Save Submission Artifacts Code Block
# ==============================================================================

import json
import torch
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

# 1. Save vocabularies.
with open("vocab_emotion.json", "w", encoding="utf-8") as f:
    json.dump(stoi, f)
with open("vocab_en.json", "w", encoding="utf-8") as f:
    json.dump(stoi_en, f)
with open("vocab_hi.json", "w", encoding="utf-8") as f:
    json.dump(stoi_hi, f)

# 2. Save the best Question 1 model, based on final validation accuracy.
best_tag = max(histories, key=lambda tag: histories[tag]["val_acc"][-1])
best_model = trained_models[best_tag]
torch.save(best_model.state_dict(), "q1_best.pt")

best_config = next(cfg for cfg in experiment_configs if cfg[0] == best_tag)
manifest = {
    "q1_best": {
        "cell": best_tag.split("-")[0],
        "num_layers": best_config[2],
        "bidirectional": best_config[3],
    }
}
with open("manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

# 3. Save Question 2, Question 3, and Scratch RNN/LSTM checkpoints.
if 'seq2seq_rnn_noattn' in globals():
    torch.save({
        "encoder": seq2seq_rnn_noattn.encoder.state_dict(),
        "decoder": seq2seq_rnn_noattn.decoder.state_dict(),
    }, "q2_seq2seq_rnn_noattn.pt")

if 'seq2seq_lstm_noattn' in globals():
    torch.save({
        "encoder": seq2seq_lstm_noattn.encoder.state_dict(),
        "decoder": seq2seq_lstm_noattn.decoder.state_dict(),
    }, "q2_seq2seq_lstm_noattn.pt")

torch.save({
    "encoder": seq2seq_teacher_forcing.encoder.state_dict(),
    "decoder": seq2seq_teacher_forcing.decoder.state_dict(),
}, "q2_seq2seq_teacher_forcing.pt")

torch.save({
    "encoder": seq2seq_scheduled_sampling.encoder.state_dict(),
    "decoder": seq2seq_scheduled_sampling.decoder.state_dict(),
}, "q2_seq2seq_scheduled_sampling.pt")

torch.save({
    "encoder": seq2seq_noattn.encoder.state_dict(),
    "decoder": seq2seq_noattn.decoder.state_dict(),
}, "q2_seq2seq_noattn.pt")

torch.save({
    "encoder": seq2seq_dot.encoder.state_dict(),
    "decoder": seq2seq_dot.decoder.state_dict(),
}, "q3_seq2seq_dot.pt")

torch.save({
    "encoder": seq2seq_bah.encoder.state_dict(),
    "decoder": seq2seq_bah.decoder.state_dict(),
}, "q3_seq2seq_bah.pt")

# 4. Save all three native Question 4 models.
torch.save({
    "encoder": native_seq2seq_noattn.encoder.state_dict(),
    "decoder": native_seq2seq_noattn.decoder.state_dict(),
}, "q4_native_noattn.pt")

torch.save({
    "encoder": native_seq2seq_dot.encoder.state_dict(),
    "decoder": native_seq2seq_dot.decoder.state_dict(),
}, "q4_native_dot.pt")

torch.save({
    "encoder": native_seq2seq_bah.encoder.state_dict(),
    "decoder": native_seq2seq_bah.decoder.state_dict(),
}, "q4_native_bah.pt")

print("All submission artifacts saved successfully.")

# --- Verify artifacts and create the ZIP archive ---
REQUIRED_SUBMISSION_FILES = [
    "q1_best.pt",
    "q2_seq2seq_rnn_noattn.pt",
    "q2_seq2seq_lstm_noattn.pt",
    "q2_seq2seq_noattn.pt",
    "q2_seq2seq_teacher_forcing.pt",
    "q2_seq2seq_scheduled_sampling.pt",
    "q3_seq2seq_dot.pt",
    "q3_seq2seq_bah.pt",
    "q4_native_noattn.pt",
    "q4_native_dot.pt",
    "q4_native_bah.pt",
    "model_defs.py",
    "manifest.json",
    "vocab_emotion.json",
    "vocab_en.json",
    "vocab_hi.json",
]

missing = [name for name in REQUIRED_SUBMISSION_FILES if not Path(name).is_file()]
if missing:
    raise FileNotFoundError("Missing required submission files: " + ", ".join(missing))

for checkpoint_name in [name for name in REQUIRED_SUBMISSION_FILES if name.endswith(".pt") and name != "q1_best.pt"]:
    checkpoint = torch.load(checkpoint_name, map_location="cpu")
    if not isinstance(checkpoint, dict) or set(checkpoint) != {"encoder", "decoder"}:
        raise ValueError(f"{checkpoint_name} must be an encoder/decoder state_dict pair.")

q1_state = torch.load("q1_best.pt", map_location="cpu")
if not isinstance(q1_state, dict) or "state_dict" in q1_state:
    raise ValueError("q1_best.pt must be a plain state_dict, not a wrapper checkpoint.")

SUBMISSION_BASENAME = "CS781-A2-[RollNo]"
archive_name = f"{SUBMISSION_BASENAME}.zip"
with ZipFile(archive_name, "w", ZIP_DEFLATED) as archive:
    for name in REQUIRED_SUBMISSION_FILES:
        archive.write(name)
print(f"Verification passed. Created submission archive: {archive_name}")

#**Hidden Test Cases[400 marks]**

Your submitted models, checkpoints, and code pipelines will be evaluated against unreleased hidden test cases to assess generalization and correctness beyond the visible validation/test splits.

* **Total Hidden Evaluation Marks**: **400 Marks**
* **Emotion Prediction (Question 1)**: 100 Marks (evaluated on multi-label classification accuracy, micro-F1, and robustness across diverse test text inputs).
* **Machine Translation & Attention (Questions 2–4)**: 300 Marks (evaluated on autoregressive generation quality, token alignment, and corpus-level SacreBLEU scores using your exported encoder-decoder state dicts).

**Important Submission Notice for Automated Grading:**

* Ensure all required `.pt` checkpoints, JSON vocabularies (`vocab_en.json`, `vocab_hi.json`, `vocab_emotion.json`), and `model_defs.py` are correctly generated and packaged inside your final submission ZIP archive.
* The evaluation script will automatically load your custom classes from `model_defs.py` and test weights via standard PyTorch state dictionary loading. Any renaming of required classes, function signatures, or expected output formats will cause the hidden test script to fail.

#Bonus (10 Marks):

Awarded for correct submissions that strictly follow all guidelines, file naming conventions (nomenclature), and include the presence of all required files.

**Do not change content of this cell**

In [ ]:
# ==============================================================================
# CS781 A2 -- Local Verification Cell
# ==============================================================================
import json, importlib.util, re
from pathlib import Path
import pandas as pd
import torch
import sacrebleu
from datasets import load_dataset

SUBMISSION_DIR = Path(".")   # folder containing model_defs.py + all checkpoints + jsons

PAD, UNK, SOS, EOS = "<pad>", "<unk>", "<sos>", "<eos>"
MAX_LEN_EMO = MAX_LEN_EN = MAX_LEN_HI = 30
DEVICE = torch.device("cpu")
TOKEN_RE = re.compile(r"[A-Za-z']+|[.,!?;]")
_GOEMOTIONS_28 = ["admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval","disgust",
    "embarrassment","excitement","fear","gratitude","grief","joy","love",
    "nervousness","optimism","pride","realization","relief","remorse",
    "sadness","surprise","neutral"]

def tokenize_en(t): return TOKEN_RE.findall(t.lower())

def load_vocab(p):
    stoi = json.load(open(p)); return stoi, {v: k for k, v in stoi.items()}

def encode(text, stoi, max_len):
    ids = [stoi.get(t, stoi[UNK]) for t in tokenize_en(text)][:max_len]
    return ids + [stoi[PAD]] * (max_len - len(ids))

def load_emotion_from_hf():
    ds = load_dataset(
        REPO_ID,
        data_files={"test": "Assignment2/goemotions_3_1.parquet"},
        token=hf_token,
        split="test"
    )
    df = ds.to_pandas()
    onehot = df[_GOEMOTIONS_28].to_numpy()
    mask = onehot.sum(axis=1) == 1
    labels = onehot[mask].argmax(axis=1)
    return list(zip(df.loc[mask, "text"].tolist(), labels.tolist()))

def load_mt_from_hf():
    ds = load_dataset(
        REPO_ID,
        data_files={"test": "Assignment2/EnglishHindi_test.parquet"},
        token=hf_token,
        split="test"
    )
    df = ds.to_pandas()
    return list(zip(df["translation"].apply(lambda d: d["en"]).tolist(),
                     df["translation"].apply(lambda d: d["hi"]).tolist()))

def emb_shape(sd):
    k = next(k for k in sd if k.endswith("embedding.weight")); return sd[k].shape

def bridge_hidden(enc_sd):
    k = next(k for k in enc_sd if k.endswith("bridge.weight") or k.endswith("bridge.0.weight"))
    return enc_sd[k].shape[0]

# --- Import your saved model_defs.py, injecting ambient globals ---
stoi_emo, _ = load_vocab(SUBMISSION_DIR / "vocab_emotion.json")
stoi_en, _ = load_vocab(SUBMISSION_DIR / "vocab_en.json")
stoi_hi, itos_hi = load_vocab(SUBMISSION_DIR / "vocab_hi.json")

spec = importlib.util.spec_from_file_location("model_defs", SUBMISSION_DIR / "model_defs.py")
mod = importlib.util.module_from_spec(spec)
mod.__dict__.update({"PAD": PAD, "UNK": UNK, "SOS": SOS, "EOS": EOS,
                      "MAX_LEN_EMO": MAX_LEN_EMO, "MAX_LEN_EN": MAX_LEN_EN, "MAX_LEN_HI": MAX_LEN_HI,
                      "DEVICE": DEVICE, "stoi": stoi_emo, "stoi_emo": stoi_emo,
                      "stoi_en": stoi_en, "stoi_hi": stoi_hi, "itos_hi": itos_hi})
spec.loader.exec_module(mod)

manifest = json.load(open(SUBMISSION_DIR / "manifest.json"))

# --- Q1 Evaluation ---
sd = torch.load(SUBMISSION_DIR / "q1_best.pt", map_location="cpu")
cfg = manifest["q1_best"]
vocab_size, emb_dim = emb_shape(sd)
hidden_key = next(k for k in sd if k.endswith((".h2h.weight", ".h2g.weight", ".h2z.weight")))
out_dim = sd[hidden_key].shape[0]
hidden_size = out_dim // 4 if hidden_key.endswith(".h2g.weight") else out_dim
cell_map = {"RNN": mod.RNNCell, "LSTM": mod.LSTMCell, "GRU": mod.GRUCell}
num_classes = sd[next(k for k in sd if k.endswith("fc.weight"))].shape[0]
q1_model = mod.EmotionClassifier(vocab_size=vocab_size, emb_dim=emb_dim, hidden_size=hidden_size,
    num_classes=num_classes, cell_cls=cell_map[cfg["cell"]], num_layers=cfg["num_layers"],
    bidirectional=cfg["bidirectional"], is_lstm=(cfg["cell"] == "LSTM"))
q1_model.load_state_dict(sd); q1_model.eval()

emotion_data = load_emotion_from_hf()
correct = 0
with torch.no_grad():
    for text, label in emotion_data:
        x = torch.tensor([encode(text, stoi_emo, MAX_LEN_EMO)], dtype=torch.long)
        correct += int(q1_model(x).argmax(dim=-1).item() == label)
print(f"q1_accuracy: {correct / len(emotion_data):.4f}")

# --- Q2/Q3/Q4 Evaluation ---
CHECKPOINTS = {
    "q2_rnn_noattn": ("q2_seq2seq_rnn_noattn.pt", "CustomEncoder", "CustomDecoderStep", "Seq2Seq", None, False, "RNNCell", False),
    "q2_lstm_noattn": ("q2_seq2seq_lstm_noattn.pt", "CustomEncoder", "CustomDecoderStep", "Seq2Seq", None, True, "LSTMCell", False),
    "q2_noattn": ("q2_seq2seq_noattn.pt", "Encoder", "DecoderStep", "Seq2Seq", None, False, None, False),
    "q2_teacher_forcing": ("q2_seq2seq_teacher_forcing.pt", "Encoder", "DecoderStep", "Seq2Seq", None, False, None, False),
    "q2_scheduled_sampling": ("q2_seq2seq_scheduled_sampling.pt", "Encoder", "DecoderStep", "Seq2Seq", None, False, None, False),
    "q3_dot": ("q3_seq2seq_dot.pt", "Encoder", "AttnDecoderStep", "Seq2SeqAttn", "dot", False, None, True),
    "q3_bah": ("q3_seq2seq_bah.pt", "Encoder", "AttnDecoderStep", "Seq2SeqAttn", "bahdanau", False, None, True),
    "q4_noattn": ("q4_native_noattn.pt", "NativeGRUEncoder", "NativeGRUDecoderStep", "NativeSeq2Seq", None, False, None, True),
    "q4_dot": ("q4_native_dot.pt", "NativeGRUEncoder", "NativeAttnGRUDecoderStep", "NativeSeq2SeqAttn", "dot", False, None, True),
    "q4_bah": ("q4_native_bah.pt", "NativeGRUEncoder", "NativeAttnGRUDecoderStep", "NativeSeq2SeqAttn", "bahdanau", False, None, True),
}
mt_data = load_mt_from_hf()

for tag, (fname, enc_cls_name, dec_cls_name, wrap_cls_name, attn_type, is_lstm, cell_name, own_greedy) in CHECKPOINTS.items():
    try:
        ckpt = torch.load(SUBMISSION_DIR / fname, map_location="cpu")
        enc_sd, dec_sd = ckpt["encoder"], ckpt["decoder"]
        src_vocab, emb_en = emb_shape(enc_sd)
        tgt_vocab, emb_hi = emb_shape(dec_sd)
        hidden = bridge_hidden(enc_sd)

        enc_cls = getattr(mod, enc_cls_name)
        if enc_cls_name == "CustomEncoder":
            encoder = enc_cls(vocab_size=src_vocab, emb_dim=emb_en, hidden_size=hidden,
                               cell_cls=getattr(mod, cell_name), is_lstm=is_lstm, num_layers=2)
        else:
            encoder = enc_cls(vocab_size=src_vocab, emb_dim=emb_en, hidden_size=hidden, num_layers=2)
        encoder.load_state_dict(enc_sd)

        dec_cls = getattr(mod, dec_cls_name)
        if attn_type is None:
            if dec_cls_name == "CustomDecoderStep":
                decoder = dec_cls(vocab_size=tgt_vocab, emb_dim=emb_hi, hidden_size=hidden,
                                   cell_cls=getattr(mod, cell_name), is_lstm=is_lstm, num_layers=2)
            else:
                decoder = dec_cls(vocab_size=tgt_vocab, emb_dim=emb_hi, hidden_size=hidden, num_layers=2)
        else:
            if attn_type == "dot":
                attn = mod.DotProductAttention(dec_hidden=hidden, enc_hidden=hidden * 2)
            else:
                wa_key = next((k for k in dec_sd if k.endswith("attention.W_a.weight")), None)
                attn_dim = dec_sd[wa_key].shape[0] if wa_key else 128
                attn = mod.BahdanauAttention(dec_hidden=hidden, enc_hidden=hidden * 2, attn_dim=attn_dim)
            decoder = dec_cls(vocab_size=tgt_vocab, emb_dim=emb_hi, hidden_size=hidden,
                               enc_hidden=hidden * 2, attention_module=attn, num_layers=2)
        decoder.load_state_dict(dec_sd)

        wrap_cls = getattr(mod, wrap_cls_name)
        model = wrap_cls(encoder, decoder, num_layers=2, is_lstm=is_lstm) if wrap_cls_name != "Seq2SeqAttn" \
            else wrap_cls(encoder, decoder, num_layers=2)
        model.eval()

        hyps, refs = [], []
        with torch.no_grad():
            for src_text, tgt_text in mt_data:
                src = torch.tensor([encode(src_text, stoi_en, MAX_LEN_EN)], dtype=torch.long)
                if own_greedy:
                    out = model.greedy_decode(src)
                    ids = (out[0] if isinstance(out, tuple) else out)[0].tolist()
                else:
                    enc_out = model.encoder(src)
                    if is_lstm:
                        h0, c0 = enc_out[1]; states = [(h0, c0)] * model.num_layers
                    else:
                        states = [enc_out[1]] * model.num_layers
                    inp = torch.tensor([stoi_hi[SOS]], dtype=torch.long)
                    ids = []
                    for _ in range(MAX_LEN_HI):
                        logits, states = model.decoder(inp, states)
                        inp = logits.argmax(dim=-1)
                        idx = inp.item()
                        if idx == stoi_hi[EOS]: break
                        ids.append(idx)
                words = []
                for idx in ids:
                    if idx == stoi_hi[EOS]: break
                    if idx not in (stoi_hi[PAD], stoi_hi[SOS]): words.append(itos_hi.get(idx, UNK))
                hyps.append(" ".join(words)); refs.append(tgt_text)

        bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
        print(f"{tag}_bleu: {bleu:.4f}")
    except Exception as e:
        print(f"{tag}: ERROR -- {e}")

print("\nIf every line above printed a number with no ERROR, your verification is complete.")

#Submission Guidelines and Deliverables Checklist
**Notebook**
- [ ] Download this Colab notebook with all your solutions, written analysis, and displayed plot/diagram outputs, and save it as `CS781-A2-[Firstname]-[Lastname]-[Rollno].ipynb`.
- [ ] `hf_token` is **cleared/blank** before you submit (or you used Colab Secrets instead).
- [ ] Name/Roll/Email and the LLM Usage Declaration are filled in completely.

- [ ] Do not rename any .pt checkpoint file, any .json file, model_defs.py, or any required class since the autograder matches all of these by exact name.
In each class-definition cell, include only the class definition itself, no testing, debugging, or print statements in the same cell, and no changes to the class name.

- [ ] Every code cell has been run top-to-bottom at least once and its output is visible in the saved notebook.
- [ ] Variable names are standardized: `trained_models` dict populated during the Q1 sweep loop, plus `seq2seq_rnn_noattn`, `seq2seq_lstm_noattn`, `seq2seq_noattn`, `seq2seq_dot`, `seq2seq_bah`, `native_seq2seq_noattn`, `native_seq2seq_dot`, `native_seq2seq_bah`.

**Question 1**
- [ ] RNN, LSTM, GRU cells implemented from scratch with docstrings + parameter tables.
- [ ] Scratch classes do **not** use `nn.RNN`, `nn.LSTM`, or `nn.GRU`.
- [ ] All 9 single/stacked/bidirectional configurations trained; 3x3 confusion-matrix grid displayed.
- [ ] Written answers to all Question-1 questions (conceptual, logical, analytical, experimental) with your own numbers.
- [ ] Best model's checkpoint saved as `q1_best.pt`, as a **plain state_dict** (not a wrapper dict).

**Question 2**
- [ ] Encoder/Decoder/Seq2Seq implemented from scratch, reusing Question 1's recurrent cell wrappers.
- [ ] Training curve + test BLEU + short-vs-long BLEU split reported.
- [ ] Written answers to all Question-2 questions.
- [ ] `seq2seq_rnn_noattn` and `seq2seq_lstm_noattn` saved as `q2_seq2seq_rnn_noattn.pt` and `q2_seq2seq_lstm_noattn.pt` encoder/decoder state_dict pairs.
- [ ] `seq2seq_noattn` checkpoint saved as an encoder/decoder state_dict pair (not the full wrapper model).
- [ ] `seq2seq_teacher_forcing` saved as `q2_seq2seq_teacher_forcing.pt`, and `seq2seq_scheduled_sampling` saved as `q2_seq2seq_scheduled_sampling.pt`; each is an encoder/decoder state_dict pair.

**Question 3**
- [ ] Dot-product and Bahdanau attention implemented from scratch.
- [ ] BLEU comparison table (no-attn / dot / Bahdanau) and 5 attention heatmaps.
- [ ] Written answers to all Question-3 questions.
- [ ] `seq2seq_dot` and `seq2seq_bah` checkpoints saved as encoder/decoder state_dict pairs.

**Question 4**
- [ ] Native PyTorch re-implementations of the encoder, plain decoder, and attention decoder for RNN, LSTM, and GRU.
- [ ] Native classes use official PyTorch recurrent modules (`nn.RNN`, `nn.LSTM`, `nn.GRU`).
- [ ] Filled-in benchmark table (params, sec/epoch, peak memory, BLEU) for custom vs. native.
- [ ] Written answers to all Question-4 questions.
- [ ] `native_seq2seq_noattn`, `native_seq2seq_dot`, and `native_seq2seq_bah` are trained; `native_dot` and `native_bah` remain compatible aliases.
- [ ] `q4_native_noattn.pt`, `q4_native_dot.pt`, and `q4_native_bah.pt` are saved as encoder/decoder state_dict pairs.

**Question 5**
- [ ] Written observations for every sub-section, grounded in your own results.
- [ ] Result tables filled with real numbers.

**`model_defs.py`**
- [ ] Contains only the required class definitions, copied from your solved notebook.
- [ ] No class, `__init__` argument, or required internal attribute has been renamed.
- [ ] `forward()` return order is preserved exactly for encoders, decoders, and attention submodules.
- [ ] Contains no data loading, no training loop, no HF token, and no install commands — must import cleanly on its own.

**Checkpoints**
- [ ] All checkpoint files saved using the fixed naming convention (`q1_best.pt`, `q2_seq2seq_rnn_noattn.pt`, `q2_seq2seq_lstm_noattn.pt`, `q2_seq2seq_noattn.pt`, `q2_seq2seq_teacher_forcing.pt`, `q2_seq2seq_scheduled_sampling.pt`, `q3_seq2seq_dot.pt`, `q3_seq2seq_bah.pt`, `q4_native_noattn.pt`, `q4_native_dot.pt`, `q4_native_bah.pt`).
- [ ] Each checkpoint file is under the specified size limit.

**JSON Files**
- [ ] `manifest.json` created, specifying your Q1 best model's configuration.
- [ ] `vocab_emotion.json`, `vocab_en.json`, and `vocab_hi.json` created from your vocabulary dictionaries.
- [ ] `vocab_hi.json` includes the `<sos>` and `<eos>` entries.

**Verification**
- [ ] Verification script run against your own weights and the visible test set before submitting.
- [ ] Any errors reported by the verification script fixed before submitting.


**ZIP**
- [ ] Zip the notebook, `model_defs.py`, all checkpoints, and all JSON files into `CS781-A2-[RollNo].zip`.
- [ ] Once the zip has been created using code shared in the colab notebook above, make a zip named `CS781-A2-[firstname]-[lastname]-[rollno].zip` containing the `.ipynb` file and the previously created `.zip` file (containing `model_defs.py`, all checkpoints, and all JSON files), then submit it to Hello-IITK.
- [ ] Double-check that the zip filename matches the required format exactly.
- [ ] Submit on Hello-IITK **at least 30 minutes** before the deadline.

#References & Citations

- Elman, J. L. (1990). *Finding Structure in Time*. Cognitive Science, 14(2), 179-211.
- Bengio, Y., Simard, P., & Frasconi, P. (1994). *Learning Long-Term Dependencies with Gradient Descent is Difficult*. IEEE Trans. Neural Networks.
- Hochreiter, S. (1991). *Untersuchungen zu dynamischen neuronalen Netzen* (Diploma thesis) — earliest formal treatment of vanishing gradients.
- Hochreiter, S., & Schmidhuber, J. (1997). *Long Short-Term Memory*. Neural Computation, 9(8), 1735-1780.
- Gers, F. A., Schmidhuber, J., & Cummins, F. (2000). *Learning to Forget: Continual Prediction with LSTM*. Neural Computation, 12(10), 2451-2471.
- Schuster, M., & Paliwal, K. K. (1997). *Bidirectional Recurrent Neural Networks*. IEEE Trans. Signal Processing.
- Graves, A., & Schmidhuber, J. (2005). *Framewise phoneme classification with bidirectional LSTM and other neural network architectures*. Neural Networks, 18(5-6), 602-610.
- Cho, K., van Merrienboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H., & Bengio, Y. (2014). *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*. EMNLP. arXiv:1406.1078
- Chung, J., Gulcehre, C., Cho, K., & Bengio, Y. (2014). *Empirical Evaluation of Gated Recurrent Neural Networks on Sequence Modeling*. NeurIPS DL Workshop. arXiv:1412.3555
- Sutskever, I., Vinyals, O., & Le, Q. V. (2014). *Sequence to Sequence Learning with Neural Networks*. NeurIPS. arXiv:1409.3215
- Bahdanau, D., Cho, K., & Bengio, Y. (2015). *Neural Machine Translation by Jointly Learning to Align and Translate*. ICLR. arXiv:1409.0473
- Luong, M.-T., Pham, H., & Manning, C. D. (2015). *Effective Approaches to Attention-based Neural Machine Translation*. EMNLP. arXiv:1508.04025
- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2017). *Attention Is All You Need*. NeurIPS.
- Demszky, D., Movshovitz-Attias, D., Ko, J., Cowen, A., Nemade, G., & Ravi, S. (2020). *GoEmotions: A Dataset of Fine-Grained Emotions*. ACL. arXiv:2005.00547
- Kunchukuttan, A., Mehta, P., & Bhattacharyya, P. (2018). *The IIT Bombay English-Hindi Parallel Corpus*. LREC.
- Papineni, K., Roukos, S., Ward, T., & Zhu, W.-J. (2002). *BLEU: a Method for Automatic Evaluation of Machine Translation*. ACL.
- PyTorch documentation: `nn.GRU`, `nn.GRUCell`, `nn.LSTM`, `nn.RNN` — https://pytorch.org/docs/stable/nn.html
- HuggingFace `datasets` library documentation — https://huggingface.co/docs/datasets

- Williams, R. J., & Zipser, D. (1989). *A Learning Algorithm for Continually Running Fully Recurrent Neural Networks*. Neural Computation, 1(2), 270–280. https://doi.org/10.1162/neco.1989.1.2.270
- Bengio, S., Vinyals, O., Jaitly, N., & Shazeer, N. (2015). *Scheduled Sampling for Sequence Prediction with Recurrent Neural Networks*. NeurIPS. https://proceedings.neurips.cc/paper_files/paper/2015/file/e995f98d56967d946471af29d7bf99f1-Paper.pdf
- Freitag, M., & Al-Onaizan, Y. (2017). *Beam Search Strategies for Neural Machine Translation*. NMT@ACL. https://aclanthology.org/W17-3207/
- Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y. (2020). *The Curious Case of Neural Text Degeneration*. ICLR. https://arxiv.org/abs/1904.09751

- [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) (*Christopher Olah*) — Excellent visual explanations of LSTM gating mechanisms.

- [Long Short-Term Memory (Hochreiter & Schmidhuber, 1997)](https://www.google.com/search?q=https://www.bioinf.jku.at/publications/older/2604.pdf) — The foundational paper introducing LSTMs.

- [Recurrent Neural Network Regularization (Zaremba et al., 2014)](https://arxiv.org/abs/1409.2329) — Practical guidelines on applying Dropout to LSTMs.

- [Bidirectional Recurrent Neural Networks (Schuster & Paliwal, 1997)](https://ieeexplore.ieee.org/document/650093) — The seminal paper introducing bidirectional context processing.

- [Speech Recognition with Deep Recurrent Neural Networks (Graves et al., 2013)](https://arxiv.org/abs/1303.5778) — Demonstrates the power of multi-layer stacked RNN architectures.
